# Multi-Asset Scenario Engine — standalone research engine

A **Run All** front end for physical-measure scenario generation, cross-asset dependence modelling,
multi-horizon instrument repricing and portfolio tail-risk analytics across equities, Treasury rates,
credit spreads and FX.

**Current public architecture**

- asset-class-specific equity / rates / credit / FX risk drivers;
- conditional-volatility filtered historical simulation with stationary block resampling;
- **hierarchical global/local cross-asset dependence** as the research architecture;
- **blockwise empirical dependence** retained as the hard benchmark;
- direct 1W / 1M / 3M / 1Y horizon simulation and repricing;
- corporate-credit migration/default, FX forwards and Treasury/corporate-bond repricing;
- portfolio VaR/ES, tail contributions and minimum-CVaR analysis.

The heavier architecture ablations, duration-conditioned PIT experiments and proper-scoring-rule
adjudication live in the separate notebooks in this repository.



## Research positioning

The notebook is now structured as a **cross-asset scenario engine under the physical measure** with three explicit layers:

1. a **risk-free curve factor generator**,
2. a **plug-and-play equity-premium engine**,
3. an **idiosyncratic residual block** compressed through PCA.

The goal is not point forecasting, but **joint horizon scenario generation** (one week by default; one month, three months or one year via `HORIZON_MODE`), **instrument-level horizon repricing**, and **portfolio tail-risk analysis**. The current implementation also lets the user **choose the ERP simulation engine**.


In [ ]:
# Core imports and global configuration
import warnings
warnings.filterwarnings("ignore")

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf
from pandas_datareader import data as web

from sklearn.decomposition import PCA
from statsmodels.tsa.api import VAR

from scipy import sparse
from scipy.optimize import linprog
from scipy.optimize import minimize
from scipy.stats import chi2
from scipy.stats import t as student_t
from scipy.stats import norm
from scipy.special import gammaln
from scipy.linalg import expm, logm

from pandas.tseries.offsets import BDay

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")
plt.rcParams["figure.figsize"] = (10, 6)

# Broad US equity prototype universe (multi-sector)
TICKERS = ["MSFT", "JPM", "XOM", "JNJ", "PG", "HD", "CAT", "NEM", "PLD", "DUK"]
SECTOR_MAP = {
    "MSFT": "Information Technology",
    "JPM": "Financials",
    "XOM": "Energy",
    "JNJ": "Health Care",
    "PG": "Consumer Staples",
    "HD": "Consumer Discretionary",
    "CAT": "Industrials",
    "NEM": "Materials",
    "PLD": "Real Estate",
    "DUK": "Utilities",
}

MARKET_PROXY = "SPY"

# Issue #8: richer Treasury key-rate grid (ascending tenor). These DGS* series are continuously
# available from FRED since well before START_DATE (DGS20 is intentionally excluded because it was
# discontinued 1987-1993 and only reintroduced in Oct-2020, which would create a large NaN gap and
# shrink the aligned sample). With 10 tenors the rates PCA genuinely compresses and the
# level/slope/curvature interpretation becomes meaningful rather than forced on 3 points.
RATE_TICKERS = ["DGS1MO", "DGS3MO", "DGS6MO", "DGS1", "DGS2", "DGS3", "DGS5", "DGS7", "DGS10", "DGS30"]
TENOR_YEARS = {
    "DGS1MO": 1.0 / 12.0,
    "DGS3MO": 0.25,
    "DGS6MO": 0.5,
    "DGS1": 1.0,
    "DGS2": 2.0,
    "DGS3": 3.0,
    "DGS5": 5.0,
    "DGS7": 7.0,
    "DGS10": 10.0,
    "DGS30": 30.0,
}

# Credit block: ICE BofA option-adjusted spreads (OAS) by rating bucket, ascending credit risk
# (AAA -> CCC). These form a credit-quality curve that the PCA compresses to a few factors
# (level / IG-HY slope / curvature) and that feeds corporate-bond repricing at Treasury-zero + OAS.
CREDIT_OAS_SERIES = {
    "BAMLC0A1CAAA": "AAA",   # ICE BofA AAA US Corporate OAS
    "BAMLC0A2CAA": "AA",     # ICE BofA AA US Corporate OAS
    "BAMLC0A3CA": "A",       # ICE BofA Single-A US Corporate OAS
    "BAMLC0A4CBBB": "BBB",   # ICE BofA BBB US Corporate OAS
    "BAMLH0A1HYBB": "BB",    # ICE BofA BB US High Yield OAS
    "BAMLH0A2HYB": "B",      # ICE BofA Single-B US High Yield OAS
    "BAMLH0A3HYC": "CCC",    # ICE BofA CCC & Lower US High Yield OAS
}
TARGET_CREDIT_EXPLAINED_VAR = 0.99

# Option A (credit parsimony): hard-cap the credit factors retained from the rating-bucket OAS curve.
# The buckets are highly collinear, so PC1 ~ credit level, PC2 ~ IG-HY slope, PC3 ~ curvature capture
# essentially all the variance; keeping all 7 would inject near-degenerate dimensions into the joint
# VAR. PC1 is also the systematic credit factor a later 1-year CreditMetrics view would reuse. Set to
# None to retain the full TARGET_CREDIT_EXPLAINED_VAR set (the v7 behaviour).
CREDIT_MAX_FACTORS = 3
# Credit variance the compression drops (v22). The 3-factor cap above is right for the DYNAMICS (7
# near-collinear buckets would inject weakly identified dimensions into the joint VAR), but its claim to
# keep "essentially all the variance" holds for the TOTAL, which the HY buckets dominate -- not bucket by
# bucket: on the 2023-2026 real data the IG buckets came out at 0.88-0.91x their historical weekly vol,
# i.e. a quarter of their own variance gone. With CREDIT_BLOCK_SOURCE="moody_2pt" the 2-point -> bucket
# bridge drops the bucket-specific variance as well (daily r^2 0.22-0.42 on the real overlap).
#   "drop"    -- as up to v21: the compressed variance is gone (byte-identical);
#   "restore" -- keep the parsimonious drivers, and add back at the horizon the h-day sum of the
#                compression residuals (PCA truncation, or bridge residual) over a random window of the
#                estimation sample: the empirical horizon variance, including any day-to-day noise that
#                mean-reverts, cross-bucket dependence intact. Own RNG stream: nothing else moves.
CREDIT_RESIDUAL_MODE = "restore"   # v25c RUN: exactly the validated v24 RUN setting
# Frequency at which the 2-point -> bucket bridge is estimated (v23). The v22 run measured, on the real
# 2023-2026 overlap, that a DAILY interceptless OLS carries only 16-38% of each bucket's weekly OAS
# variance (less than its daily r^2 of 0.25-0.45), and that the bridged move and its residual co-move
# at the weekly horizon (Moody's seasoned yields and ICE OAS are not synchronous), so even with the
# residual restored 12-32% of the weekly variance was missing. "horizon" estimates gamma on overlapping
# HORIZON_BDAYS-day changes (capped so the overlap still holds ~20 independent windows): the bridged
# move and the residual are then orthogonal AT the horizon and their variances add up exactly.
CREDIT_BRIDGE_FREQUENCY = "horizon"   # v25c RUN: exactly the validated v24 RUN setting

# Sample configuration
# IMPORTANT (data availability): as of April 2026 the FRED ICE BofA OAS series are licensed to
# expose only a rolling ~3 years of history (BAMLC0A4CBBB now spans roughly 2023-05 to 2026-05).
# To keep the credit block reproducible against live FRED, the whole sample lives on that recent
# window; all four blocks (equity, rates, Fama-French, credit) then share common dates. To run the
# original 2018-2022 / 2023-01-01 setup you would need a non-truncated credit source (e.g. Moody's
# seasoned Aaa/Baa yields DAAA/DBAA differenced against a Treasury tenor, a coarser 2-point curve).
START_DATE = "2023-05-15"
# Sample mode. "short" uses the single START_DATE above for every block -- the original ~3-year
# window, byte-identical to v17. "long" lets EACH BLOCK bring its full available history
# (BLOCK_START_DATES below); the aligned joint window is then whatever the data jointly support --
# the data cell prints each block's first usable date and which one binds. With the current FRED/
# Ken-French universe the binding constraint is typically the Treasury panel (DGS1MO starts 2001,
# DGS30 has a gap until 2006-02), giving a ~20-year joint window: more regimes for the loadings and
# the innovation pool, stabler GARCH persistence (the v17 caveat), many more independent backtest
# windows (the 1Y weakness), and a long-run drift instead of one estimated on a 3-year bull sample.
# Honest costs: full-sample betas/loadings become multi-regime averages (the point-in-time backtest
# self-audits this), and runtime grows roughly linearly in the window length.
SAMPLE_MODE = "long"                     # "short" or "long"
BLOCK_START_DATES = {                     # per-block starts used when SAMPLE_MODE = "long"
    "equity": "1995-01-01",               # tickers/proxy/FF factors (all current names predate this)
    "rates": "1995-01-01",                # panel self-truncates to where ALL tenors exist (~2006-02)
    "credit": "1996-12-31",               # OAS inception -- but FRED serves a rolling ~3y only:
                                          # in long mode this block BINDS the joint window unless
                                          # CREDIT_BLOCK_SOURCE = "moody_2pt" (below)
    "fx": "1999-01-04",                   # EUR/USD spot exists only since the euro launch
}

# Credit block source. The FRED ICE BofA OAS panel is licensed to a rolling ~3-year history, so in
# "long" mode it BINDS the whole joint window to ~3 years regardless of the other blocks (the v18
# lesson). "moody_2pt" replaces the credit DRIVER panel with a real, never-truncated 2-point curve:
# Moody's seasoned Aaa/Baa yields minus the 10Y Treasury (daily since 1986, point in time, nothing
# reconstructed) -- the joint credit dynamics, including every postwar crisis, come from real data.
# The 7-bucket OAS panel is still downloaded and used for what it is genuinely good for: today's
# bucket ANCHOR and a sensitivity BRIDGE (per-bucket OLS of bucket-OAS changes on 2-point changes,
# estimated on the overlap) that maps simulated 2-point moves into bucket space for bond REPRICING.
# Declared costs: Moody's yields are long-maturity seasoned composites against a 10Y Treasury leg
# (maturity mismatch), and a linear calm-window bridge understates crisis bucket decompression
# (CCC convexity) -- a repricing-granularity caveat, not a credit-dynamics one. Recommended
# pairing: SAMPLE_MODE = "long" (that is what unlocks the pre-2023 history).
CREDIT_BLOCK_SOURCE = "moody_2pt"       # "oas_buckets" or "moody_2pt"

# Long-mode contiguity guard: if the aligned long window contains an internal calendar gap longer
# than this (the known case: the DGS30 discontinuation 2002-02 -> 2006-02), the window is trimmed
# to start AFTER the last such gap and the trim is printed. A day-to-day diff taken across a multi-
# year hole would otherwise enter the bootstrap pool as one fake extreme observation -- poison for
# a tail engine. Short mode never trims.
LONG_GAP_TOLERANCE_DAYS = 21              # calendar days; weekends/holiday runs never exceed ~10
AS_OF_DATE = pd.Timestamp("2026-05-01")
DOWNLOAD_END = (AS_OF_DATE + pd.Timedelta(days=10)).strftime("%Y-%m-%d")

# Active equity engine: choose among
#   "latent_statistical"
#   "proxy_market"
#   "apt_ff3"
#   "apt_carhart"
ACTIVE_EQUITY_ENGINE = "apt_carhart"

# Expected returns and changes over the horizon (v22). Up to v21 every block drifted at its SAMPLE
# MEAN over the estimation window: each stock at its own historical mean excess return (alpha
# included), each yield, spread and FX rate at its historical trend. On the 2023-2026 short sample that
# is a bull market projected forward -- CAT at about +59% a year, the equal-weight book at about +21% --
# and it drives the one-year equity VaR95 to roughly zero; it also means the four "equity-premium
# engines" above all carry the SAME premium (they differ only in the dependence structure).
#   "sample" -- as up to v21 (byte-identical).
#   "factor" -- an actual premium model: each stock earns beta' * lambda, with lambda the LONG-RUN
#               Fama-French premia (daily history from FF_LONG_RUN_START up to each as-of date, so the
#               backtest stays point-in-time) and alpha = 0. Factors outside the Fama-French set (the
#               market proxy, the latent factor) are priced through their market beta. Yields,
#               spreads and FX drift at zero (martingale; the OU option still pulls levels if gated).
#   "zero"   -- no drift anywhere: equities earn the risk-free rate, levels are martingales.
# The walk-forward backtest at 1M/3M/1Y (BACKTEST_HORIZONS) is where the three are compared.
DRIFT_MODE = "factor"                     # v25b: align with the validated v24 RUN
FF_LONG_RUN_START = "1926-07-01"          # "factor" only: start of the long-run premium estimate

# Scenario engine configuration
# HORIZON_MODE selects the risk horizon for the SAME engine: "1W" (5 business days, the original),
# "1M" (21), "3M" (63) or "1Y" (252). The horizon flows through HORIZON_BDAYS to the simulator (it
# cumulates this many daily steps), the horizon date, the bond/forward repricing, the validation
# window, and the backtest. Risks that are ~0 over a week become first-order as the horizon grows:
# credit rating MIGRATION and DEFAULT (modelled CreditMetrics-style below, on from 1M), the compounded
# equity risk premium (the daily factor/alpha means accumulate -- handled automatically by the
# simulator's mean term), and the carry/forward-points on the FX forward. 1M and 3M are not just two
# more reporting horizons: they are the longest horizons a walk-forward backtest can still validate
# with NON-overlapping windows (about 250 and 85 of them on the 2001-2026 long window, against about
# 20 at 1Y), and they bracket the 34-business-day half-life of the volatility filter -- a vol shock is
# ~90% alive after a week and ~0% after a year, so only 1M/3M test how fast it reverts.
HORIZON_MODE = "1W"                                  # "1W", "1M", "3M" or "1Y"
HORIZON_BDAYS_BY_MODE = {"1W": 5, "1M": 21, "3M": 63, "1Y": 252}
HORIZON_LABELS = {"1W": "one-week", "1M": "one-month", "3M": "three-month", "1Y": "one-year"}
if HORIZON_MODE not in HORIZON_BDAYS_BY_MODE:
    # Up to v20 any value other than "1W" silently became 252 days; an unknown mode is now an error.
    raise ValueError(f"Unknown HORIZON_MODE {HORIZON_MODE!r}; expected one of {list(HORIZON_BDAYS_BY_MODE)}")
HORIZON_BDAYS = HORIZON_BDAYS_BY_MODE[HORIZON_MODE]
HORIZON_YEARS = HORIZON_BDAYS / 252.0
HORIZON_LABEL = HORIZON_LABELS[HORIZON_MODE]
N_SCENARIOS = 5_000
RNG_SEED = 42
VALIDATION_SEED = 2024          # independent seed for the out-of-sample (fresh-draw) check (issue #4)
BOOTSTRAP_EXPECTED_BLOCK = 10
# Block-length selection for the stationary bootstrap. "fixed" uses BOOTSTRAP_EXPECTED_BLOCK above.
# "politis_white" sets it data-drivenly via Politis-White (2004) automatic selection -- computed on
# the FHS STANDARDIZED residuals when conditional vol is on (the GARCH filter has already removed the
# volatility-clustering serial dependence, so the residuals admit a short block), or on the absolute
# raw innovations otherwise (so the block still captures volatility clustering). The joint-state cell
# prints both so the interaction is explicit. "fixed" keeps the engine byte-identical to v15.
BOOTSTRAP_BLOCK_MODE = "fixed"           # "fixed" or "politis_white"
TARGET_RATE_EXPLAINED_VAR = 0.995
TARGET_RESID_EXPLAINED_VAR = 0.95

# Conditional volatility (Filtered Historical Simulation) -- applies to the block_bootstrap generator.
# "none" leaves the engine unconditional and byte-identical to v13. "ewma_fhs" standardizes each joint
# FACTOR by its own time-varying EWMA/GARCH volatility, resamples the standardized residuals, and
# re-scales them by TODAY's conditional volatility, so the simulated risk is conditional on the current
# regime while the empirical fat tails are preserved. Conditional asset CORRELATION is not modelled
# directly: it emerges from the conditional volatility of the shared factors (when the common market
# factor's vol is high, every asset's loading on it scales up together, so correlations rise in
# stress). The variance mean-reverts to its unconditional level over ~VOL_REVERSION_HALFLIFE_DAYS, so
# today's vol matters at 1W but washes out by the 1Y horizon (and supplies the low-frequency variance
# mean reversion the daily VAR could not capture).
CONDITIONAL_VOL = "ewma_fhs"              # "none" or "ewma_fhs"
VOL_EWMA_LAMBDA = 0.94                    # RiskMetrics decay; news weight = 1 - lambda
VOL_REVERSION_HALFLIFE_DAYS = 34.0        # half-life of vol shocks toward the unconditional level
# FHS volatility-parameter mode. "fixed" applies the RiskMetrics-style constants above (news weight
# 1 - VOL_EWMA_LAMBDA, variance half-life VOL_REVERSION_HALFLIFE_DAYS) to EVERY factor -- robust,
# assumption-driven, and byte-identical to v16. "estimated" fits a GARCH(1,1) PER FACTOR by Gaussian
# quasi-ML with variance targeting (long-run variance pinned to each factor's sample variance): every
# factor gets its own news weight and its own persistence/half-life, so the market factor's
# slow-decaying volatility clustering and a high-order residual PC's absence of clustering are no
# longer forced onto one 34-day half-life -- the per-factor generalization of the heterogeneity the
# block-length arc exposed. Factors with no likelihood evidence of clustering (LR vs constant variance
# below ~6) fall back to constant variance automatically: conditioning only where the data supports
# it. Persistence is capped at 0.998 (~350-business-day variance half-life) because ~700 observations
# cannot distinguish 0.998 from 1 and the cap keeps the 1Y horizon mean-reverting. The joint-state
# cell prints the per-factor estimates; the backtest re-estimates per as-of (point-in-time, no
# look-ahead), which adds some runtime. "estimated_gjr" additionally allows a GJR leverage term per
# factor (Glosten-Jagannathan-Runkle): volatility may react MORE to negative shocks (gamma > 0), the
# equity stylized fact that acts specifically on the left tail -- i.e. on VaR and ES. The asymmetric
# term is kept only where it beats the symmetric fit by LR >= ~3.84 (chi-sq(1) 5%, indicative), so
# gamma is nonzero only where the data supports leverage; everything else behaves as "estimated".
VOL_PARAMS_MODE = "estimated_gjr"                 # "fixed", "estimated", or "estimated_gjr"
# Mean reversion for LEVEL blocks (Treasury yields and credit spreads). "none" keeps the pure
# random-walk cumulation of bootstrapped level changes -- byte-identical to v17, fine at 1W but known
# to overstate 1Y level variance, since rates and spreads mean-revert over long horizons for economic
# reasons (unlike equity returns, which are left untouched). "ou" fits a per-series Vasicek/OU
# discretization Delta y = a + b*y + eps on the sample levels and adds the estimated pull
# kappa*(theta - y) along every simulated path, replacing the RW drift for gated series. Gating is
# deliberately strict: a series mean-reverts only if the OLS t-stat of b is below OU_TSTAT_GATE
# (default -2.86, the Dickey-Fuller 5% critical value -- under the unit-root null the t-stat is not
# t-distributed and an ordinary -2 would find spurious reversion). Non-gated series keep the random
# walk exactly. Designed for the 1Y horizon; at 1W the pull is negligible by construction. On the
# short 3-year sample most series will (honestly) fail the gate -- SAMPLE_MODE="long" is what makes
# OU estimable and defensible: reversion only where the data support it.
LEVEL_MEAN_REVERSION = "ou"             # "none" or "ou"
OU_TSTAT_GATE = -2.86                     # Dickey-Fuller 5% critical value (with constant)

# Option A (parsimony): joint-state dynamics. "block_bootstrap" drops the VAR(1) autoregressive term
# and resamples the demeaned joint drivers in stationary blocks (a block-bootstrap of the joint
# innovations around their mean); "var1" keeps the fitted VAR(1) dynamics. A third mode, "t_copula"
# (v11), replaces the bootstrap with a fitted Student-t copula on the FACTOR innovations: marginals
# stay empirical but the dependence is a tail-dependent t-copula. A fourth mode, "asset_copula" (v12),
# fits the t-copula on the OBSERVABLES instead (equity returns, key-rate changes, spread changes, FX
# return) and generates them directly (reduced-form, no factor reconstruction) -- this is where the
# real cross-sectional correlation, and hence tail dependence, lives, since the PCA factors are
# orthogonal. The VAR diagnostic cell reports the one-step R^2 and both copulas' dof / tail
# dependence; the validation cell compares the equity co-tail across all generators. The backtest and
# out-of-sample draw honour this flag.

DRIVER_DYNAMICS = "block_bootstrap" # "block_bootstrap", "var1", "t_copula", or "asset_copula"

# Cross-asset factor structure (v24). Up to v23 each block is compressed on its own (equity APT factors
# + residual PCs, rate PCs, credit PCs, FX) and the four sets are stacked: nothing in the model is a
# cross-asset factor, and the dependence between blocks lives only in the co-dated rows the bootstrap
# resamples. "hierarchical" adds an explicit layer ON TOP of the same drivers (repricing untouched):
#     d_t = Lambda g_t + u_t,
# with k_G GLOBAL factors g that load on every block -- the linear combinations of the drivers whose
# removal leaves the least cross-block covariance -- and block-specific parts u. Three things become
# explicit, and each can be switched on alone:
#   * HIER_LOCAL_RESAMPLING="independent": g and each block's u are resampled on their OWN date streams,
#     so the blocks depend on each other ONLY through g. That is a restriction -- it can lose the
#     non-linear (tail, volatility) dependence the co-dated rows carried -- so it is testable.
#     "joint" keeps one date stream: the empirical joint distribution, rotated;
#   * with CONDITIONAL_VOL="ewma_fhs" each global factor carries ONE variance state (GJR on g); the
#     block-specific parts keep their own filters;
#   * GLOBAL_STRUCTURE_HALFLIFE_YEARS estimates the global structure on exponentially weighted moments
#     (the cross-asset structure of the CURRENT regime) and whitens the global factors' full history:
#     global shocks from any date, 2008 included, are transmitted through today's loadings -- the
#     stock-yield sign flip the v23 run measured. It acts through the independent resampling (with
#     "joint" the old regime's co-movement stays in the co-dated block parts).
# "blockwise" is v23, byte-identical. With "joint", CONDITIONAL_VOL="none" and no half-life the
# hierarchical engine reproduces the blockwise one exactly: it contains it as a special case. The
# structure (number of global factors, cross-block correlation left, per-series global / block /
# idiosyncratic variance shares) is printed in every mode as a pre-test.
FACTOR_STRUCTURE = "hierarchical"            # "blockwise" or "hierarchical"
GLOBAL_FACTORS = "auto"                   # "auto" or an int: number of global factors k_G
GLOBAL_FACTORS_MAX = 3                    # "auto": the smallest k_G <= this leaving no cross-block pair
GLOBAL_RESID_CORR_TOL = None              # with more than this unexplained correlation (None: noise floor,
                                          # max(0.05, 3.5 / sqrt(effective sample size)))
HIER_LOCAL_RESAMPLING = "independent"     # "independent" or "joint"

# v25b challenger lab. The Kalman/PIT-Kalman branches remain available in the utility code for
# forensic reproducibility but are no longer built or backtested.  The only live challenger asks
# whether estimating the hierarchy AFTER removing each original driver's GJR volatility improves
# cross-asset dependence without adding latent-state dynamics.
V25_BUILD_CHALLENGERS = True
V25_KALMAN_REFINEMENTS = 2              # dormant in v25b; retained for reproducibility
V25_KALMAN_AR_CAP = 0.98                # dormant in v25b
V25_KALMAN_R_FLOOR = 1e-4               # dormant in v25b
V25_PIT_CLIP = 1e-6                      # dormant in v25b
V25_CHALLENGER_NAMES = ("standardized_hierarchical",)
GLOBAL_STRUCTURE_HALFLIFE_YEARS = 1.5    # None = equal weights; e.g. 3.0 = current-regime structure
if FACTOR_STRUCTURE not in ("blockwise", "hierarchical"):
    raise ValueError(f"Unknown FACTOR_STRUCTURE {FACTOR_STRUCTURE!r}; expected 'blockwise' or 'hierarchical'")
if HIER_LOCAL_RESAMPLING not in ("independent", "joint"):
    raise ValueError(f"Unknown HIER_LOCAL_RESAMPLING {HIER_LOCAL_RESAMPLING!r}; expected 'independent' or 'joint'")
if FACTOR_STRUCTURE == "hierarchical" and DRIVER_DYNAMICS != "block_bootstrap":
    raise ValueError("FACTOR_STRUCTURE='hierarchical' resamples [g, u] and requires DRIVER_DYNAMICS='block_bootstrap'")

# Robust portfolio search (min-CVaR is now an exact LP; alpha/max_weight are the live knobs)
CVAR_ALPHA = 0.05
CVAR_MAX_WEIGHT = 0.50

# Walk-forward VaR backtest configuration (issue #2). This is the most expensive cell: it refits
# the engine at each as-of date on live data. Defaults are modest so it completes in reasonable
# time; increase BACKTEST_N_DATES (and re-run) for more statistical power in the coverage tests.
BACKTEST_N_DATES = 40
# Spacing between as-of dates. It must be at least the horizon, or consecutive forecast windows
# overlap: the violation sequence is then autocorrelated BY CONSTRUCTION and the chi2 laws behind the
# Kupiec / Christoffersen / Acerbi-Szekely tests no longer hold. Up to v20 the step was 5 days at every
# horizon, so the 1Y backtest compared 40 windows sharing 98% of their days -- about ONE independent
# observation, with p-values that were invalid rather than merely weak. At 1W this is 5, unchanged.
BACKTEST_STEP_BDAYS = max(5, HORIZON_BDAYS)
BACKTEST_N_SCENARIOS = 2_000

# Backtest SCOPE (v20). "recent" walks back BACKTEST_N_DATES from the as-of date -- the v19 panel,
# reproduced exactly. "full" walks the WHOLE aligned window from the first date with at least
# BACKTEST_MIN_TRAIN_DAYS of training data, which is what the long sample was bought for: with
# SAMPLE_MODE="long" + CREDIT_BLOCK_SOURCE="moody_2pt" the window spans ~2001-2026, so the coverage
# tests finally see 2008, 2011, 2020 and 2022 instead of one calm recent year. The estimation window
# stays EXPANDING and point-in-time (the engine is refit at each date on data up to that date only),
# so nothing here introduces lookahead. Cost: runtime is linear in the number of as-of dates and each
# date refits PCA + VAR + (GJR-)GARCH + OU -- "full" with a few hundred dates is minutes-to-an-hour,
# not seconds. If the eligible dates exceed BACKTEST_MAX_DATES they are thinned evenly (spacing
# grows from a base that is already at least the horizon, so the windows stay non-overlapping and the
# independence test stays valid -- a v20 claim that, before v21's horizon-aware step, held at 1W only).
BACKTEST_MODE = "recent"                # "recent" or "full"
BACKTEST_MIN_TRAIN_DAYS = 750             # "full" only: minimum training rows before the first as-of
BACKTEST_MAX_DATES = 260                  # "full" only: cap on as-of dates (None = no cap)
# "recent" only: at 1M/3M, walking back BACKTEST_N_DATES non-overlapping windows reaches years before
# the short sample begins; an as-of date whose point-in-time window would hold less than this many
# rows is skipped (a refit on less than a trading year is not a model). No 1W date is affected.
BACKTEST_MIN_REFIT_DAYS = 252

# Multi-horizon backtest (v21). None (default) backtests HORIZON_MODE only. A tuple such as
# ("1W", "1M", "3M", "1Y") refits the engine ONCE per as-of date and simulates every listed horizon
# from that same refit, each on its own NON-overlapping sub-grid of the as-of dates (every k-th date,
# k = ceil(horizon / grid spacing)): coverage as a function of the horizon, from a single run, at the
# cost of one refit per grid date. The grid spacing is set by the SHORTEST listed horizon, and
# HORIZON_MODE must be in the tuple (it drives the headline summary and the regime / year slices).
BACKTEST_HORIZONS = None

# Expected-shortfall backtest (Acerbi-Szekely 2014). The portfolio layer OPTIMIZES CVaR but the
# coverage tests only validated VaR -- v20 closes that gap. ES is not elicitable on its own, so the
# test statistic is standardized by the model's own ES and its null distribution is obtained by
# simulation from the same predictive scenario sets the engine produced at each as-of date.
ES_BACKTEST_N_SIM = 2_000                 # Monte-Carlo draws for the Z-statistic null distribution

# Multi-asset books in the walk-forward backtest (v24). Up to v23 it validated ONE book, equal-weight
# equity: nothing cross-asset -- the stock-bond hedge, the credit-equity link, the min-CVaR allocation
# (76% corporates on the v23 run) -- had ever been confronted with realized history. Each listed book is
# priced IDENTICALLY on the simulated and on the realized horizon moves, so a breach is a model error,
# not a pricing mismatch:
#   "equity_ew"        -- the v20-v23 headline book (mean of single-name log returns); always listed;
#   "stock_bond_40_60" -- 40% equal-weight equity + 60% constant-maturity 10Y Treasury (simple returns);
#   "credit_40_20_40"  -- 40% equity + 20% Treasury + 40% a 10Y corporate at the 10Y yield plus the
#                         credit driver's representative spread (Baa - DGS10 with moody_2pt, BBB OAS
#                         otherwise);
#   "min_cvar"         -- the long-only min-CVaR LP re-solved at EVERY as-of date on that date's scenarios
#                         (10 stocks + Treasury + corporate; CVAR_ALPHA, CVAR_MAX_WEIGHT): weights chosen on
#                         half the scenarios, VaR/ES read on the other half, so the optimizer cannot grade
#                         its own selection noise.
# Bonds: a par bond bought at the starting yield, repriced at the horizon yield with maturity N - h,
# plus the coupon carried (constant-maturity approximation, Swinkels 2019); no default or migration leg.
BACKTEST_PORTFOLIOS = ("equity_ew", "stock_bond_40_60", "credit_40_20_40", "min_cvar")
BACKTEST_REGIME_SPLIT = "2022-01-01"      # the books are also reported before / from this date
# v25c challenger list is frozen for forensic reproducibility; v26 uses its own 2x2 architecture lab.
BACKTEST_STRUCTURES = None

# v26 fair architecture lab ---------------------------------------------------------------
# Four-way factorial:
#   B_S = blockwise_static          legacy equal-weight within-block geometry
#   B_R = blockwise_adaptive        EW means / weighted OLS / weighted PCA within each block
#   H_S = hierarchical_static       static blocks + static global/local hierarchy
#   H_R = hierarchical_adaptive     adaptive blocks + adaptive global/local hierarchy
#
# The adaptive pair ALWAYS receives the same half-life at the same as-of date and forecast horizon.
# This prevents hyperparameter tuning from being mistaken for an architectural win.
V26_STRUCTURES = (
    "blockwise_static",
    "blockwise_adaptive",
    "hierarchical_static",
    "hierarchical_adaptive",
)
V26_PRIMARY_STRUCTURE = "hierarchical_adaptive"

# Half-life selection:
#   "shared_fixed"  -> use the horizon schedule below, identical across adaptive architectures.
#   "shared_nested" -> at each outer as-of/horizon, choose ONE H using only earlier raw cross-asset
#                      outcomes, then give that same H to both adaptive architectures.
# Headline research mode is nested; switch to shared_fixed for a fast smoke test.
V26_HALFLIFE_MODE = "shared_nested"
V26_FIXED_HALFLIFE_YEARS_BY_HORIZON = {"1W": 1.5, "1M": 1.5, "3M": 1.5, "1Y": 1.5}
V26_HALFLIFE_GRID_YEARS = (0.25, 0.50, 1.0, 1.5, 3.0, None)   # None = static/equal weights
V26_FALLBACK_HALFLIFE_YEARS = 1.5
V26_HALFLIFE_MIN_TRAIN_DAYS = 504
V26_HALFLIFE_MAX_INNER_ORIGINS = 60
V26_HALFLIFE_ONE_SE_RULE = True

# Treasury note setup
COUPON_RATE = 0.02
FACE_VALUE = 100.0
PAY_FREQ = 2

# Corporate bonds priced off Treasury-zero + the simulated OAS for their rating bucket. Coupons are
# set near prevailing same-rating yields so prices start close to par; what matters for the demo is
# that they reprice through BOTH the rate channel and the (equity-linked) credit-spread channel.
CORP_BONDS = [
    {"name": "CORP_BBB_7Y", "rating": "BBB", "coupon": 0.052, "maturity_years": 7},
    {"name": "CORP_BB_5Y", "rating": "BB", "coupon": 0.068, "maturity_years": 5},
]

REPRESENTATIVE_EQUITY = TICKERS[0]
REPRESENTATIVE_RATING = "BBB"

# Credit migration / default (CreditMetrics-style), material from 1M on. Over a month, a quarter or a
# year a corporate can be downgraded, upgraded, or DEFAULT, which a spread-MtM repricing ignores. Each
# bond's horizon rating is drawn from the horizon transition matrix via a threshold (Gaussian-copula)
# model in which every name's migration latent variable shares a SYSTEMATIC credit factor (the
# simulated credit state), so downgrades and defaults CLUSTER in the scenarios where credit blows out
# -- the same wrong-way dependence the spread channel already carries. Migrated bonds reprice on the
# new rating's OAS curve; defaulted bonds are worth RECOVERY_RATE of face. The matrix is one-year; at
# 1M/3M it is scaled through its generator, P(h) = exp(h Q) (transition_matrix_for_horizon), and at 1Y
# it is used as is. At 1W migration stays off (the one-week BB downgrade probability is ~0.3%) and the
# corporates reprice exactly as before.
CREDIT_MIGRATION_ENABLED = (HORIZON_MODE != "1W")
RECOVERY_RATE = 0.40                         # recovery on default, as a fraction of face value
MIGRATION_FACTOR_LOADING = 0.30              # sqrt asset correlation: each name's latent = loading*M + sqrt(1-loading^2)*idio
TRANSITION_RATINGS = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC", "D"]   # D = default (absorbing)

# FX block: currency-pair spot rates from FRED (long history, no truncation). DEXUSEU is USD per
# euro, so a long-EUR position gains when it rises. The block is written to take a *list* of pairs
# but is configured here with the single EUR/USD pair (the lightest FX exposure); add FRED codes to
# the mapping to extend it (e.g. DEXJPUS / DEXUSUK), at which point the PCA starts to compress.
FX_SERIES = {
    "DEXUSEU": "EURUSD",   # US dollars per euro
}
TARGET_FX_EXPLAINED_VAR = 0.99
FX_MAX_FACTORS = None       # cap on retained FX factors; None = one component budget per pair

# FX-exposed instrument: a long-EUR forward struck at-the-money at the valuation date, so its MTM is
# zero at inception and the horizon P&L is pure FX (spot move) plus USD discounting. Numeraire is
# USD. `foreign_funding_rate` is the EUR funding rate, held CONSTANT over the horizon (covered
# interest parity) as a declared approximation: the forward-points / carry term is then fixed and at
# one week immaterial (order a few bp) next to ~100-150 bp of weekly spot volatility; at 3M-1Y it is
# a real simplification (a simulated foreign curve would be the fix). When the horizon reaches the
# forward's tenor, the maturity is anchored PAST the horizon so the tenor is still outstanding there.
# notional_foreign is
# in EUR; the USD notional (the capital base used to normalize the forward's return) is
# notional_foreign * spot.
FX_FORWARDS = [
    {"name": "FXFWD_EURUSD_3M", "pair": "EURUSD", "maturity_years": 0.25,
     "foreign_funding_rate": 0.020, "notional_foreign": 100.0},
]

# Crisis-tail overlay. The OAS / equity / FX windows are only ~3 calm years (the ICE BofA truncation),
# so the empirical bootstrap tail is calibrated to a period with no real crisis. Moody's seasoned
# Baa/Aaa yields (DBAA/DAAA) carry decades including every postwar crisis and are used to characterize
# how often and how severely credit blows out. The crisis overlay is a JOINT two-state regime applied
# in the simulator (a stress step amplifies the whole joint innovation, so a crisis co-moves all four
# asset classes). Two modes are provided:
#   * "regime_jump"    -- parametric: amplify by CRISIS_SCALE on a stress step.
#   * "moody_backbone" -- empirical: amplify by a real historical Baa-Aaa stress-day severity.
# CRISIS_MODE = "none" (default) leaves the engine byte-identical to the calm-window v9; the calm
# validation still matches the sample. Turning a crisis mode ON deliberately makes the simulated tail
# FATTER than the calm sample -- that is the point. Set CRISIS_PROB / CRISIS_SCALE to None to use the
# Moody's-calibrated values, or to a number to override them.
MOODYS_SERIES = {
    "DAAA": "Aaa",
    "DBAA": "Baa",
}
MOODYS_START_DATE = "1986-01-01"        # long history; FRED daily Moody's series start in the mid-1980s
CRISIS_SIGMA_THRESHOLD = 4.0            # a daily Baa-Aaa move beyond this many calm-sigma is a stress day
CRISIS_MODE = "none"                    # "none" | "regime_jump" | "moody_backbone"
CRISIS_PROB = None                      # None -> use the Moody's-calibrated stress-day frequency
CRISIS_SCALE = None                     # None -> use the Moody's-calibrated stress/calm volatility ratio


In [ ]:
# Utilities: data loading, factor preparation, risk-driver extraction, scenario generation, repricing, portfolio analytics

def download_market_data(tickers, market_proxy, rate_tickers, start_date, download_end):
    """Download adjusted-close equity prices, a market proxy, and FRED Treasury key rates."""
    all_equity_tickers = list(dict.fromkeys(list(tickers) + [market_proxy]))
    stock_raw = yf.download(
        all_equity_tickers,
        start=start_date,
        end=download_end,
        auto_adjust=False,
        progress=False,
        actions=False,
    )

    if isinstance(stock_raw.columns, pd.MultiIndex):
        if "Adj Close" not in stock_raw.columns.get_level_values(0):
            raise ValueError("Adjusted close prices not found in yfinance download output.")
        prices_all = stock_raw["Adj Close"].copy()
    else:
        prices_all = stock_raw.copy()
        if isinstance(prices_all, pd.Series):
            prices_all = prices_all.to_frame(name=all_equity_tickers[0])
        prices_all.columns = all_equity_tickers

    stock_prices = prices_all[tickers].copy()
    market_proxy_prices = prices_all[[market_proxy]].copy()
    rates_raw = web.DataReader(rate_tickers, "fred", start_date, download_end).astype(float) / 100.0
    return stock_prices, market_proxy_prices, rates_raw


def download_credit_oas(credit_oas_series, start_date, download_end):
    """Download ICE BofA option-adjusted credit spreads (OAS) by rating bucket from FRED.

    `credit_oas_series` is an ordered mapping {FRED_code: rating_label} sorted by ascending credit
    risk (AAA -> CCC), so the resulting columns form a credit-quality curve. FRED reports OAS in
    *percent*, so we divide by 100 to express spreads in decimals, consistent with the Treasury
    key-rate convention (a 1.50% OAS becomes 0.0150).

    NOTE ON DATA AVAILABILITY: as of April 2026 the ICE BofA OAS series on FRED are licensed to
    expose only a rolling ~3 years of history. The credit block therefore lives on a recent window
    (see START_DATE / AS_OF_DATE in the config cell). For a longer sample one would substitute a
    non-truncated source (e.g. Moody's seasoned Aaa/Baa yields DAAA/DBAA differenced against a
    Treasury tenor), at the cost of a coarser two-point credit curve.
    """
    codes = list(credit_oas_series.keys())
    oas = web.DataReader(codes, "fred", start_date, download_end).astype(float) / 100.0
    oas = oas.rename(columns=dict(credit_oas_series))
    # preserve the ascending-risk column order
    oas = oas[[credit_oas_series[c] for c in codes]]
    return oas


def download_fx_rates(fx_series, start_date, download_end):
    """Download FX spot rates from FRED (one column per currency pair).

    `fx_series` is an ordered mapping {FRED_code: pair_label}. FRED quotes these pairs as
    'US dollars per 1 unit of foreign currency' (e.g. DEXUSEU = USD per EUR), so each column is the
    price in USD of one unit of the foreign currency and a long-foreign position gains when the
    column rises. Unlike the ICE BofA credit series these spot series carry long history with no
    3-year truncation.
    """
    codes = list(fx_series.keys())
    fx = web.DataReader(codes, "fred", start_date, download_end).astype(float)
    fx = fx.rename(columns=dict(fx_series))
    fx = fx[[fx_series[c] for c in codes]]
    return fx


def compute_fx_log_returns(fx_rates):
    return np.log(fx_rates / fx_rates.shift(1)).dropna()


def download_moodys_credit(moodys_series, start_date, download_end):
    """Download Moody's seasoned corporate bond yields from FRED (long history, no 3-year truncation).

    `moodys_series` maps FRED codes to labels, e.g. {"DAAA": "Aaa", "DBAA": "Baa"}. These daily
    seasoned-yield series run back to the 1980s and carry every postwar credit crisis, so the
    Baa-Aaa spread is used as a long-history credit-stress backbone to calibrate how often and how
    severely credit blows out relative to the short, calm OAS window the main engine is fit on.
    Yields are in percent on FRED; divided by 100 to decimals here.
    """
    codes = list(moodys_series.keys())
    mood = web.DataReader(codes, "fred", start_date, download_end).astype(float) / 100.0
    mood = mood.rename(columns=dict(moodys_series))
    return mood


def compute_credit_crisis_calibration(moodys, horizon_bdays, aaa_label="Aaa", baa_label="Baa",
                                      calm_lookback_years=3, calm_sigma_threshold=4.0):
    """Calibrate empirical credit-crisis statistics from the long Moody's Baa-Aaa history.

    The Baa-Aaa seasoned-yield spread is a coarse (two-point, investment-grade) but very long credit
    risk proxy. We mark a day as a credit-stress day when its daily Baa-Aaa change exceeds
    `calm_sigma_threshold` times the volatility of a recent calm window, then summarize crises by:
      * crisis_prob  -- empirical fraction of days that are credit-stress days (a per-step crisis
        probability for the regime overlay);
      * crisis_scale -- ratio of stress-day to calm-day volatility (how much bigger crisis moves are);
      * dof          -- a Student-t tail index from the full-history excess kurtosis (6/(dof-4));
      * stress_magnitude_pool -- the realized stress-day move sizes (in multiples of the calm std),
        i.e. *real* historical crisis severities, used by the "moody_backbone" simulator mode.

    This is a credit-derived, joint stress calibration: real crises move all asset classes together,
    so the regime overlay applies the scale to the whole joint innovation, not credit alone. The
    coarse IG proxy understates how much high-yield blows out in a crisis (a documented limitation);
    the rating-bucket loadings still apply more of the systematic move to HY, but a true crisis would
    widen HY convexly beyond what the calm-window loadings imply.
    """
    baa_aaa = (moodys[baa_label] - moodys[aaa_label]).dropna()
    dchg = baa_aaa.diff().dropna()
    calm_start = dchg.index.max() - pd.DateOffset(years=calm_lookback_years)
    calm = dchg[dchg.index >= calm_start]
    calm_std = float(calm.std()) if len(calm) > 5 else float(dchg.std())
    if not np.isfinite(calm_std) or calm_std <= 0:
        calm_std = float(dchg.std())

    thresh = calm_sigma_threshold * calm_std
    stress_mask = dchg.abs() > thresh
    n_stress = int(stress_mask.sum())
    crisis_prob = float(stress_mask.mean())
    if n_stress > 5:
        stress_std = float(dchg[stress_mask].std())
        stress_pool = (dchg[stress_mask].abs() / calm_std).to_numpy()
    else:
        stress_std = calm_std * 5.0
        stress_pool = np.array([5.0])
    crisis_scale = stress_std / calm_std if calm_std > 0 else 5.0
    ek = float(dchg.kurtosis())
    dof = float(4.0 + 6.0 / ek) if ek > 0.1 else 8.0
    return {
        "baa_aaa_spread": baa_aaa,
        "daily_changes": dchg,
        "calm_std": calm_std,
        "stress_std": stress_std,
        "crisis_prob": crisis_prob,
        "crisis_scale": crisis_scale,
        "dof": dof,
        "stress_magnitude_pool": stress_pool,
        "n_stress_days": n_stress,
        "n_total_days": int(len(dchg)),
        "history_start": dchg.index.min(),
        "history_end": dchg.index.max(),
    }


def _normalize_ff_columns(columns):
    mapping = {}
    for col in columns:
        cleaned = re.sub(r"[^A-Za-z0-9]+", "_", str(col).strip()).upper().strip("_")
        mapping[col] = cleaned
    return mapping


def _ff_table_to_returns(df):
    """Convert raw Fama-French / momentum tables to *simple daily returns* (decimals).

    Issue #7 fix: the FF series are excess / long-short spread returns (MKT-RF, SMB, HML, MOM).
    Applying log1p to a zero-cost long-short spread has no clean interpretation and is
    inconsistent with combining the factor contributions *additively* in the excess-return
    reconstruction. We therefore keep them as simple, additive daily returns. At the daily scale
    log1p(x) ~= x, so this does not move the numbers materially; it only removes the conceptual
    mismatch. (RF is also a level return and is left in the same simple-return convention so that
    'equity excess = equity log-return - RF' uses exactly the same risk-free that MKT-RF embeds.)
    """
    out = df.copy()
    out.index = pd.to_datetime(out.index.astype(str))
    out = out.apply(pd.to_numeric, errors="coerce")
    # Ken French's daily files are ALWAYS in percent -- both through the pandas_datareader
    # famafrench reader and through the raw CSV zips -- so the conversion is deterministic.
    # (The previous per-frame median heuristic sat almost exactly on the momentum factor's
    # median |daily return| of ~0.5%: the momentum table is converted ALONE, so on long windows
    # it could silently stay in percent, entering the joint state at 100x scale. The OLS betas
    # absorb a pure column rescaling, which is why results stayed sane, but the landmine is
    # removed rather than relied upon.)
    out = out / 100.0
    _med = float(out.abs().median().max())
    if _med > 0.05:
        raise ValueError(
            f"Fama-French factors look mis-scaled after /100 (max median |daily| = {_med:.4f}); "
            "the source no longer appears to be in percent -- inspect it before proceeding."
        )
    return out


def _ff_zip_to_frame(url):
    """Parse a Ken French daily CSV zip directly (fallback when pandas_datareader's famafrench reader
    fails, e.g. on parser/dtype incompatibilities with newer pandas). Keeps only the daily table
    (rows starting with an 8-digit date), robust to preamble, annual tables, and copyright footer."""
    import io as _io, zipfile as _zipfile, urllib.request as _urlreq
    raw = _urlreq.urlopen(url, timeout=60).read()
    txt = _zipfile.ZipFile(_io.BytesIO(raw)).read(
        _zipfile.ZipFile(_io.BytesIO(raw)).namelist()[0]).decode("latin-1").splitlines()
    first = next(i for i, ln in enumerate(txt) if re.match(r"^\s*\d{8}\s*,", ln))
    header = ["DATE"] + [h.strip() for h in txt[first - 1].split(",")[1:]]
    data = []
    for ln in txt[first:]:
        if not re.match(r"^\s*\d{8}\s*,", ln):
            break
        data.append([x.strip() for x in ln.split(",")])
    df = pd.DataFrame(data, columns=header[:len(data[0])]).set_index("DATE")
    df.index = pd.to_datetime(df.index, format="%Y%m%d")
    return df.apply(pd.to_numeric, errors="coerce")


def download_fama_french_daily_factors(start_date, end_date):
    """Download daily Fama-French market/style factors plus the daily momentum factor.

    The series are converted to simple daily returns (see `_ff_table_to_returns`) so they can be
    combined additively with the log-return machinery used elsewhere in the notebook. If the
    pandas_datareader famafrench reader fails (parser/dtype issues appear with some pandas versions),
    the function falls back to fetching and parsing Ken French's CSV zips directly.
    """
    try:
        ff3 = web.DataReader("F-F_Research_Data_Factors_daily", "famafrench", start=start_date)[0]
        mom = web.DataReader("F-F_Momentum_Factor_daily", "famafrench", start=start_date)[0]
    except Exception:
        _base = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
        ff3 = _ff_zip_to_frame(_base + "F-F_Research_Data_Factors_daily_CSV.zip").loc[pd.Timestamp(start_date):]
        mom = _ff_zip_to_frame(_base + "F-F_Momentum_Factor_daily_CSV.zip").loc[pd.Timestamp(start_date):]

    ff3 = ff3.rename(columns=_normalize_ff_columns(ff3.columns))
    mom = mom.rename(columns=_normalize_ff_columns(mom.columns))

    ff3 = _ff_table_to_returns(ff3)
    mom = _ff_table_to_returns(mom)

    mom_col = None
    for candidate in ["MOM", "MOMENTUM", "MOM_FACTOR"]:
        if candidate in mom.columns:
            mom_col = candidate
            break
    if mom_col is None:
        mom_col = mom.columns[0]

    ff = ff3.join(mom[[mom_col]].rename(columns={mom_col: "MOM"}), how="left")

    keep = [c for c in ["MKT_RF", "SMB", "HML", "RF", "MOM"] if c in ff.columns]
    ff = ff[keep].sort_index().loc[:pd.Timestamp(end_date)]
    return ff.dropna(how="any")


def choose_effective_asof(stock_prices, market_proxy_prices, rates, ff_factors, as_of_date, credit=None, fx=None):
    common_dates = stock_prices.index.intersection(market_proxy_prices.index).intersection(rates.index).intersection(ff_factors.index)
    if credit is not None:
        common_dates = common_dates.intersection(credit.index)
    if fx is not None:
        common_dates = common_dates.intersection(fx.index)
    eligible = common_dates[common_dates <= as_of_date]
    if len(eligible) == 0:
        raise ValueError("No common market date available at or before the chosen AS_OF_DATE.")
    return eligible.max()


def align_market_data(stock_prices, market_proxy_prices, rates, ff_factors, effective_date, credit=None, fx=None):
    """Align all blocks on their common dates up to the effective date.

    Returns a 6-tuple (prices, market, yields, ff, credit, fx); `credit` and `fx` are None when the
    corresponding frame is not supplied, so the narrower (equity-rates, or equity-rates-credit)
    paths are unchanged.
    """
    prices = stock_prices.loc[:effective_date].dropna(how="any")
    market = market_proxy_prices.loc[:effective_date].dropna(how="any")
    yields = rates.loc[:effective_date].dropna(how="any")
    ff = ff_factors.loc[:effective_date].dropna(how="any")
    common_dates = prices.index.intersection(market.index).intersection(yields.index).intersection(ff.index)
    cr = None
    if credit is not None:
        cr = credit.loc[:effective_date].dropna(how="any")
        common_dates = common_dates.intersection(cr.index)
    fxr = None
    if fx is not None:
        fxr = fx.loc[:effective_date].dropna(how="any")
        common_dates = common_dates.intersection(fxr.index)
    if cr is not None:
        cr = cr.loc[common_dates]
    if fxr is not None:
        fxr = fxr.loc[common_dates]
    return (prices.loc[common_dates], market.loc[common_dates], yields.loc[common_dates],
            ff.loc[common_dates], cr, fxr)


def compute_equity_log_returns(prices):
    return np.log(prices).diff().dropna()


def compute_yield_changes(yields):
    return yields.diff().dropna()


def compute_spread_changes(spreads):
    """Daily changes in credit OAS levels (decimals), analogous to yield changes."""
    return spreads.diff().dropna()


def compute_excess_log_returns_daily(log_returns, rf_daily):
    """Excess log returns using a *daily* risk-free series (already per-day, in decimals).

    Issue #6 fix: equity excess returns now use the SAME daily risk-free that defines the
    Fama-French factors (the FF 'RF' column), instead of DGS2/252. This removes the double
    risk-free convention (DGS2 for the equity excess vs the FF RF embedded inside MKT-RF) and
    stops using a 2Y point as an overnight-rate proxy.
    """
    rf = rf_daily.reindex(log_returns.index).ffill()
    excess = log_returns.sub(rf, axis=0)
    return excess, rf


def compute_market_excess_log_returns_daily(market_prices, rf_daily):
    market_log_returns = np.log(market_prices.squeeze()).diff().dropna()
    rf = rf_daily.reindex(market_log_returns.index).ffill()
    market_excess = market_log_returns.sub(rf, axis=0)
    market_excess.name = f"{market_prices.columns[0]}_excess_log_return"
    return market_excess


def plot_time_series(df, title, ylabel=None):
    ax = df.plot(figsize=(11, 6), lw=1.2)
    ax.set_title(title)
    if ylabel is not None:
        ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    plt.show()


def plot_histogram(series, title, xlabel):
    ax = pd.Series(series).plot(kind="hist", bins=60, alpha=0.75)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.grid(True, alpha=0.3)
    plt.show()


def distribution_summary(series):
    s = pd.Series(series, dtype=float).dropna()
    return pd.Series({
        "mean": s.mean(),
        "std": s.std(),
        "q01": s.quantile(0.01),
        "q05": s.quantile(0.05),
        "median": s.median(),
        "q95": s.quantile(0.95),
        "q99": s.quantile(0.99),
        "min": s.min(),
        "max": s.max(),
    })


def loss_summary_from_pnl(series, alpha=0.05):
    s = pd.Series(series, dtype=float).dropna()
    threshold = s.quantile(alpha)
    tail = s[s <= threshold]
    return pd.Series({
        "mean": s.mean(),
        "std": s.std(),
        "q01": s.quantile(0.01),
        "q05": s.quantile(0.05),
        "median": s.median(),
        "q95": s.quantile(0.95),
        "q99": s.quantile(0.99),
        "VaR_95_loss": -threshold,
        "ES_95_loss": -tail.mean() if len(tail) > 0 else np.nan,
    })


def _compress_residual_block(residuals, resid_explained_var_target=0.95):
    resid_pca_full = PCA().fit(residuals)
    resid_cumvar = np.cumsum(resid_pca_full.explained_variance_ratio_)
    k_resid = int(np.searchsorted(resid_cumvar, resid_explained_var_target) + 1)
    k_resid = max(1, min(k_resid, residuals.shape[1] - 1))

    resid_pca = PCA(n_components=k_resid)
    resid_scores = pd.DataFrame(
        resid_pca.fit_transform(residuals),
        index=residuals.index,
        columns=[f"resid_pc{i+1}" for i in range(k_resid)],
    )

    resid_loadings = pd.DataFrame(
        resid_pca.components_.T,
        index=residuals.columns,
        columns=resid_scores.columns,
    )

    resid_explained_ratio = pd.Series(
        resid_pca.explained_variance_ratio_,
        index=resid_scores.columns,
        name="resid_explained_variance_ratio",
    )

    # Issue #11: PCA on the residuals discards the variance beyond the retained components. That
    # dropped idiosyncratic variance never re-enters the simulation, so single-name tails would be
    # understated. We measure, per asset, the standard deviation of the part of the residual NOT
    # captured by the retained PCs, and re-inject it as iid noise in the simulator.
    reconstructed = resid_scores.to_numpy() @ resid_loadings.to_numpy().T
    leftover = residuals.to_numpy() - reconstructed
    resid_idiosyncratic_std = pd.Series(
        leftover.std(axis=0, ddof=1),
        index=residuals.columns,
        name="resid_idiosyncratic_std",
    )

    return {
        "resid_scores": resid_scores,
        "resid_loadings": resid_loadings,
        "resid_explained_ratio": resid_explained_ratio,
        "resid_cum_explained_ratio": resid_explained_ratio.cumsum(),
        "resid_idiosyncratic_std": resid_idiosyncratic_std,
        "resid_pca": resid_pca,
    }


def build_factor_driven_equity_block(excess_returns, factor_df, block_name, resid_explained_var_target=0.95):
    common_dates = excess_returns.index.intersection(factor_df.index)
    y = excess_returns.loc[common_dates].copy()
    f = factor_df.loc[common_dates].copy()

    mu_excess = y.mean()
    mu_factors = f.mean()
    centered_y = y - mu_excess
    centered_f = f - mu_factors

    X = centered_f.to_numpy()
    Y = centered_y.to_numpy()
    beta_array = np.linalg.lstsq(X, Y, rcond=None)[0].T  # assets x factors

    betas = pd.DataFrame(beta_array, index=y.columns, columns=centered_f.columns)
    fitted = pd.DataFrame(X @ beta_array.T, index=y.index, columns=y.columns)
    residuals = centered_y - fitted
    asset_r2 = 1.0 - residuals.var(ddof=1) / centered_y.var(ddof=1)

    residual_block = _compress_residual_block(residuals, resid_explained_var_target)

    return {
        "engine_name": block_name,
        "factors": centered_f,
        "factor_means": mu_factors,
        "betas": betas,
        "residuals": residuals,
        "asset_r2": asset_r2,
        "mean_excess_returns": mu_excess,
        "excess_returns": y,
        **residual_block,
    }


def build_latent_statistical_equity_block(log_returns, rf_daily, resid_explained_var_target=0.95):
    excess_returns, short_rate_daily = compute_excess_log_returns_daily(log_returns, rf_daily)
    centered = excess_returns - excess_returns.mean()
    pca = PCA(n_components=1)
    factor_scores = pd.DataFrame(
        pca.fit_transform(centered),
        index=centered.index,
        columns=["latent_equity_factor"],
    )
    block = build_factor_driven_equity_block(
        excess_returns=excess_returns,
        factor_df=factor_scores,
        block_name="latent_statistical",
        resid_explained_var_target=resid_explained_var_target,
    )
    block["short_rate_daily"] = short_rate_daily.reindex(block["excess_returns"].index)
    block["latent_explained_variance"] = float(pca.explained_variance_ratio_[0])
    return block


def build_proxy_market_equity_block(log_returns, market_proxy_prices, rf_daily, resid_explained_var_target=0.95):
    excess_returns, short_rate_daily = compute_excess_log_returns_daily(log_returns, rf_daily)
    market_excess = compute_market_excess_log_returns_daily(market_proxy_prices, rf_daily)
    factor_df = market_excess.to_frame(name="market_erp_proxy")
    block = build_factor_driven_equity_block(
        excess_returns=excess_returns,
        factor_df=factor_df,
        block_name="proxy_market",
        resid_explained_var_target=resid_explained_var_target,
    )
    block["short_rate_daily"] = short_rate_daily.reindex(block["excess_returns"].index)
    block["market_excess"] = market_excess.reindex(block["excess_returns"].index)
    return block


def build_apt_equity_block(log_returns, rf_daily, ff_factor_panel, selected_factors, block_name, resid_explained_var_target=0.95):
    excess_returns, short_rate_daily = compute_excess_log_returns_daily(log_returns, rf_daily)
    factor_df = ff_factor_panel[selected_factors].copy()
    block = build_factor_driven_equity_block(
        excess_returns=excess_returns,
        factor_df=factor_df,
        block_name=block_name,
        resid_explained_var_target=resid_explained_var_target,
    )
    block["short_rate_daily"] = short_rate_daily.reindex(block["excess_returns"].index)
    block["selected_factors"] = list(selected_factors)
    return block


def summarize_equity_engine(block):
    return pd.Series({
        "n_equity_factors": block["factors"].shape[1],
        "n_residual_factors": block["resid_scores"].shape[1],
        "mean_asset_R2": block["asset_r2"].mean(),
        "median_asset_R2": block["asset_r2"].median(),
        "max_asset_R2": block["asset_r2"].max(),
        "min_asset_R2": block["asset_r2"].min(),
    }, name=block["engine_name"])


def _build_change_factor_model(changes, explained_var_target, prefix, mean_key, max_factors=None):
    """Generic PCA factor model for a panel of daily changes (yield changes or spread changes).

    Shared by the rates curve and the credit-spread curve (issue: a single asset-class adds a new
    risk-driver block of exactly this shape). Returns scores/loadings/mean plus explained variance.

    `max_factors` (Option A, credit parsimony) hard-caps the retained factors. The rating-bucket OAS
    curve is highly collinear, so its PCA is near-degenerate: keeping all buckets injects ~7 weakly
    identified driver dimensions into an already large joint VAR. Capping to a few systematic factors
    (PC1 ~ credit level, PC2 ~ IG-HY slope, PC3 ~ curvature) keeps essentially all the variance while
    removing parameters; PC1 is also the systematic credit factor a later 1-year CreditMetrics view
    would reuse. The floor is 2 so level + slope are always present.
    """
    mean_chg = changes.mean()
    centered = changes - mean_chg

    pca_full = PCA().fit(centered)
    cumvar = np.cumsum(pca_full.explained_variance_ratio_)
    k = int(np.searchsorted(cumvar, explained_var_target) + 1)
    k = max(2, min(k, centered.shape[1]))
    if max_factors is not None:
        k = max(2, min(k, int(max_factors)))

    pca = PCA(n_components=k)
    scores = pd.DataFrame(
        pca.fit_transform(centered),
        index=centered.index,
        columns=[f"{prefix}_pc{i+1}" for i in range(k)],
    )
    loadings = pd.DataFrame(
        pca.components_.T,
        index=centered.columns,
        columns=scores.columns,
    )
    explained_ratio = pd.Series(
        pca.explained_variance_ratio_,
        index=scores.columns,
        name="explained_variance_ratio",
    )
    # What the k retained factors do NOT reproduce, day by day (v22): the compression residual. Kept
    # with the block; the simulator only uses it when CREDIT_RESIDUAL_MODE="restore" attaches it.
    residuals = centered - scores.to_numpy() @ loadings.to_numpy().T
    return {
        "scores": scores,
        "loadings": loadings,
        mean_key: mean_chg,
        "explained_ratio": explained_ratio,
        "cum_explained_ratio": explained_ratio.cumsum(),
        "n_factors": k,
        "pca": pca,
        "residuals": residuals,
    }


def build_rates_factor_model(yield_changes, explained_var_target=0.995):
    return _build_change_factor_model(yield_changes, explained_var_target, prefix="rate", mean_key="mean_dy")


def build_credit_factor_model(spread_changes, explained_var_target=0.99, max_factors=None):
    """PCA factor model of the credit-spread curve (issue: credit block; Option A: parsimony cap).

    The rating-bucket OAS curve compresses to a few factors with a clean interpretation: pc1 is the
    overall credit-risk level (spreads widening/tightening together), pc2 the IG-vs-HY slope, etc.
    These credit factors are stacked into the joint state so the equity-credit (Merton-type)
    dependence is carried by the joint innovation bootstrap. `max_factors` caps the retained factors
    (Option A) so the highly collinear curve does not inflate the joint VAR with near-degenerate
    dimensions; the buckets are still all reconstructed from the few factors via the loadings.
    """
    return _build_change_factor_model(spread_changes, explained_var_target, prefix="credit",
                                      mean_key="mean_dspread", max_factors=max_factors)


def build_curve_lsc_factors(key_rates):
    """Interpretable level / slope / curvature proxies, generalized to whatever tenors are present.

    Issue #8: with a richer curve we no longer hard-code DGS2/DGS5/DGS10. Assuming the columns are
    in ascending tenor order: level = average yield, slope = longest - shortest,
    curvature = 2*mid - shortest - longest.
    """
    cols = list(key_rates.columns)
    short, long = cols[0], cols[-1]
    mid = cols[len(cols) // 2]
    level = key_rates.mean(axis=1)
    slope = key_rates[long] - key_rates[short]
    curvature = 2.0 * key_rates[mid] - key_rates[short] - key_rates[long]
    return pd.DataFrame({
        "curve_level": level,
        "curve_slope": slope,
        "curve_curvature": curvature,
    }, index=key_rates.index)


def estimate_credit_bucket_bridge(bucket_panel, spread_changes_2pt, anchor_date, min_obs=120, horizon_bdays=1):
    """Sensitivity bridge from the simulated 2-point Moody's credit panel to the rating-bucket OAS
    curve, for REPRICING only (`CREDIT_BLOCK_SOURCE = "moody_2pt"`).

    The 2-point panel (Aaa/Baa seasoned yields minus the 10Y Treasury) supplies real, never-
    truncated point-in-time credit DYNAMICS; the rolling ~3y bucket OAS panel supplies (a) today's
    bucket ANCHOR and (b) this map of bucket sensitivities Gamma: per bucket, an interceptless OLS
    of daily bucket-OAS changes on the 2-point spread changes over the overlap window. Horizon
    bucket OAS = today's real bucket OAS + Gamma * (simulated 2-pt move). So the bridge lives on
    SENSITIVITIES, not on data: no past bucket history is fabricated, today's anchor is real, and
    the joint credit dynamics come from real long history. Declared limitation: a linear map fitted
    on a calm overlap understates the convex decompression of the low-quality buckets (CCC) in a
    crisis, so bridged bucket dispersion in deep-stress scenarios is conservative-looking on the
    HY end -- the caveat applies to the repricing granularity, not to the credit factor dynamics.
    Returns {"gamma": DataFrame(buckets x 2pt-cols), "current_buckets": Series, "r2": Series,
    "n_overlap": int}.
    """
    bp = bucket_panel.dropna(how="any")
    if bp.shape[0] < min_obs + 1:
        raise ValueError(f"Bucket OAS panel too short for the bridge ({bp.shape[0]} rows).")
    d_b = bp.diff().dropna()
    common = d_b.index.intersection(spread_changes_2pt.index)
    if len(common) < min_obs:
        raise ValueError(f"Bucket/2-point overlap too short for the bridge ({len(common)} rows).")
    X = spread_changes_2pt.loc[common].to_numpy(dtype=float)          # (T, 2)
    Y = d_b.loc[common].to_numpy(dtype=float)                         # (T, n_buckets)
    # v23: optionally fit on overlapping h-day changes (CREDIT_BRIDGE_FREQUENCY="horizon"), capped so
    # the overlap still holds ~20 independent windows. At h = 1 this is the v19-v22 daily fit, exactly.
    h = max(1, min(int(horizon_bdays), len(common) // 20))
    if h > 1:
        Xf = pd.DataFrame(X).rolling(h).sum().dropna().to_numpy()
        Yf = pd.DataFrame(Y).rolling(h).sum().dropna().to_numpy()
    else:
        Xf, Yf = X, Y
    G, _, _, _ = np.linalg.lstsq(Xf, Yf, rcond=None)                  # (2, n_buckets)
    fitted_f = Xf @ G
    ss_res = np.sum((Yf - fitted_f) ** 2, axis=0)
    ss_tot = np.sum(Yf ** 2, axis=0)                                  # interceptless convention
    r2 = 1.0 - np.divide(ss_res, ss_tot, out=np.zeros_like(ss_res), where=ss_tot > 0)
    fitted = X @ G                                                    # daily moves under the fitted gamma
    anchor = bp.loc[:anchor_date].iloc[-1]
    return {
        "gamma": pd.DataFrame(G.T, index=bp.columns, columns=spread_changes_2pt.columns),
        "current_buckets": anchor,
        "r2": pd.Series(r2, index=bp.columns),                       # at the fit frequency
        "n_overlap": int(len(common)),
        "frequency_bdays": int(h),
        # bucket-specific daily moves the 2-point map does not explain (v22, "restore" only)
        "residuals": pd.DataFrame(Y - fitted, index=common, columns=bp.columns),
    }


def build_credit_lsc_factors(credit_spreads):
    """Interpretable credit level / quality-slope / curvature proxies across the rating buckets.

    Columns are assumed ordered by ascending credit risk (AAA -> CCC). level = average spread,
    slope = widest(HY) - tightest(IG), curvature = 2*mid - tight - wide.
    """
    cols = list(credit_spreads.columns)
    tight, wide = cols[0], cols[-1]
    level = credit_spreads.mean(axis=1)
    slope = credit_spreads[wide] - credit_spreads[tight]
    out = {"credit_level": level, "credit_quality_slope": slope}
    if len(cols) >= 3:
        mid = cols[len(cols) // 2]
        out["credit_curvature"] = 2.0 * credit_spreads[mid] - credit_spreads[tight] - credit_spreads[wide]
    return pd.DataFrame(out, index=credit_spreads.index)


def build_fx_factor_block(fx_log_returns, explained_var_target=0.99, max_factors=None):
    """PCA factor block for a panel of daily FX log-returns (one column per currency pair).

    FX rates are NOT a redundant curve like the rating-bucket OAS, so for a small set of pairs this
    is essentially an identity (about one factor per pair); the PCA is kept for uniformity with the
    other blocks and so the block compresses gracefully if many pairs are added later. Unlike the
    rates / credit blocks, where the driver is a level *change* added onto a running level, the FX
    driver is a daily *log-return*: in the simulator it is cumulated over the horizon and then
    exponentiated onto the current spot level. `max_factors` caps the retained factors (None = one
    component budget per pair); the floor is 1 (a single pair yields a single factor).
    """
    mean_ret = fx_log_returns.mean()
    centered = fx_log_returns - mean_ret
    n_pairs = centered.shape[1]

    pca_full = PCA().fit(centered)
    cumvar = np.cumsum(pca_full.explained_variance_ratio_)
    k = int(np.searchsorted(cumvar, explained_var_target) + 1)
    cap = n_pairs if max_factors is None else min(int(max_factors), n_pairs)
    k = max(1, min(k, cap))

    pca = PCA(n_components=k)
    scores = pd.DataFrame(
        pca.fit_transform(centered),
        index=centered.index,
        columns=[f"fx_pc{i+1}" for i in range(k)],
    )
    loadings = pd.DataFrame(
        pca.components_.T,
        index=centered.columns,
        columns=scores.columns,
    )
    explained_ratio = pd.Series(
        pca.explained_variance_ratio_,
        index=scores.columns,
        name="explained_variance_ratio",
    )
    return {
        "scores": scores,
        "loadings": loadings,
        "mean_fx_logret": mean_ret,
        "explained_ratio": explained_ratio,
        "cum_explained_ratio": explained_ratio.cumsum(),
        "n_factors": k,
        "pca": pca,
    }


def build_joint_driver_matrix(equity_block, rates_block, credit_block=None, fx_block=None):
    parts = [
        equity_block["factors"],
        equity_block["resid_scores"],
        rates_block["scores"],
    ]
    if credit_block is not None:
        parts.append(credit_block["scores"])
    if fx_block is not None:
        parts.append(fx_block["scores"])
    drivers = pd.concat(parts, axis=1).dropna()
    return drivers


# ---- v24: hierarchical (global / block-specific) cross-asset factor structure -----------------

def driver_block_labels(columns):
    """Block of each joint driver, read off the column names build_joint_driver_matrix produces (v24):
    equity factors and residual PCs -> "equity", rate_pc* -> "rates", credit_pc* -> "credit",
    fx_pc* -> "fx"."""
    out = []
    for c in columns:
        s = str(c)
        out.append("rates" if s.startswith("rate_pc") else "credit" if s.startswith("credit_pc")
                   else "fx" if s.startswith("fx_pc") else "equity")
    return np.array(out, dtype=object)


def block_canonical_correlations(drivers, labels, n_top=2):
    """Leading canonical correlations between every pair of driver blocks (v24 pre-test).

    Each block is whitened on its own covariance; the singular values of the cross-covariance of the
    whitened blocks are the canonical correlations (Andreou, Gagliardini, Ghysels & Rubin 2019 count the
    factors that groups of variables SHARE from exactly these). Near 0: the pair shares nothing linear;
    near 1: it shares a factor. Sampling noise puts a floor of about sqrt((p_a + p_b) / T) under the first."""
    X = np.asarray(drivers, dtype=float)
    X = X - X.mean(axis=0)
    T = X.shape[0]
    labels = np.asarray(labels)
    blocks = list(dict.fromkeys(labels.tolist()))
    white = {}
    for b in blocks:
        Xb = X[:, labels == b]
        ev, V = np.linalg.eigh(np.atleast_2d(Xb.T @ Xb / T))
        keep = ev > ev.max() * 1e-10
        white[b] = Xb @ (V[:, keep] / np.sqrt(ev[keep]))
    rows = {}
    for i, a in enumerate(blocks):
        for b in blocks[i + 1:]:
            s = np.linalg.svd(white[a].T @ white[b] / T, compute_uv=False)
            rows[f"{a} / {b}"] = {f"canon_corr_{j + 1}": (float(s[j]) if j < len(s) else np.nan)
                                  for j in range(n_top)}
    return pd.DataFrame(rows).T


def _offblock_lowrank_fit(S, mask_off, labels, k, n_iter=3000, tol=1e-10, max_block_share=0.95):
    """Least-squares rank-k fit L L' to the OFF-block entries of S only (multi-battery factor analysis,
    unweighted least squares), the diagonal blocks treated as missing and imputed by the current fit
    (hard-impute iteration). A Heywood guard keeps the fitted global variance inside each block below
    `max_block_share` of the block's own variance. Used only to seed `_crossblock_factor_weights`."""
    blocks = list(dict.fromkeys(labels.tolist()))
    idx = {b: np.where(labels == b)[0] for b in blocks}
    inv_half = {}
    for b in blocks:
        ev, V = np.linalg.eigh(np.atleast_2d(S[np.ix_(idx[b], idx[b])]))
        ev = np.clip(ev, ev.max() * 1e-12, None)
        inv_half[b] = (V / np.sqrt(ev)) @ V.T
    A = S.copy()
    off_energy = float(np.sum((S * mask_off) ** 2))
    prev, best, it = None, None, 0
    for it in range(1, int(n_iter) + 1):
        ev, V = np.linalg.eigh(A)
        top = np.argsort(ev)[::-1][:k]
        L = V[:, top] * np.sqrt(np.clip(ev[top], 0.0, None))
        for b in blocks:
            Lb = L[idx[b]]
            r = float(np.linalg.eigvalsh(np.atleast_2d(inv_half[b] @ Lb @ Lb.T @ inv_half[b]))[-1])
            if r > max_block_share:
                L[idx[b]] = Lb * np.sqrt(max_block_share / r)
        LL = L @ L.T
        obj = float(np.sum(((S - LL) * mask_off) ** 2))
        if best is None or obj < best[1]:
            best = (L.copy(), obj)
        if prev is not None and abs(prev - obj) <= tol * max(off_energy, 1e-300):
            break
        prev = obj
        A = np.where(mask_off, S, LL)
    return best[0], best[1], off_energy, it


def _bartlett_weights(S, L, labels):
    """Bartlett factor-score weights for loadings L under a block-diagonal uniqueness
    Psi = blockdiag(S - L L'), repaired to positive definite block by block (a starting point only)."""
    Psi = np.zeros_like(S)
    for b in dict.fromkeys(labels.tolist()):
        m = np.where(labels == b)[0]
        B = S[np.ix_(m, m)] - L[m] @ L[m].T
        ev, V = np.linalg.eigh(np.atleast_2d(B))
        floor = 1e-3 * float(np.trace(np.atleast_2d(S[np.ix_(m, m)]))) / len(m)
        Psi[np.ix_(m, m)] = (V * np.clip(ev, floor, None)) @ V.T
    PiL = np.linalg.solve(Psi, L)
    return PiL @ np.linalg.inv(L.T @ PiL)


def _crossblock_objective(x, S, mask_off, kg):
    """Cross-block covariance energy left after regressing the drivers on g = Ds W, with its analytic
    gradient: f(W) = ||offblock(S - S W (W'SW)^-1 W'S)||_F^2 (v24)."""
    W = x.reshape(S.shape[0], kg)
    SW = S @ W
    try:
        Ai = np.linalg.inv(W.T @ SW)
    except np.linalg.LinAlgError:
        return 1e300, np.zeros_like(x)
    G = mask_off * (S - SW @ Ai @ SW.T)
    SGS = S @ G @ S
    grad = -4.0 * (SGS @ W @ Ai - SW @ (Ai @ (W.T @ SGS @ W) @ Ai))
    return float(np.sum(G * G)), grad.ravel()


def _crossblock_factor_weights(S, mask_off, labels, kg, extra_starts=()):
    """Weights W (k, kg) of the kg linear combinations of the block-normalized drivers whose removal by
    regression leaves the LEAST cross-block covariance (v24). Smooth and invariant to W -> W Q; solved by
    L-BFGS with the analytic gradient from several starts (a multi-battery least-squares factor fit's
    Bartlett weights, the leading eigenvectors, and any supplied start)."""
    starts = [np.asarray(s_, dtype=float) for s_ in extra_starts]
    L, _, _, _ = _offblock_lowrank_fit(S, mask_off, labels, kg, n_iter=300)   # a seed: a few hundred iterations suffice
    starts.append(_bartlett_weights(S, L, labels))
    ev, V = np.linalg.eigh(S)
    starts.append(V[:, np.argsort(ev)[::-1][:kg]])
    best = None
    for W0 in starts:
        r = minimize(_crossblock_objective, W0.ravel(), args=(S, mask_off, kg), jac=True,
                     method="L-BFGS-B", options={"maxiter": 5000})
        if best is None or r.fun < best.fun:
            best = r
    return best.x.reshape(S.shape[0], kg)


def _nested_unit_factors(Ds, W, w, prev_g=None):
    """Factors spanning Ds @ W, orthonormal under the weights w (sum w = 1), ordered so the first ones are
    as close as possible to the previous (k-1) solution: g_1 is always the best single global factor, g_2
    the best addition to it, and so on. Returns (T, kg)."""
    g = Ds @ W
    ev, V = np.linalg.eigh(np.atleast_2d((g * w[:, None]).T @ g))
    h = g @ (V / np.sqrt(np.clip(ev, 1e-300, None)))
    kg = h.shape[1]
    if prev_g is None or prev_g.shape[1] == 0:
        return h[:, ::-1]
    A = (h * w[:, None]).T @ prev_g
    Q, _ = np.linalg.qr(np.hstack([A, np.eye(kg)]))
    Q = Q[:, :kg]
    for j in range(A.shape[1]):
        if float(Q[:, j] @ A[:, j]) < 0:
            Q[:, j] *= -1.0
    return h @ Q


def build_hierarchical_structure(drivers, labels=None, n_global="auto", max_global=3,
                                 resid_corr_tol=None, halflife_days=None, k_table=4,
                                 local_resampling="independent"):
    """Hierarchical (global / block-specific) re-parametrization of the joint drivers (v24).

        d_t = Lambda g_t + u_t

    The drivers d are the existing per-block factors (equity APT factors + residual PCs, rate PCs, credit
    PCs, FX), so nothing downstream changes. The k_G GLOBAL factors g load on every block; u are the
    block-specific parts. Estimation:

      * each block is normalized by its first singular value (as in multiple factor analysis), so 8 rate
        PCs do not outvote one FX pair;
      * g are the k_G linear combinations of the drivers whose removal by regression leaves the LEAST
        cross-block covariance (`_crossblock_factor_weights`), nested across k (g_1 is the best single
        global factor, g_2 the best addition, ...). That is exactly the property independent resampling
        of the blocks relies on. Factor-analysis scores were tried and rejected: with one or two drivers
        per block the common factors are weakly determined, and their scores leave much of the
        cross-block covariance behind;
      * Lambda are the slopes of each driver on g, and u = d - Lambda g.

    `halflife_days` (GLOBAL_STRUCTURE_HALFLIFE_YEARS): None estimates all of the above on equal weights
    (static). A number estimates it on EXPONENTIALLY weighted moments -- the cross-asset structure of the
    CURRENT regime -- and then whitens the global factors over their full history, so a global shock
    drawn from any date (2008 and 2020 included) is transmitted through today's loadings. (Fixed-weight
    factors with EWMA loadings were tried first and rejected: on synthetic data with a known stock-yield
    sign flip they moved the model correlation from +0.10 to +0.06 when the 3-year-memory target was
    -0.11; this construction hits it.)

    k_G: "auto" picks the smallest k <= max_global that leaves no cross-block pair with more than
    `resid_corr_tol` of unexplained correlation (|cov(u_i, u_j)| / (sd_i sd_j), in the drivers' own
    units); None sets it at the sampling-noise floor, max(0.05, 3.5 / sqrt(effective sample size)).

    Returns, besides diagnostics: "X" = [g, u], what the simulator resamples; "M" = [Lambda, I], with
    d - mean = M x; "groups", the column sets resampled on their own date streams ("global" plus one per
    block when local_resampling="independent", a single group when "joint"). With joint resampling, no
    volatility filter and static estimation, M x_t = d_t - mean on every date and the simulator
    reproduces the blockwise engine.
    """
    if local_resampling not in ("independent", "joint"):
        raise ValueError(f"Unknown local_resampling {local_resampling!r}; expected 'independent' or 'joint'")
    cols = list(drivers.columns)
    index = drivers.index
    D = np.asarray(drivers, dtype=float)
    T, k = D.shape
    labels = driver_block_labels(cols) if labels is None else np.asarray(labels, dtype=object)
    blocks = list(dict.fromkeys(labels.tolist()))
    if len(blocks) < 2:
        raise ValueError("the hierarchical structure needs at least two driver blocks")
    if halflife_days is None:
        w = np.full(T, 1.0 / T)
    else:
        w = 0.5 ** ((T - 1 - np.arange(T)) / float(halflife_days))
        w = w / w.sum()
    ess = 1.0 / float(np.sum(w ** 2))
    tol = max(0.05, 3.5 / np.sqrt(ess)) if resid_corr_tol is None else float(resid_corr_tol)
    mean = D.mean(axis=0)
    Dc = D - mean
    scale = np.ones(k)
    for b in blocks:
        m = labels == b
        lam1 = float(np.linalg.eigvalsh(np.atleast_2d((Dc[:, m] * w[:, None]).T @ Dc[:, m]))[-1])
        scale[m] = 1.0 / np.sqrt(max(lam1, 1e-300))
    Ds = Dc * scale
    S = (Ds * w[:, None]).T @ Ds
    Sd = (Dc * w[:, None]).T @ Dc
    sdd = np.sqrt(np.clip(np.diag(Sd), 1e-300, None))
    mask_off = labels[:, None] != labels[None, :]
    off_energy = float(np.sum((S * mask_off) ** 2))

    def _metrics(U):
        Cs = ((U * scale) * w[:, None]).T @ (U * scale)
        Rf = np.where(mask_off, np.abs(((U * w[:, None]).T @ U) / np.outer(sdd, sdd)), -1.0)
        i_, j_ = np.unravel_index(int(np.argmax(Rf)), Rf.shape)
        R = Rf[mask_off]
        return (1.0 - float(np.sum((Cs * mask_off) ** 2)) / max(off_energy, 1e-300),
                float(R.max()), float(np.sqrt(np.mean(R ** 2))), f"{cols[min(i_, j_)]} / {cols[max(i_, j_)]}")

    _, r0max, r0rms, r0pair = _metrics(Dc)
    rows = [{"k_G": 0, "offblock_cov_share": 0.0, "resid_max_abs_corr": r0max, "resid_rms_corr": r0rms,
             "resid_max_pair": r0pair}]
    fits = {}
    kmax_tab = min(int(max(k_table, max_global if n_global == "auto" else int(n_global))), k - 1)
    prev_g = None
    for kk in range(1, kmax_tab + 1):
        extra = []
        if prev_g is not None:
            prev_W = np.linalg.lstsq(Ds, prev_g, rcond=None)[0]
            res_var = (((Ds - prev_g @ ((Ds * w[:, None]).T @ prev_g).T) ** 2) * w[:, None]).sum(axis=0)
            extra.append(np.hstack([prev_W, np.eye(k)[:, [int(np.argmax(res_var))]]]))
        W = _crossblock_factor_weights(S, mask_off, labels, kk, extra_starts=extra)
        g = _nested_unit_factors(Ds, W, w, prev_g)
        lam = (Dc * w[:, None]).T @ g                          # slopes, since Cov_w(g) = I
        share, rmax, rrms, rpair = _metrics(Dc - g @ lam.T)
        rows.append({"k_G": kk, "offblock_cov_share": share, "resid_max_abs_corr": rmax,
                     "resid_rms_corr": rrms, "resid_max_pair": rpair})
        fits[kk] = g
        prev_g = g
    k_table_df = pd.DataFrame(rows).set_index("k_G")
    if n_global == "auto":
        ok = [kk for kk in range(1, min(int(max_global), kmax_tab) + 1)
              if k_table_df.loc[kk, "resid_max_abs_corr"] <= tol]
        kg = ok[0] if ok else min(int(max_global), kmax_tab)
    else:
        kg = int(n_global)
        if kg < 1 or kg > kmax_tab:
            raise ValueError(f"GLOBAL_FACTORS={n_global!r} outside 1..{kmax_tab}")
    g = fits[kg].copy()
    # Sign convention: each global factor positively correlated with the first equity driver (the market
    # factor in every equity engine) when that link is material, else its largest loading positive.
    eq0 = int(np.where(labels == "equity")[0][0]) if "equity" in blocks else None
    for j in range(kg):
        cj = (Dc * w[:, None]).T @ g[:, j]
        if eq0 is not None and abs(cj[eq0] / sdd[eq0]) >= 0.10:
            sgn = np.sign(cj[eq0])
        else:
            sgn = np.sign((cj * scale)[np.argmax(np.abs(cj * scale))])
        g[:, j] *= (sgn if sgn != 0 else 1.0)
    lam = (Dc * w[:, None]).T @ g                              # (k, kg)
    U = Dc - g @ lam.T
    if halflife_days is not None:                              # whiten the factors' FULL history
        ev, V = np.linalg.eigh(np.atleast_2d(g.T @ g / T))
        g = g @ (V @ np.diag(1.0 / np.sqrt(np.clip(ev, 1e-300, None))) @ V.T)
    gcols = [f"global_{j + 1}" for j in range(kg)]
    X = pd.DataFrame(np.hstack([g, U]), index=index, columns=gcols + [f"u_{c}" for c in cols])
    M = np.hstack([lam, np.eye(k)])                            # d - mean = M x
    groups = [np.arange(kg)] + [kg + np.where(labels == b)[0] for b in blocks]
    group_labels = ["global"] + blocks
    if local_resampling == "joint":
        groups, group_labels = [np.arange(kg + k)], ["joint"]
    return {
        "columns": cols, "labels": labels, "blocks": blocks, "k_global": kg, "k_table": k_table_df,
        "resid_corr_tol": tol, "ess": ess, "weights": w, "halflife_days": halflife_days,
        "g": pd.DataFrame(g, index=index, columns=gcols), "u": pd.DataFrame(U, index=index, columns=cols),
        "loadings": pd.DataFrame(lam, index=cols, columns=gcols),
        "X": X, "M": M, "groups": groups, "group_labels": group_labels,
        "local_resampling": local_resampling, "driver_mean": pd.Series(mean, index=cols),
        "block_scale": pd.Series(scale, index=cols),
    }


def hierarchical_sim_spec(hs):
    """What simulate_joint_horizon needs from a hierarchical structure (v24): the map d - mean = M x and
    the column groups resampled on independent date streams."""
    return {"M": hs["M"], "groups": hs["groups"], "group_labels": hs["group_labels"]}


# ---- v25: standardized-dependence and latent-state challenger utilities ------------------------------
def _v25_weights(T, halflife_days=None):
    if halflife_days is None:
        return np.full(T, 1.0 / T)
    w = 0.5 ** ((T - 1 - np.arange(T)) / float(halflife_days))
    return w / w.sum()


def _v25_weighted_loadings(Yc, G, w):
    """Weighted OLS loadings Y ~= G Lambda' (Yc: T x k, G: T x r)."""
    sw = np.sqrt(np.asarray(w, dtype=float))
    B, *_ = np.linalg.lstsq(G * sw[:, None], Yc * sw[:, None], rcond=None)  # r x k
    return B.T


def _v25_stable_transition(G, w=None, cap=0.98):
    """Small latent VAR(1), stabilized by spectral-radius clipping (r <= 3 in the registered lab)."""
    G = np.asarray(G, dtype=float)
    X, Y = G[:-1], G[1:]
    if w is None:
        sw = np.ones(len(X))
    else:
        ww = np.asarray(w, dtype=float)[1:]
        ww = ww / max(ww.sum(), 1e-300)
        sw = np.sqrt(ww)
    B, *_ = np.linalg.lstsq(X * sw[:, None], Y * sw[:, None], rcond=None)  # X @ B = Y
    A = B.T
    rho = float(np.max(np.abs(np.linalg.eigvals(A)))) if A.size else 0.0
    if rho > cap and rho > 0:
        A = A * (float(cap) / rho)
        rho = float(cap)
    return A, rho


def _v25_kalman_filter(Yc, Lambda, A, Q, Rdiag, m0=None, P0=None):
    """Fast linear-Gaussian FILTER using the information form; only r x r matrices are inverted.

    Gaussianity is used to infer the latent state, not to generate the final shocks: the caller
    stationary-bootstraps the fitted state innovations and local residuals afterwards.
    """
    Yc = np.asarray(Yc, dtype=float)
    L = np.asarray(Lambda, dtype=float)
    A = np.asarray(A, dtype=float)
    Q = np.asarray(Q, dtype=float)
    rv = np.maximum(np.asarray(Rdiag, dtype=float), 1e-10)
    T, k = Yc.shape
    r = L.shape[1]
    m = np.zeros(r) if m0 is None else np.asarray(m0, dtype=float).copy()
    P = np.eye(r) if P0 is None else np.atleast_2d(np.asarray(P0, dtype=float)).copy()
    Rinv = 1.0 / rv
    LtRinv = L.T * Rinv[None, :]
    J_obs = LtRinv @ L
    out = np.empty((T, r))
    eye = np.eye(r)
    for t in range(T):
        mp = A @ m
        Pp = A @ P @ A.T + Q
        Pinv = np.linalg.pinv(Pp + 1e-10 * eye)
        P = np.linalg.pinv(Pinv + J_obs + 1e-10 * eye)
        m = P @ (Pinv @ mp + LtRinv @ Yc[t])
        out[t] = m
    return out


def _v25_normal_scores(Z, clip=1e-6):
    """Empirical PIT then Phi^{-1}, column by column. Returns normal scores and sorted raw Z marginals."""
    Z = np.asarray(Z, dtype=float)
    T, k = Z.shape
    Y = np.empty_like(Z)
    for j in range(k):
        # average ranks make ties deterministic; plotting-position avoids 0/1.
        r = pd.Series(Z[:, j]).rank(method="average").to_numpy(dtype=float)
        u = np.clip((r - 0.5) / T, clip, 1.0 - clip)
        Y[:, j] = norm.ppf(u)
    return Y, np.sort(Z, axis=0)


def _v25_inverse_empirical_pit(Y, sorted_z, clip=1e-6):
    """Phi -> empirical inverse CDF; preserves the standardized drivers' empirical marginals."""
    Y = np.asarray(Y, dtype=float)
    S = np.asarray(sorted_z, dtype=float)
    n, k = S.shape
    u = np.clip(norm.cdf(Y), clip, 1.0 - clip)
    pos = u * (n - 1)
    lo = np.floor(pos).astype(int)
    hi = np.minimum(lo + 1, n - 1)
    frac = pos - lo
    out = np.empty_like(Y)
    for j in range(k):
        out[:, j] = S[lo[:, j], j] * (1.0 - frac[:, j]) + S[hi[:, j], j] * frac[:, j]
    return out


def _v25_crossblock_residual_metrics(U, labels, w=None, reference=None):
    """v24-compatible residual cross-block metric.

    The denominator is the volatility of the ORIGINAL coordinates whose dependence is being explained,
    not the volatility of the residual itself.  This is exactly the convention used by the v24
    hierarchy: it answers "how much original cross-block dependence remains unexplained?" and avoids
    exploding the statistic merely because a successful factor extraction leaves a tiny residual.
    """
    U = np.asarray(U, dtype=float)
    labels = np.asarray(labels, dtype=object)
    Ref = U if reference is None else np.asarray(reference, dtype=float)
    if w is None:
        w = np.full(len(U), 1.0 / len(U))
    else:
        w = np.asarray(w, dtype=float)
        w = w / w.sum()
    Uc = U - (w @ U)[None, :]
    Rc = Ref - (w @ Ref)[None, :]
    C_u = (Uc * w[:, None]).T @ Uc
    C_ref = (Rc * w[:, None]).T @ Rc
    ref_sd = np.sqrt(np.clip(np.diag(C_ref), 1e-300, None))
    R = C_u / np.outer(ref_sd, ref_sd)
    mask = labels[:, None] != labels[None, :]
    vals = np.abs(R[mask])
    return {"resid_max_abs_corr": float(vals.max()),
            "resid_rms_corr": float(np.sqrt(np.mean(vals ** 2)))}


def _v25_reconstruction_scale(X, M, groups):
    """Marginal std of M x when the listed groups are resampled on independent date streams.

    The original-driver GJR recursion requires its innovation z to have unit unconditional variance.
    Independent resampling of global and local pieces can move the reconstructed z variance slightly
    away from one even when the input GJR residuals were standardized exactly.  Column-wise rescaling
    restores unit marginal variance without changing any cross-sectional correlation.
    """
    X = np.asarray(X, dtype=float)
    M = np.asarray(M, dtype=float)
    C = np.zeros((M.shape[0], M.shape[0]), dtype=float)
    for gcols in groups:
        gcols = np.asarray(gcols, dtype=int)
        Xg = X[:, gcols]
        Xg = Xg - Xg.mean(axis=0, keepdims=True)
        Cg = Xg.T @ Xg / max(len(Xg), 1)
        Mg = M[:, gcols]
        C += Mg @ Cg @ Mg.T
    sd = np.sqrt(np.clip(np.diag(C), 1e-12, None))
    return sd


def build_v25_challenger_structure(drivers, raw_fhs_filter, kind="standardized_hierarchical",
                                    labels=None, n_global="auto", max_global=3, resid_corr_tol=None,
                                    halflife_days=None, local_resampling="independent",
                                    kalman_refinements=2, ar_cap=0.98, r_floor=1e-4, pit_clip=1e-6):
    """Build one of the three v25 challengers on GJR-standardized ORIGINAL drivers.

    Volatility and dependence are deliberately separated. `raw_fhs_filter` owns every original
    driver's conditional variance. This function sees only z_t = (d_t-mu)/sigma_t and learns a
    global/local dependence representation. The simulator reconstructs z first and only then applies
    the original drivers' forward GJR volatilities -- no double scaling.

    Kinds:
      standardized_hierarchical : v24 covariance-targeted hierarchy in z-space, no latent dynamics.
      kalman_hierarchical       : Gaussian filter on z-space; empirical bootstrap of eta and u.
      pit_kalman_hierarchical   : PIT->N scores, same filter, inverse PIT back to empirical z marginals.
    """
    if kind not in ("standardized_hierarchical", "kalman_hierarchical", "pit_kalman_hierarchical"):
        raise ValueError(f"unknown v25 challenger {kind!r}")
    if raw_fhs_filter is None:
        raise ValueError("v25 challengers require the original-driver GJR/FHS filter")
    cols = list(drivers.columns)
    labels = driver_block_labels(cols) if labels is None else np.asarray(labels, dtype=object)
    Z = np.asarray(raw_fhs_filter["standardized_residuals"], dtype=float)
    Zdf = pd.DataFrame(Z, index=drivers.index, columns=cols)
    T, k = Z.shape
    w = _v25_weights(T, halflife_days)

    if kind == "standardized_hierarchical":
        hs = build_hierarchical_structure(
            Zdf, labels=labels, n_global=n_global, max_global=max_global,
            resid_corr_tol=resid_corr_tol, halflife_days=halflife_days,
            k_table=max_global, local_resampling=local_resampling
        )

        # Keep the exact v24 architecture in standardized coordinates:
        # current-regime Lambda/u are estimated first; the full-history global shock library may then
        # be whitened, but Lambda is NOT re-estimated afterwards.
        X = hs["X"].to_numpy(dtype=float)
        Lam = hs["loadings"].to_numpy(dtype=float)
        U = hs["u"].to_numpy(dtype=float)
        M = np.asarray(hs["M"], dtype=float)
        z_mean = hs["driver_mean"].to_numpy(dtype=float)
        kg = int(hs["k_global"])
        groups = [np.asarray(g_, dtype=int) for g_ in hs["groups"]]
        group_labels = list(hs["group_labels"])

        # The selected row in k_table is the authoritative v24 metric.
        _sel = hs["k_table"].loc[kg]
        met = {
            "resid_max_abs_corr": float(_sel["resid_max_abs_corr"]),
            "resid_rms_corr": float(_sel["resid_rms_corr"]),
        }

        # Unit-variance guard for the ORIGINAL-driver GJR recursion.  This changes marginal scale only;
        # correlations are invariant to the positive column-wise scaling.
        z_reconstruction_scale = _v25_reconstruction_scale(X, M, groups)

        # Forensic counterfactual only: what the rejected post-whitening loading refit would leave behind,
        # scored with the SAME denominator convention as v24.
        G_white = hs["g"].to_numpy(dtype=float)
        Zc = Z - z_mean[None, :]
        Lam_refit = _v25_weighted_loadings(Zc, G_white, w)
        U_refit = Zc - G_white @ Lam_refit.T
        met_refit = _v25_crossblock_residual_metrics(U_refit, labels, w, reference=Zc)

        recon_gap = Zc - X @ M.T
        recon_gap_rms = float(np.sqrt(np.mean(recon_gap ** 2)))
        z_rms = float(np.sqrt(np.mean(Zc ** 2)))
        recon_gap_ratio = recon_gap_rms / max(z_rms, 1e-12)

        return {
            "kind": kind,
            "X": X,
            "groups": groups,
            "group_labels": group_labels,
            "M": M,
            "z_mean": z_mean,
            "raw_fhs": raw_fhs_filter,
            "k_global": kg,
            "labels": labels,
            "columns": cols,
            "loadings": Lam,
            "resid_metrics": met,
            "halflife_days": halflife_days,
            "k_table": hs.get("k_table"),
            "resid_corr_tol": hs.get("resid_corr_tol"),
            "counterfactual_refit_metrics": met_refit,
            "historical_reconstruction_gap_rms": recon_gap_rms,
            "historical_reconstruction_gap_ratio": recon_gap_ratio,
            "z_reconstruction_scale": z_reconstruction_scale,
        }

    # Kalman challengers. PIT changes only the observation coordinates; GJR stays in raw-driver space.
    if kind == "pit_kalman_hierarchical":
        Y, sorted_z = _v25_normal_scores(Z, clip=pit_clip)
    else:
        Y, sorted_z = Z.copy(), None
    y_mean = Y.mean(axis=0)
    Yc = Y - y_mean[None, :]
    Ydf = pd.DataFrame(Y, index=drivers.index, columns=cols)
    hs0 = build_hierarchical_structure(Ydf, labels=labels, n_global=n_global, max_global=max_global,
                                       resid_corr_tol=resid_corr_tol, halflife_days=halflife_days,
                                       k_table=max_global, local_resampling="independent")
    G = hs0["g"].to_numpy(dtype=float)
    kg = G.shape[1]
    # A few cheap fixed-point refinements: OLS moments -> Gaussian filter -> moments.  This is a
    # low-dimensional QML-style challenger, not a giant full-MLE state-space fit.
    Lam = _v25_weighted_loadings(Yc, G, w)
    rho = np.nan
    for _ in range(max(1, int(kalman_refinements))):
        A, rho = _v25_stable_transition(G, w=w, cap=ar_cap)
        eta = G[1:] - G[:-1] @ A.T
        ww = w[1:]; ww = ww / ww.sum()
        Q = (eta * ww[:, None]).T @ eta + 1e-7 * np.eye(kg)
        Lam = _v25_weighted_loadings(Yc, G, w)
        U0 = Yc - G @ Lam.T
        Rdiag = np.maximum((U0 ** 2 * w[:, None]).sum(axis=0), float(r_floor))
        P0 = np.atleast_2d(np.cov(G.T)) if len(G) > kg + 2 else np.eye(kg)
        G = _v25_kalman_filter(Yc, Lam, A, Q, Rdiag, m0=np.zeros(kg), P0=P0)
    # final moments on filtered states
    A, rho = _v25_stable_transition(G, w=w, cap=ar_cap)
    Lam = _v25_weighted_loadings(Yc, G, w)
    eta = G[1:] - G[:-1] @ A.T
    U = Yc[1:] - G[1:] @ Lam.T
    X = np.hstack([eta, U])
    blocks = list(dict.fromkeys(labels.tolist()))
    groups = [np.arange(kg)] + [kg + np.where(labels == b)[0] for b in blocks]
    group_labels = ["global_state_innov"] + blocks
    met = _v25_crossblock_residual_metrics(U, labels, w[1:] / w[1:].sum())
    # latent one-step R2: how much of the filtered global-state variance A g_{t-1} explains.
    pred = G[:-1] @ A.T
    den = np.var(G[1:], axis=0)
    r2 = 1.0 - np.var(G[1:] - pred, axis=0) / np.maximum(den, 1e-12)
    return {"kind": kind, "X": X, "groups": groups, "group_labels": group_labels,
            "raw_fhs": raw_fhs_filter, "k_global": kg, "labels": labels, "columns": cols,
            "loadings": Lam, "A": A, "g_last": G[-1].copy(), "y_mean": y_mean,
            "sorted_z": sorted_z, "pit_clip": pit_clip, "resid_metrics": met,
            "latent_r2": r2, "spectral_radius": rho, "halflife_days": halflife_days}


def v25_challenger_diagnostics(spec):
    row = {"k_global": spec["k_global"], **spec["resid_metrics"]}
    if "counterfactual_refit_metrics" in spec:
        row["refit_resid_max_abs_corr"] = spec["counterfactual_refit_metrics"]["resid_max_abs_corr"]
        row["refit_resid_rms_corr"] = spec["counterfactual_refit_metrics"]["resid_rms_corr"]
        row["historical_reconstruction_gap_ratio"] = spec["historical_reconstruction_gap_ratio"]
    if "z_reconstruction_scale" in spec:
        _zs = np.asarray(spec["z_reconstruction_scale"], dtype=float)
        row["raw_reconstructed_z_sd_min"] = float(_zs.min())
        row["raw_reconstructed_z_sd_max"] = float(_zs.max())
    if "spectral_radius" in spec:
        row["latent_AR_spectral_radius"] = spec["spectral_radius"]
        row["latent_mean_R2"] = float(np.mean(spec["latent_r2"]))
    else:
        row["latent_AR_spectral_radius"] = np.nan
        row["latent_mean_R2"] = np.nan
    return row


def _weighted_r2(y, Z, w):
    y = y - float(w @ y)
    if Z.shape[1] == 0:
        return 0.0
    Z = Z - (w @ Z)[None, :]
    sw = np.sqrt(w)
    beta, *_ = np.linalg.lstsq(Z * sw[:, None], y * sw, rcond=None)
    e = y - Z @ beta
    den = float(w @ (y * y))
    return float(1.0 - (w @ (e * e)) / den) if den > 0 else 0.0


def hierarchical_variance_decomposition(hs, series_by_block):
    """Vasicek / GCorr-style reading of the hierarchical structure (v24): for every OBSERVED series, the
    share of its daily variance explained by the global factors ("global", the analogue of a one-factor
    asset correlation rho), the additional share explained by its own block's common factors
    ("block_common"), and the rest ("idiosyncratic"). Sequential R^2 on the history, weighted like the
    structure itself, as in the variance decompositions of multi-level factor models (Kose, Otrok &
    Whiteman 2003). Block common factors: for equities the block-specific parts of the equity FACTORS (the
    residual PCs are stock-specific and count as idiosyncratic); for rates, credit and FX the
    block-specific parts of all the block's drivers."""
    g = hs["g"]
    u = hs["u"]
    w_all = pd.Series(hs["weights"], index=g.index)
    rows = []
    for block, frame in series_by_block.items():
        if frame is None or block not in hs["blocks"]:
            continue
        common = frame.index.intersection(g.index)
        if len(common) < 50:
            continue
        w = w_all.loc[common].to_numpy()
        w = w / w.sum()
        G = g.loc[common].to_numpy()
        bcols = [c for c, lab in zip(hs["columns"], hs["labels"]) if lab == block
                 and not (block == "equity" and str(c).startswith("resid_pc"))]
        B = u.loc[common, bcols].to_numpy() if bcols else np.zeros((len(common), 0))
        for s in frame.columns:
            y = frame.loc[common, s].to_numpy(dtype=float)
            r_g = _weighted_r2(y, G, w)
            r_gb = max(_weighted_r2(y, np.hstack([G, B]), w), r_g)
            rows.append({"block": block, "series": s, "global": r_g, "block_common": r_gb - r_g,
                         "idiosyncratic": 1.0 - r_gb})
    return pd.DataFrame(rows).set_index(["block", "series"])


def global_transmission_table(hs, observables, split_date):
    """Correlation of each global factor with a few observables (the equal-weight equity return, the 10Y
    yield change, the representative credit-spread change, the FX return) over the whole window and
    before / from `split_date` (v24). The global factors are defined once, so a sign change across the
    split is a change in TRANSMISSION -- in the loadings -- not in the factor."""
    g = hs["g"]
    obs = observables.reindex(g.index)
    split = pd.Timestamp(split_date)
    out = {}
    for lab, msk in (("whole window", np.ones(len(g), dtype=bool)),
                     (f"before {split.date()}", np.asarray(g.index < split)),
                     (f"from {split.date()}", np.asarray(g.index >= split))):
        if msk.sum() < 60:
            continue
        gg = g.loc[msk]
        oo = obs.loc[msk]
        for j in g.columns:
            out[(lab, j)] = {c: float(gg[j].corr(oo[c])) for c in obs.columns}
    return pd.DataFrame(out).T


def par_bond_horizon_return(y0, dy, maturity_years, horizon_years, freq=2):
    """Horizon total return of a par bond bought at yield y0 (coupon = y0), repriced at y0 + dy with
    maturity N - h, plus the coupon carried over the horizon (constant-maturity approximation, as in
    Swinkels 2019; roll-down ignored). Vectorized; applied identically to simulated and realized moves."""
    y0 = np.asarray(y0, dtype=float)
    y1 = y0 + np.asarray(dy, dtype=float)
    c = y0
    n = float(freq) * max(float(maturity_years) - float(horizon_years), 1e-6)
    y1s = np.where(np.abs(y1) < 1e-10, 1e-10, y1)
    disc = (1.0 + y1s / float(freq)) ** (-n)
    return c / y1s * (1.0 - disc) + disc + c * float(horizon_years) - 1.0


def compute_fhs_filter_for_matrix(X, ewma_lambda=0.94, reversion_halflife=34.0, params_mode="fixed"):
    """compute_ewma_fhs_filter on a hierarchical [g, u] matrix (v24), guarding columns with (numerically)
    zero variance -- a block-specific part a global factor has absorbed entirely. Those columns get a
    zero standardized residual and zero variance, so they contribute nothing; every other column is
    filtered exactly as a joint driver would be."""
    A = np.asarray(X, dtype=float)
    mu = A.mean(axis=0)
    var = A.var(axis=0)
    live = var > 1e-14 * max(float(var.max()), 1e-300)
    f = compute_ewma_fhs_filter(A[:, live], mu[live], ewma_lambda=ewma_lambda,
                                reversion_halflife=reversion_halflife, params_mode=params_mode)
    if live.all():
        return f
    T, k = A.shape
    out = dict(f)
    z = np.zeros((T, k))
    z[:, live] = f["standardized_residuals"]
    out["standardized_residuals"] = z
    for key in ("cond_var_now", "omega", "uncond_var"):
        v = np.zeros(k)
        v[live] = np.asarray(f[key], dtype=float)
        out[key] = v
    for key in ("alpha", "beta", "gamma", "persistence"):
        v = np.zeros(k)
        v[live] = np.broadcast_to(np.asarray(f[key], dtype=float), (int(live.sum()),))
        out[key] = v
    cv = np.ones(k)
    cv[live] = np.asarray(f["current_vol_ratio"], dtype=float)
    out["current_vol_ratio"] = cv
    if f.get("per_factor") is not None:
        pf = {}
        for key, val in f["per_factor"].items():
            v = np.zeros(k)
            v[live] = np.asarray(val, dtype=float)
            pf[key] = v
        out["per_factor"] = pf
    return out


def build_var1_model(drivers):
    return VAR(drivers).fit(maxlags=1, trend="c")


def var_one_step_r2(var_res, drivers):
    """Per-equation in-sample one-step-ahead R^2 of the VAR(1) vs a constant-only baseline.

    Issue #9 diagnostic: if these R^2 are near zero, the VAR(1) mean dynamics add little and the
    simulated risk is effectively coming from the bootstrapped innovations + factor loadings, not
    from genuine cross-predictability. In that case the model could be simplified (drop the VAR and
    block-bootstrap the demeaned joint innovations) or regularized (Minkowski/ridge / block prior).
    """
    resid = var_res.resid
    y = drivers.loc[resid.index]
    r2 = 1.0 - resid.var(ddof=0) / y.var(ddof=0)
    return r2.sort_values(ascending=False)


def stationary_bootstrap_index_matrix(n_obs, n_paths, horizon, avg_block_length, rng):
    """Vectorized stationary bootstrap (Politis-Romano) index matrix, shape (n_paths, horizon).

    Issue #5: replaces the per-path Python loop. Same generative process as before: at each step,
    with probability 1/avg_block_length restart at a fresh uniform index, otherwise advance the
    previous index by one (mod n_obs). Only `horizon` vectorized iterations are needed.
    """
    p_restart = 1.0 / avg_block_length
    restart = rng.random((n_paths, horizon)) < p_restart
    restart[:, 0] = True
    fresh = rng.integers(0, n_obs, size=(n_paths, horizon))
    idx = np.empty((n_paths, horizon), dtype=int)
    idx[:, 0] = fresh[:, 0]
    for h in range(1, horizon):
        advanced = (idx[:, h - 1] + 1) % n_obs
        idx[:, h] = np.where(restart[:, h], fresh[:, h], advanced)
    return idx


def _flat_top_kernel(x):
    """Politis (2003) flat-top lag window: 1 on [0, 1/2], linearly down to 0 on [1/2, 1], 0 beyond."""
    w = np.ones_like(x, dtype=float)
    mid = (x >= 0.5) & (x <= 1.0)
    w[mid] = 2.0 * (1.0 - x[mid])
    w[x > 1.0] = 0.0
    return w


def politis_white_block_length(x):
    """Automatic block-length selection for the stationary bootstrap (Politis & White 2004, with the
    Patton-Politis-White 2009 correction), computed per column.

    Returns the optimal *expected* block length per column. The optimum is
    b = (2 G^2 / D_SB)^{1/3} n^{1/3} with G = sum_k lam(k/M)|k|R(k), D_SB = 2 (sum_k lam(k/M)R(k))^2,
    R the autocovariance and M a flat-top bandwidth chosen where the autocorrelation becomes
    negligible. Because the procedure reads the *level* autocovariance, applying it to returns gives a
    short block (returns are nearly serially uncorrelated) while applying it to the absolute/squared
    series captures volatility clustering -- which is exactly why, after the FHS filter has removed the
    clustering, the standardized residuals admit a short block.
    """
    x = np.asarray(x, dtype=float)
    if x.ndim == 1:
        x = x[:, None]
    n, k = x.shape
    kn = max(5, int(np.ceil(np.sqrt(np.log10(n)))))
    mmax = int(np.ceil(np.sqrt(n))) + kn
    bmax = float(np.ceil(min(3.0 * np.sqrt(n), n / 3.0)))
    cval = 2.0 * np.sqrt(np.log10(n) / n)
    out = np.empty(k)
    for j in range(k):
        e = x[:, j] - x[:, j].mean()
        acv = np.array([np.sum(e[:n - lag] * e[lag:]) / n for lag in range(mmax + 1)])
        if acv[0] <= 0:
            out[j] = 1.0
            continue
        rho = acv / acv[0]
        sig = np.abs(rho) > cval
        mhat = None
        for mm in range(1, mmax - kn + 1):
            if not sig[mm + 1:mm + 1 + kn].any():
                mhat = mm
                break
        if mhat is None:
            mhat = mmax
        M = int(min(2 * mhat, mmax))
        lags = np.arange(-M, M + 1)
        w = _flat_top_kernel(np.abs(lags) / M)
        g_acv = acv[np.abs(lags)]
        g_hat = np.sum(w * np.abs(lags) * g_acv)
        g0 = np.sum(w * g_acv)
        if g0 <= 0 or g_hat == 0:
            out[j] = 1.0
        else:
            b = (2.0 * g_hat ** 2 / (2.0 * g0 ** 2)) ** (1.0 / 3.0) * n ** (1.0 / 3.0)
            out[j] = float(min(max(b, 1.0), bmax))
    return out


def _nearest_pd_correlation(corr):
    """Project a symmetric matrix to a positive-definite correlation matrix (eigenvalue clipping).

    The Kendall-tau -> R = sin(pi*tau/2) transform is applied element-wise and need not stay PD, so
    we clip the eigenvalues to a small floor and rescale the diagonal back to one.
    """
    c = np.asarray(corr, dtype=float)
    c = 0.5 * (c + c.T)
    vals, vecs = np.linalg.eigh(c)
    vals = np.clip(vals, 1e-8, None)
    c_pd = (vecs * vals) @ vecs.T
    d = np.sqrt(np.diag(c_pd))
    c_pd = c_pd / np.outer(d, d)
    np.fill_diagonal(c_pd, 1.0)
    return c_pd


def _t_copula_loglik(tq, R_inv, logdet_R, nu, d):
    """t-copula log-likelihood at pseudo-observations whose univariate-t quantiles are tq (n, d)."""
    maha = np.einsum("ni,ij,nj->n", tq, R_inv, tq)
    mvt = (gammaln((nu + d) / 2.0) - gammaln(nu / 2.0) - 0.5 * d * np.log(nu * np.pi)
           - 0.5 * logdet_R - 0.5 * (nu + d) * np.log1p(maha / nu))
    univ = (gammaln((nu + 1.0) / 2.0) - gammaln(nu / 2.0) - 0.5 * np.log(nu * np.pi)
            - 0.5 * (nu + 1.0) * np.log1p(tq ** 2 / nu)).sum(axis=1)
    return float(np.sum(mvt - univ))


def t_copula_tail_dependence(rho, nu):
    """Lower/upper tail-dependence coefficient of a bivariate t-copula with correlation rho, dof nu.

    lambda = 2 * t_{nu+1}( -sqrt((nu+1)(1-rho)/(1+rho)) ). Unlike the Gaussian copula (lambda = 0 for
    any rho < 1), the t-copula has positive tail dependence that grows as nu falls -- the probability
    that two drivers are jointly extreme does not vanish in the tail.
    """
    rho = np.clip(rho, -0.999, 0.999)
    arg = -np.sqrt((nu + 1.0) * (1.0 - rho) / (1.0 + rho))
    return 2.0 * student_t.cdf(arg, nu + 1.0)


def fit_t_copula(drivers, nu_grid=None):
    """Fit a Student-t copula to the joint driver innovations (a tail-dependent dependence model).

    A copula separates the *marginals* (kept empirical here, so each driver's own distribution is
    preserved exactly) from the *dependence structure*. Unlike a Gaussian copula, the t-copula has
    non-zero tail dependence: joint extreme moves co-occur more often than linear correlation implies,
    with the strength set by the degrees of freedom nu (low nu -> strong tail dependence; nu -> inf
    recovers the Gaussian copula). This is the property the block bootstrap lacks -- it can only
    resample observed joint vectors, whereas the t-copula can generate new joint extremes consistent
    with the fitted tail-dependent structure.

    The correlation matrix is estimated from Kendall's tau (R = sin(pi*tau/2)), which is rank-based
    and invariant to the marginals; nu is fitted by profile maximum-likelihood over `nu_grid`. The
    demeaned empirical marginals are stored (sorted) for the inversion step at simulation time.
    """
    X = np.asarray(drivers, dtype=float)
    mean = X.mean(axis=0)
    Xc = X - mean
    n, d = Xc.shape
    tau = pd.DataFrame(Xc).corr(method="kendall").to_numpy()
    R = _nearest_pd_correlation(np.sin(np.pi / 2.0 * tau))
    R_inv = np.linalg.inv(R)
    _, logdet_R = np.linalg.slogdet(R)
    ranks = pd.DataFrame(Xc).rank().to_numpy()
    U = np.clip(ranks / (n + 1.0), 1e-9, 1 - 1e-9)
    if nu_grid is None:
        nu_grid = [2.5, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0, 15.0, 25.0, 50.0]
    ll_by_nu = {}
    best_nu, best_ll = None, -np.inf
    for nu in nu_grid:
        ll = _t_copula_loglik(student_t.ppf(U, nu), R_inv, logdet_R, nu, d)
        ll_by_nu[float(nu)] = ll
        if ll > best_ll:
            best_ll, best_nu = ll, float(nu)
    # average off-diagonal correlation and the implied average pairwise tail dependence
    off = R[~np.eye(d, dtype=bool)]
    avg_abs_corr = float(np.mean(np.abs(off)))
    avg_tail_dep = float(np.mean([t_copula_tail_dependence(r, best_nu) for r in off[off > 0]])) if np.any(off > 0) else 0.0
    gaussian_ll = ll_by_nu.get(50.0, _t_copula_loglik(student_t.ppf(U, 50.0), R_inv, logdet_R, 50.0, d))
    return {
        "R": R, "chol_R": np.linalg.cholesky(R), "nu": best_nu,
        "loglik": best_ll, "loglik_by_nu": ll_by_nu, "gaussian_loglik": gaussian_ll,
        "avg_abs_corr": avg_abs_corr, "avg_tail_dependence": avg_tail_dep,
        "mean": mean, "marg_sorted": np.sort(Xc, axis=0), "n_obs": n, "dim": d,
        "columns": list(drivers.columns) if hasattr(drivers, "columns") else None,
    }


def sample_t_copula_innovations(copula, n_samples, rng):
    """Draw demeaned joint innovations from a fitted t-copula (empirical marginals + t dependence).

    Multivariate-t draw (Z ~ N(0,R), W ~ chi2(nu): T = Z / sqrt(W/nu)) -> Student-t CDF to uniforms
    carrying the t-copula's tail dependence -> inverted through each driver's empirical (demeaned)
    marginal by interpolating its sorted values at the implied rank position.
    """
    chol = copula["chol_R"]; nu = copula["nu"]; marg = copula["marg_sorted"]
    n_obs, d = marg.shape
    z = rng.standard_normal((n_samples, d)) @ chol.T
    w = rng.chisquare(nu, size=n_samples) / nu
    tvar = z / np.sqrt(w)[:, None]
    u = np.clip(student_t.cdf(tvar, nu), 1e-9, 1 - 1e-9)
    pos = u * (n_obs - 1)
    lo = np.floor(pos).astype(int)
    hi = np.clip(lo + 1, 0, n_obs - 1)
    frac = pos - lo
    out = np.empty((n_samples, d), dtype=float)
    for j in range(d):
        col = marg[:, j]
        out[:, j] = col[lo[:, j]] * (1.0 - frac[:, j]) + col[hi[:, j]] * frac[:, j]
    return out


def fit_asset_copula(equity_log_returns, yield_changes, spread_changes, fx_log_returns=None, nu_grid=None):
    """Fit a Student-t copula at the OBSERVABLE (asset / risk-driver) level, not on the PCA factors.

    The factor-level copula (v11) had little tail dependence to capture: the PCA factors are nearly
    uncorrelated by construction, so the joint extremes of the *factors* are weak, while the equity
    co-tail is really a shared-loadings phenomenon the bootstrap reproduces by resampling. Fitting the
    copula directly on the daily observables -- equity log-returns, key-rate changes (highly
    correlated across the curve), rating-bucket spread changes (highly correlated), and the FX
    log-return -- puts the copula where the real cross-sectional correlation, and hence the real tail
    dependence, lives. Generation is then reduced-form: a draw is a full daily observable innovation
    (each variable's empirical marginal + the t-copula's tail-dependent joint structure), assembled
    directly into equity/rate/credit/FX scenarios with no factor reconstruction. The trade is that it
    drops the parsimonious factor structure -- it is a parallel, dependence-faithful generator to
    compare against the factor engine, not a replacement for it.
    """
    blocks = [("equity", equity_log_returns), ("rates", yield_changes), ("credit", spread_changes)]
    if fx_log_returns is not None:
        blocks.append(("fx", fx_log_returns))
    idx = None
    for _, df in blocks:
        idx = df.index if idx is None else idx.intersection(df.index)
    panel_parts, groups, start = [], {}, 0
    for name, df in blocks:
        d = df.loc[idx]
        cols = list(d.columns)
        groups[name] = {"slice": slice(start, start + len(cols)), "columns": cols}
        start += len(cols)
        panel_parts.append(d)
    panel = pd.concat(panel_parts, axis=1)
    cop = fit_t_copula(panel, nu_grid=nu_grid)
    cop["asset_groups"] = groups
    cop["panel_columns"] = list(panel.columns)
    return cop


def _garch11_nll_grid(r2, uncond_var, alphas, persistences):
    """Vectorized Gaussian negative log-likelihood (x2, up to an additive constant) of GARCH(1,1)
    candidates under variance targeting: omega = (1 - p) * uncond_var, beta = p - alpha. Evaluates all
    (alpha, p) candidates simultaneously with a single pass over time; differences between entries are
    likelihood-ratio statistics."""
    A = np.asarray(alphas, dtype=float)
    P = np.asarray(persistences, dtype=float)
    B = P - A
    W = (1.0 - P) * uncond_var
    floor = 1e-12 * max(float(uncond_var), 1e-300)
    s2 = np.full(A.shape, float(uncond_var))
    nll = np.zeros(A.shape)
    for t in range(r2.shape[0]):
        s2 = np.maximum(s2, floor)
        nll += np.log(s2) + r2[t] / s2
        s2 = W + A * r2[t] + B * s2
    return nll


def estimate_garch11_params(series, uncond_var=None, persistence_cap=0.998, min_lr=6.0, min_alpha=0.02):
    """GARCH(1,1) parameters for ONE series by Gaussian quasi-ML with variance targeting.

    Variance targeting pins the long-run variance to the sample variance (omega = (1-p)*uncond), which
    is exactly what the FHS forward recursion needs (mean reversion to the unconditional level) and
    leaves two free parameters: the news weight alpha and the persistence p = alpha + beta. These are
    found on a coarse (alpha, p) grid evaluated in one vectorized pass, then refined on a local fine
    grid -- deterministic and robust on ~700 observations, where a full MLE adds precision the data
    cannot support. Persistence is capped at `persistence_cap` (0.998 ~ a 350-business-day variance
    half-life): a 3-year sample cannot distinguish 0.998 from 1, and the cap keeps the one-year
    horizon mean-reverting rather than freezing today's volatility. If the likelihood gain over
    constant variance is below `min_lr` (default 6, roughly the chi-squared(2) 5% point, used as an
    indicative threshold since the null puts alpha on the boundary) or the fitted news weight is
    negligible, the series shows no
    volatility clustering and the function falls back to constant variance (alpha = beta = 0), so the
    FHS filter conditions only where the data supports it. Gaussian QML point estimates remain
    consistent under fat-tailed innovations (Bollerslev-Wooldridge), which is all we use them for.
    """
    e = np.asarray(series, dtype=float)
    e = e - e.mean()
    r2 = e ** 2
    T = r2.shape[0]
    uv = float(np.var(e)) if uncond_var is None else float(uncond_var)
    out_const = {"alpha": 0.0, "beta": 0.0, "persistence": 0.0,
                 "half_life_days": np.nan, "lr_vs_const": 0.0}
    if uv <= 0.0 or T < 60:
        return out_const
    nll_const = T * np.log(uv) + float(np.sum(r2)) / uv
    a_coarse = np.array([0.01, 0.02, 0.03, 0.05, 0.07, 0.09, 0.12, 0.15, 0.18, 0.22, 0.26, 0.30])
    p_coarse = np.array([0.90, 0.92, 0.94, 0.95, 0.96, 0.97, 0.975, 0.98,
                         0.985, 0.99, 0.993, 0.995, 0.9965, persistence_cap])
    cand = [(a, p) for a in a_coarse for p in p_coarse if p >= a]
    A = np.array([c[0] for c in cand]); P = np.array([c[1] for c in cand])
    nll = _garch11_nll_grid(r2, uv, A, P)
    i = int(np.argmin(nll)); a_best, p_best, nll_best = float(A[i]), float(P[i]), float(nll[i])
    a_fine = np.clip(a_best + np.linspace(-0.02, 0.02, 9), 0.003, 0.35)
    p_fine = np.clip(p_best + np.linspace(-0.006, 0.006, 13), 0.85, persistence_cap)
    cand2 = [(a, p) for a in np.unique(a_fine) for p in np.unique(p_fine) if p >= a]
    A2 = np.array([c[0] for c in cand2]); P2 = np.array([c[1] for c in cand2])
    nll2 = _garch11_nll_grid(r2, uv, A2, P2)
    j = int(np.argmin(nll2))
    if float(nll2[j]) < nll_best:
        a_best, p_best, nll_best = float(A2[j]), float(P2[j]), float(nll2[j])
    lr = max(0.0, nll_const - nll_best)
    if lr < min_lr or a_best < min_alpha:
        out_const["lr_vs_const"] = lr
        return out_const
    return {"alpha": a_best, "beta": p_best - a_best, "persistence": p_best,
            "half_life_days": float(np.log(0.5) / np.log(p_best)), "lr_vs_const": lr}


def _gjr11_nll_grid(r2, neg, uncond_var, alphas, gammas, persistences, neg_share=0.5):
    """Vectorized Gaussian NLL (x2, up to a constant) of GJR-GARCH(1,1) candidates under variance
    targeting: sigma2_{t+1} = omega + (alpha + gamma*1[eps_t<0]) * eps_t^2 + beta * sigma2_t, with
    persistence p = alpha + gamma*m + beta and omega = (1 - p) * uncond_var, where m is the share of
    the squared innovations that falls on down days (v23: measured; up to v22 m = 1/2, the
    symmetric-innovation approximation). One pass over time evaluates all candidates."""
    A = np.asarray(alphas, dtype=float)
    G = np.asarray(gammas, dtype=float)
    P = np.asarray(persistences, dtype=float)
    B = P - A - float(neg_share) * G
    W = (1.0 - P) * uncond_var
    floor = 1e-12 * max(float(uncond_var), 1e-300)
    s2 = np.full(A.shape, float(uncond_var))
    nll = np.zeros(A.shape)
    for t in range(r2.shape[0]):
        s2 = np.maximum(s2, floor)
        nll += np.log(s2) + r2[t] / s2
        s2 = W + (A + G * neg[t]) * r2[t] + B * s2
    return nll


def estimate_gjr_garch11_params(series, uncond_var=None, persistence_cap=0.998,
                                min_lr=6.0, min_news=0.02, min_lr_asym=3.84):
    """GJR-GARCH(1,1) parameters for ONE series by Gaussian QML with variance targeting.

    Extends the symmetric estimator with the Glosten-Jagannathan-Runkle leverage term: volatility may
    react MORE to negative shocks (gamma > 0), the equity stylized fact that acts specifically on the
    left tail. Model selection is nested and evidence-gated: the best symmetric fit (gamma = 0) is
    kept unless the asymmetric term improves the likelihood by at least `min_lr_asym` (~chi-sq(1) 5%,
    indicative -- the null puts gamma on a boundary); if not even the best fit beats constant variance
    by `min_lr`, the factor falls back to constant variance. The news gate is on the TOTAL news
    weight alpha + gamma/2 (an index-equity GJR often has alpha ~ 0 with all news arriving through
    gamma, which must not be gated away). Persistence p = alpha + gamma*m + beta is capped at
    `persistence_cap`; since v23 the identity is p = alpha + gamma*m + beta with m = E[1[eps<0] eps^2] / E[eps^2]
    MEASURED on the series (up to v22 m = 1/2, the symmetric-innovation approximation, which
    understated the persistence of negatively skewed series and let the simulated variance drift above
    its target at long horizons).
    """
    e = np.asarray(series, dtype=float)
    e = e - e.mean()
    r2 = e ** 2
    neg = (e < 0.0).astype(float)
    T = r2.shape[0]
    uv = float(np.var(e)) if uncond_var is None else float(uncond_var)
    out_const = {"alpha": 0.0, "gamma": 0.0, "beta": 0.0, "persistence": 0.0,
                 "half_life_days": np.nan, "lr_vs_const": 0.0, "lr_asym": 0.0}
    if uv <= 0.0 or T < 60:
        return out_const
    nll_const = T * np.log(uv) + float(np.sum(r2)) / uv
    # v23: the share of squared deviations that falls on down days. A GJR term gamma * 1[e<0] * e^2
    # adds gamma * m to the persistence on average; up to v22 m was ASSUMED to be 1/2. With equity
    # returns negatively skewed (m > 1/2) that understated the persistence, so the simulated variance
    # drifted ABOVE its target -- more the longer the horizon. Measured here, it keeps the targeting exact.
    m = float(np.sum(neg * r2) / np.sum(r2)) if float(np.sum(r2)) > 0.0 else 0.5
    a_g = np.array([0.0, 0.01, 0.02, 0.04, 0.06, 0.09, 0.12, 0.16])
    g_g = np.array([0.0, 0.02, 0.04, 0.07, 0.10, 0.14, 0.19, 0.25])
    p_g = np.array([0.90, 0.93, 0.95, 0.96, 0.97, 0.975, 0.98, 0.985, 0.99, 0.994, persistence_cap])
    cand = [(a, g, p) for a in a_g for g in g_g for p in p_g
            if (p - a - m * g) >= 0.0 and (a + g) > 0.0]
    A = np.array([c[0] for c in cand]); G = np.array([c[1] for c in cand]); P = np.array([c[2] for c in cand])
    nll = _gjr11_nll_grid(r2, neg, uv, A, G, P, neg_share=m)

    def _refine(a0, g0, p0, free_gamma=True):
        af = np.unique(np.clip(a0 + np.linspace(-0.02, 0.02, 5), 0.0, 0.35))
        gf = np.unique(np.clip(g0 + np.linspace(-0.03, 0.03, 7), 0.0, 0.45)) if free_gamma else np.array([g0])
        pf = np.unique(np.clip(p0 + np.linspace(-0.004, 0.004, 7), 0.85, persistence_cap))
        c2 = [(a, g, p) for a in af for g in gf for p in pf
              if (p - a - m * g) >= 0.0 and (a + g) > 0.0]
        if not c2:
            return a0, g0, p0, np.inf
        A2 = np.array([c[0] for c in c2]); G2 = np.array([c[1] for c in c2]); P2 = np.array([c[2] for c in c2])
        n2 = _gjr11_nll_grid(r2, neg, uv, A2, G2, P2, neg_share=m)
        j = int(np.argmin(n2))
        return float(A2[j]), float(G2[j]), float(P2[j]), float(n2[j])

    sym_mask = G == 0.0
    i_s = int(np.argmin(np.where(sym_mask, nll, np.inf)))
    i_a = int(np.argmin(nll))
    a_s, g_s, p_s, n_s = _refine(float(A[i_s]), 0.0, float(P[i_s]), free_gamma=False)
    a_a, g_a, p_a, n_a = _refine(float(A[i_a]), float(G[i_a]), float(P[i_a]), free_gamma=True)
    if n_a > n_s:
        a_a, g_a, p_a, n_a = a_s, g_s, p_s, n_s
    lr_all = max(0.0, nll_const - n_a)
    lr_sym = max(0.0, nll_const - n_s)
    lr_asym = max(0.0, n_s - n_a)
    if max(lr_all, lr_sym) < min_lr:
        out_const["lr_vs_const"] = lr_all
        return out_const
    if lr_asym >= min_lr_asym and g_a > 0.0:
        a_b, g_b, p_b, lr_b = a_a, g_a, p_a, lr_all
    else:
        a_b, g_b, p_b, lr_b = a_s, 0.0, p_s, lr_sym
    if (a_b + m * g_b) < min_news:
        out_const["lr_vs_const"] = lr_b
        out_const["lr_asym"] = lr_asym
        return out_const
    return {"alpha": a_b, "gamma": g_b, "beta": max(p_b - a_b - m * g_b, 0.0),
            "persistence": p_b, "half_life_days": float(np.log(0.5) / np.log(p_b)),
            "lr_vs_const": lr_b, "lr_asym": lr_asym, "neg_share": m}


def estimate_ou_level_params(levels, tstat_gate=-2.86, min_obs=120):
    """Per-series Vasicek/OU discretization fitted on LEVELS: Delta y_t = a + b * y_{t-1} + eps_t,
    so kappa = -b (daily pull), theta = -a / b (long-run level), half-life = ln(2) / kappa.

    Rates and credit spreads mean-revert over long horizons for economic reasons; equity returns do
    not, so this applies to LEVEL blocks only. A series is gated IN only when b < 0 with a t-statistic
    below `tstat_gate` -- default -2.86, the Dickey-Fuller 5% critical value (with constant), because
    under the unit-root null the OLS t-stat is NOT t-distributed and an ordinary -2 gate would find
    spurious mean reversion on short samples. Non-gated series keep the random walk exactly
    (kappa = 0 and the RW drift is retained), so reversion is applied only where the data supports it.
    Returns per-column arrays: kappa, theta, half_life_days, tstat, gated (0/1 floats).
    """
    L = np.asarray(levels, dtype=float)
    T, n = L.shape
    kappa = np.zeros(n); theta = np.zeros(n)
    tst = np.zeros(n); hl = np.full(n, np.nan); gated = np.zeros(n)
    if T < min_obs:
        return {"kappa": kappa, "theta": theta, "half_life_days": hl, "tstat": tst, "gated": gated}
    for j in range(n):
        y = L[:, j]
        x = y[:-1]; dy = np.diff(y)
        xm = x.mean(); dm = dy.mean()
        sxx = float(np.sum((x - xm) ** 2))
        if sxx <= 0.0:
            continue
        b = float(np.sum((x - xm) * (dy - dm)) / sxx)
        a = dm - b * xm
        resid = dy - a - b * x
        s2 = float(np.sum(resid ** 2)) / max(len(dy) - 2, 1)
        se_b = np.sqrt(s2 / sxx) if s2 > 0 else 0.0
        t_b = b / se_b if se_b > 0 else 0.0
        tst[j] = t_b
        if b < 0.0 and t_b <= tstat_gate:
            kappa[j] = -b
            theta[j] = -a / b
            hl[j] = float(np.log(2.0) / kappa[j])
            gated[j] = 1.0
    return {"kappa": kappa, "theta": theta, "half_life_days": hl, "tstat": tst, "gated": gated}


def compute_ewma_fhs_filter(drivers, driver_mean, ewma_lambda=0.94, reversion_halflife=34.0,
                            params_mode="fixed"):
    """Filtered Historical Simulation (FHS) volatility filter at the FACTOR level.

    Standardizes each joint driver by its own time-varying conditional volatility so the resampled
    innovations can be re-scaled by *today's* volatility -- making the simulated risk conditional on
    the current regime while keeping the empirical (fat-tailed) shape of the standardized residuals
    that made the block bootstrap win over the copulas. Crucially this is done at the FACTOR level:
    the conditional CORRELATION between assets is not modelled directly but EMERGES from the
    conditional volatility of the shared factors -- when the common market factor's volatility is high
    today, its contribution to every asset is scaled up together, so cross-asset correlation rises in
    stress (the empirical 'correlations go to one in a crash') without ever estimating an asset
    correlation matrix.

    The conditional variance follows a GARCH(1,1) in RiskMetrics form: the news weight is
    1 - ewma_lambda (so ewma_lambda is the usual EWMA decay) but the persistence is set below 1 via
    `reversion_halflife`, so the variance MEAN-REVERTS to its unconditional level. Pure EWMA (infinite
    half-life) forecasts flat forward, which over a one-year horizon would freeze today's vol; mean
    reversion lets it wash out over ~the half-life -- and, as a bonus, supplies the low-frequency
    variance mean reversion the daily VAR could not see.

    Returns the standardized residuals (unit unconditional variance per factor), the one-step-ahead
    conditional variance for the first simulated day, and the (omega, alpha, beta) the simulator rolls
    forward (driven by the simulated innovations, so volatility clusters along each path).
    """
    R = np.asarray(drivers, dtype=float) - np.asarray(driver_mean, dtype=float)[None, :]
    T, k = R.shape
    uncond_var = R.var(axis=0)
    per_factor = None
    gamma = 0.0
    if params_mode in ("estimated", "estimated_gjr"):
        # Per-factor (GJR-)GARCH(1,1) QML (variance targeting): each factor gets its OWN news weight
        # and persistence -- and, in "estimated_gjr", its own leverage term gamma (kept only where the
        # asymmetric fit beats the symmetric one). No-clustering factors come back with
        # alpha = gamma = beta = 0 -> constant variance (the filter is inert for them).
        if params_mode == "estimated_gjr":
            _est = [estimate_gjr_garch11_params(R[:, j], uncond_var=float(uncond_var[j])) for j in range(k)]
            gamma = np.array([e_["gamma"] for e_ in _est])
        else:
            _est = [estimate_garch11_params(R[:, j], uncond_var=float(uncond_var[j])) for j in range(k)]
        alpha = np.array([e_["alpha"] for e_ in _est])
        persistence = np.array([e_["persistence"] for e_ in _est])
        if params_mode == "estimated_gjr":
            beta = np.array([e_["beta"] for e_ in _est])               # v23: beta = p - alpha - gamma*m
        else:
            beta = np.maximum(persistence - alpha - 0.5 * np.asarray(gamma), 0.0)
        per_factor = {
            "alpha": alpha.copy(),
            "persistence": persistence.copy(),
            "half_life_days": np.array([e_["half_life_days"] for e_ in _est]),
            "lr_vs_const": np.array([e_["lr_vs_const"] for e_ in _est]),
        }
        if params_mode == "estimated_gjr":
            per_factor["gamma"] = np.asarray(gamma).copy()
            per_factor["lr_asym"] = np.array([e_["lr_asym"] for e_ in _est])
    else:
        alpha = 1.0 - ewma_lambda
        persistence = 0.5 ** (1.0 / reversion_halflife)        # < 1 -> mean reversion in variance
        beta = persistence - alpha
        if beta < 0.0:                                         # guard for a very short half-life
            beta = 0.0
            persistence = alpha
    omega = (1.0 - persistence) * uncond_var                   # long-run variance == unconditional
    sigma2 = np.empty((T, k))
    sigma2[0] = uncond_var
    _use_gjr = bool(np.any(np.asarray(gamma) != 0.0))
    if _use_gjr:
        _neg = (R < 0.0).astype(float)
        for t in range(1, T):
            sigma2[t] = omega + (alpha + gamma * _neg[t - 1]) * R[t - 1] ** 2 + beta * sigma2[t - 1]
    else:
        for t in range(1, T):
            sigma2[t] = omega + alpha * R[t - 1] ** 2 + beta * sigma2[t - 1]
    z = R / np.sqrt(sigma2)
    z = z / z.std(axis=0)                                      # exact unit unconditional variance
    if _use_gjr:
        cond_var_now = omega + (alpha + gamma * _neg[T - 1]) * R[T - 1] ** 2 + beta * sigma2[T - 1]
    else:
        cond_var_now = omega + alpha * R[T - 1] ** 2 + beta * sigma2[T - 1]
    omega_sim, beta_sim = uncond_var * (1.0 - persistence), beta
    if _use_gjr:
        # v23 -- the simulator drives the recursion with these standardized residuals, so its long-run
        # variance is omega / (1 - alpha - beta - gamma * m_z) with m_z = E[1(z<0) z^2] measured on them.
        # Targeting omega on m_z (not on 1/2) makes that long-run variance exactly uncond_var. A
        # persistence at or above the cap is pulled back through beta.
        g_arr = np.asarray(gamma, dtype=float)
        m_z = np.mean((z < 0.0) * z ** 2, axis=0)
        cap = 0.999
        p_sim = alpha + beta + g_arr * m_z
        beta_sim = np.where(p_sim > cap, np.maximum(beta - (p_sim - cap), 0.0), beta)
        p_sim = np.minimum(alpha + beta_sim + g_arr * m_z, cap)
        omega_sim = (1.0 - p_sim) * uncond_var
        if per_factor is not None:
            per_factor["neg_share"] = m_z
            # what the v18-v22 targeting (m = 1/2) did to the simulated long-run variance at this p
            _den = 1.0 - persistence - g_arr * (m_z - 0.5)
            per_factor["v22_longrun_var_inflation"] = np.where(_den > 1e-9, (1.0 - persistence) / np.maximum(_den, 1e-9), np.inf)
    return {
        "standardized_residuals": z,
        "cond_var_now": cond_var_now,
        "omega": omega_sim,
        "alpha": alpha if np.ndim(alpha) else float(alpha),
        "beta": beta_sim if np.ndim(beta_sim) else float(beta_sim),
        "gamma": gamma if np.ndim(gamma) else float(gamma),
        "uncond_var": uncond_var,
        "persistence": persistence if np.ndim(persistence) else float(persistence),
        "current_vol_ratio": np.sqrt(cond_var_now / uncond_var),   # >1 stressed, <1 calm (per factor)
        "params_mode": params_mode,
        "per_factor": per_factor,
    }


def _apply_credit_bridge(result, credit_bridge, current_spreads, credit_floor):
    """Map the simulated credit panel (2-point Moody's curve) into rating-bucket OAS space for
    repricing. No-op when `credit_bridge` is None (default source) -- the byte-identity path. The
    native simulated panel is preserved under "credit_spread_scenarios_raw" so the validation cell
    can compare the engine against its own history in the space it actually simulates."""
    if credit_bridge is None or "credit_spread_scenarios" not in result:
        return result
    scen = result["credit_spread_scenarios"]
    gamma = credit_bridge["gamma"].reindex(columns=list(scen.columns))
    cur2 = current_spreads.reindex(scen.columns).to_numpy(dtype=float)
    delta = scen.to_numpy(dtype=float) - cur2[None, :]
    anchor = credit_bridge["current_buckets"]
    buckets = np.clip(anchor.to_numpy(dtype=float)[None, :] + delta @ gamma.to_numpy(dtype=float).T,
                      credit_floor, None)
    result["credit_spread_scenarios_raw"] = scen
    result["credit_spread_scenarios"] = pd.DataFrame(buckets, columns=list(anchor.index))
    return result



def _restore_credit_residual(result, credit_block, credit_bridge, horizon_bdays, rng_seed, credit_floor):
    """CREDIT_RESIDUAL_MODE="restore" (v22): add back, at the horizon, the bucket-specific credit
    variance the compression step drops -- the PCA truncation (oas_buckets) or the residual of the
    2-point -> bucket bridge (moody_2pt). Each scenario receives the h-day SUM of the daily compression
    residuals over a random window of the estimation sample, so the restored variance is the empirical
    horizon variance (day-to-day noise that mean-reverts is not over-counted) and the cross-bucket
    dependence of the residuals is kept. Residuals are uncorrelated with the retained drivers by
    construction (PCA orthogonality / OLS), so variances add. Own RNG stream: equity, rates, FX and the
    systematic credit path are untouched. No-op unless the block (or the bridge) carries a
    `residual_pool`, which only CREDIT_RESIDUAL_MODE="restore" attaches."""
    pool = None
    if credit_bridge is not None:
        pool = credit_bridge.get("residual_pool")
    elif credit_block is not None:
        pool = credit_block.get("residual_pool")
    if pool is None or "credit_spread_scenarios" not in result:
        return result
    scen = result["credit_spread_scenarios"]
    R = pool.reindex(columns=list(scen.columns)).fillna(0.0).to_numpy(dtype=float)   # (T, n_buckets)
    T, h, S = R.shape[0], int(horizon_bdays), scen.shape[0]
    if T < 2 or h < 1:
        return result
    rng = np.random.default_rng([int(rng_seed), 7211])
    if h < T:
        cs = np.vstack([np.zeros((1, R.shape[1])), np.cumsum(R, axis=0)])
        starts = rng.integers(0, T - h + 1, size=S)                  # windows inside the sample
    else:                                                            # horizon longer than the sample:
        reps = int(np.ceil(h / T)) + 2                               # circular windows (declared)
        cs = np.vstack([np.zeros((1, R.shape[1])), np.cumsum(np.vstack([R] * reps), axis=0)])
        starts = rng.integers(0, T, size=S)
    e = cs[starts + h] - cs[starts]
    out = np.clip(scen.to_numpy(dtype=float) + e, credit_floor, None)
    result["credit_spread_scenarios"] = pd.DataFrame(out, columns=list(scen.columns), index=scen.index)
    return result

def simulate_joint_horizon(var_res, resid_hist, last_state, equity_block, rates_block,
                           current_prices, current_key_rates, horizon_bdays, n_scenarios,
                           avg_block_length, rng_seed, daily_rf, resid_idio_std=None,
                           rate_floor=-0.02, credit_block=None, current_spreads=None,
                           credit_floor=1e-4, fx_block=None, current_fx=None,
                           dynamics="var1", demeaned_drivers=None, driver_mean=None,
                           crisis_mode="none", crisis_prob=0.0, crisis_scale=1.0,
                           crisis_pool=None, copula=None, asset_copula=None, fhs_filter=None,
                           ou_rates=None, ou_credit=None, credit_bridge=None, _boot_idx=None, hier=None, v25_structure=None):
    """Vectorized one-week joint scenario simulation (issue #5).

    Changes vs the original loop version:
      * fully vectorized across scenarios; only `horizon_bdays` steps are iterated, with batch
        matrix operations over the (n_scenarios, state_dim) state.
      * equity total-return reconstruction adds back a *constant* daily risk-free `daily_rf` (the
        FF daily RF at the as-of date) instead of the simulated DGS2/252, which unifies the
        risk-free convention with the factor block (issue #6). The variation of the short rate over
        a 5-business-day horizon is immaterial (order 1e-5), so treating it as constant is cleaner.
      * optional iid idiosyncratic noise `resid_idio_std` re-injects the residual variance that the
        PCA compression discards, so single-name tails are not understated (issue #11).

    Note: vectorization changes the order in which random numbers are consumed, so individual
    scenario paths differ from the old implementation; the simulated *distribution* is unchanged.
    `_boot_idx` is an internal testing hook (pass a precomputed index matrix to make the run
    deterministic).

    `v25_structure` is one of the standardized / Kalman challengers. It owns dependence in
    GJR-standardized driver space while `raw_fhs` inside the spec owns the original drivers'
    conditional variances. The simulator reconstructs a standardized driver shock first, rescales it
    by the ORIGINAL driver GJR variance, and only then enters the unchanged factor reconstruction.

    `hier` (v24, FACTOR_STRUCTURE="hierarchical"): {"M", "groups"} from hierarchical_sim_spec. The
    resampled matrix (`demeaned_drivers`, and the FHS filter) is then the hierarchical [g, u] matrix; each
    column group is drawn on its OWN stationary-bootstrap date stream (group 0 on this function's main
    stream, so a single "joint" group draws exactly the blockwise indices), and the day's driver vector
    is rebuilt as driver_mean + M x before the unchanged factor reconstruction. Block bootstrap only.
    """
    rng = np.random.default_rng(rng_seed)
    intercept = np.asarray(var_res.intercept, dtype=float)            # (k,)
    coef = np.asarray(var_res.coefs[0], dtype=float)                  # (k, k)
    resid_hist_arr = np.asarray(resid_hist, dtype=float)
    n_hist = resid_hist_arr.shape[0]

    # Option A: selectable joint-state dynamics. "var1" propagates the fitted VAR(1) and bootstraps
    # its residuals; "block_bootstrap" drops the autoregressive term entirely and resamples the
    # demeaned joint drivers in stationary blocks (state = driver_mean + demeaned_driver[boot]) -- a
    # block-bootstrap of the joint innovations around their mean. When the VAR one-step R^2 is ~0 the
    # two give near-identical distributions, but block_bootstrap removes the k + k^2 VAR parameters.
    # The factor reconstruction below (loadings -> daily returns/changes/spreads/FX, cumulated) is
    # identical in both modes; only the per-day state draw differs.
    if dynamics not in ("var1", "block_bootstrap", "t_copula", "asset_copula"):
        raise ValueError(f"Unknown dynamics: {dynamics!r} "
                         "(expected 'var1', 'block_bootstrap', 't_copula', 'asset_copula')")
    use_hier = hier is not None
    use_v25 = v25_structure is not None
    if use_hier and use_v25:
        raise ValueError("hier and v25_structure are alternative factor structures; pass only one")
    if (use_hier or use_v25) and dynamics != "block_bootstrap":
        raise ValueError("hierarchical / v25 structures require dynamics='block_bootstrap'")
    if dynamics == "block_bootstrap":
        if demeaned_drivers is None or driver_mean is None:
            raise ValueError("block_bootstrap dynamics requires demeaned_drivers and driver_mean")
        dd_arr = np.asarray(v25_structure["X"] if use_v25 else demeaned_drivers, dtype=float)
        dmean_arr = np.asarray(driver_mean, dtype=float)
        n_innov = dd_arr.shape[0]
    elif dynamics == "t_copula":
        if copula is None or driver_mean is None:
            raise ValueError("t_copula dynamics requires a fitted copula and driver_mean")
        dmean_arr = np.asarray(driver_mean, dtype=float)
    else:
        n_innov = n_hist

    # Crisis-tail overlay (opt-in; the calm-window default is crisis_mode="none", which leaves the
    # draw byte-identical to the no-crisis engine). A crisis is a JOINT event, so on a crisis step the
    # entire joint innovation is amplified, blowing out equity, rates, credit and FX together:
    #   * "regime_jump": parametric -- amplify by a fixed `crisis_scale` (manual or Moody's-calibrated).
    #   * "moody_backbone": empirical -- amplify by a real historical crisis severity drawn from
    #     `crisis_pool` (Moody's Baa-Aaa stress-day move sizes, in multiples of the calm std).
    # `crisis_prob` is the per-step probability a scenario enters stress. The crisis RNG draws are
    # only consumed when crisis_mode != "none", so the off path does not disturb the base sequence.
    if crisis_mode not in ("none", "regime_jump", "moody_backbone"):
        raise ValueError(f"Unknown crisis_mode: {crisis_mode!r} (expected 'none', 'regime_jump', 'moody_backbone')")
    use_crisis = crisis_mode != "none" and crisis_prob > 0.0
    if crisis_mode == "moody_backbone" and use_crisis:
        crisis_pool_arr = np.asarray(crisis_pool, dtype=float) if crisis_pool is not None else np.array([float(crisis_scale)])
        if crisis_pool_arr.size == 0:
            crisis_pool_arr = np.array([float(crisis_scale)])

    # Asset-level copula (v12): a reduced-form generator that bypasses the factor reconstruction. A
    # draw from the observable-level t-copula IS a full daily observable innovation (equity log-returns
    # + key-rate changes + spread changes + FX log-return), assembled directly into scenarios. The
    # marginals are demeaned in the copula, so the stored means are added back per draw; the crisis
    # overlay (if any) scales the demeaned innovation exactly as in the factor path.
    if dynamics == "asset_copula":
        if asset_copula is None:
            raise ValueError("asset_copula dynamics requires a fitted asset_copula")
        groups = asset_copula["asset_groups"]
        mean = np.asarray(asset_copula["mean"], dtype=float)
        S = n_scenarios
        has_cr = "credit" in groups
        has_fx = "fx" in groups
        eq_cols = groups["equity"]["columns"]; eq_sl = groups["equity"]["slice"]
        rt_cols = groups["rates"]["columns"]; rt_sl = groups["rates"]["slice"]
        cum_eq = np.zeros((S, len(eq_cols)))
        cum_dy = np.zeros((S, len(rt_cols)))
        cum_cr = np.zeros((S, len(groups["credit"]["columns"]))) if has_cr else None
        cum_fx = np.zeros((S, len(groups["fx"]["columns"]))) if has_fx else None
        for _h in range(horizon_bdays):
            draw = sample_t_copula_innovations(asset_copula, S, rng)      # (S, P) demeaned
            if use_crisis:
                is_crisis = rng.random(S) < crisis_prob
                if crisis_mode == "regime_jump":
                    cscale = np.where(is_crisis, crisis_scale, 1.0)
                else:
                    cscale = np.ones(S)
                    n_c = int(is_crisis.sum())
                    if n_c > 0:
                        cscale[is_crisis] = crisis_pool_arr[rng.integers(0, crisis_pool_arr.size, n_c)]
                draw = draw * cscale[:, None]
            obs = mean[None, :] + draw
            cum_eq += obs[:, eq_sl]
            cum_dy += obs[:, rt_sl]
            if has_cr:
                cum_cr += obs[:, groups["credit"]["slice"]]
            if has_fx:
                cum_fx += obs[:, groups["fx"]["slice"]]
        cur_p = current_prices.reindex(eq_cols).to_numpy()
        cur_r = current_key_rates.reindex(rt_cols).to_numpy()
        key_rate_scn = np.clip(cur_r[None, :] + cum_dy, rate_floor, None)
        result = {
            "stock_excess_return_scenarios": pd.DataFrame(cum_eq - daily_rf * horizon_bdays, columns=eq_cols),
            "stock_log_return_scenarios": pd.DataFrame(cum_eq, columns=eq_cols),
            "stock_price_scenarios": pd.DataFrame(cur_p[None, :] * np.exp(cum_eq), columns=eq_cols),
            "key_rate_scenarios": pd.DataFrame(key_rate_scn, columns=rt_cols),
        }
        if has_cr:
            cr_cols = groups["credit"]["columns"]
            cur_s = current_spreads.reindex(cr_cols).to_numpy()
            result["credit_spread_scenarios"] = pd.DataFrame(np.clip(cur_s[None, :] + cum_cr, credit_floor, None), columns=cr_cols)
        if has_fx:
            fx_cols = groups["fx"]["columns"]
            cur_fx_v = current_fx.reindex(fx_cols).to_numpy()
            result["fx_log_return_scenarios"] = pd.DataFrame(cum_fx, columns=fx_cols)
            result["fx_spot_scenarios"] = pd.DataFrame(cur_fx_v[None, :] * np.exp(cum_fx), columns=fx_cols)
        return _restore_credit_residual(_apply_credit_bridge(result, credit_bridge, current_spreads, credit_floor),
                                        credit_block, credit_bridge, horizon_bdays, rng_seed, credit_floor)

    asset_names = current_prices.index
    rate_names = current_key_rates.index
    n_stocks = len(asset_names)
    n_eq = equity_block["factors"].shape[1]
    n_resid = equity_block["resid_loadings"].shape[1]
    n_rate = rates_block["loadings"].shape[1]

    mu_excess = equity_block["mean_excess_returns"].to_numpy()        # (n_stocks,)
    beta = equity_block["betas"].to_numpy()                           # (n_stocks, n_eq)
    resid_load = equity_block["resid_loadings"].to_numpy()            # (n_stocks, n_resid)
    mu_dy = rates_block["mean_dy"].to_numpy()                         # (n_key_rates,)
    rate_load = rates_block["loadings"].to_numpy()                    # (n_key_rates, n_rate)

    # Credit block (optional). Stacked in the joint state AFTER the rates PCs, so its slice starts
    # at n_eq + n_resid + n_rate. Reconstruction mirrors the rates curve: spread changes are rebuilt
    # from the credit PCs and cumulated onto the current OAS levels. No extra RNG is consumed, so the
    # equity/rates draws are identical whether or not credit is present.
    use_credit = credit_block is not None and current_spreads is not None
    if use_credit:
        n_credit = credit_block["loadings"].shape[1]
        mu_dspread = credit_block["mean_dspread"].to_numpy()          # (n_ratings,)
        credit_load = credit_block["loadings"].to_numpy()            # (n_ratings, n_credit)
        spread_names = current_spreads.index
        spread_path = np.tile(current_spreads.to_numpy().astype(float), (n_scenarios, 1))
    else:
        n_credit = 0

    # FX block (optional). Stacked in the joint state AFTER the credit PCs, so its slice starts at
    # n_eq + n_resid + n_rate + n_credit (n_credit is 0 when no credit block is supplied). Unlike the
    # rates/credit curves, the FX driver is a daily *log-return*: it is cumulated over the horizon and
    # exponentiated onto the current spot level (one column per currency pair). No extra RNG is
    # consumed, so the equity/rates/credit draws are identical whether or not FX is present.
    use_fx = fx_block is not None and current_fx is not None
    if use_fx:
        n_fx = fx_block["loadings"].shape[1]
        mu_fx = fx_block["mean_fx_logret"].to_numpy()                 # (n_pairs,)
        fx_load = fx_block["loadings"].to_numpy()                     # (n_pairs, n_fx)
        fx_names = current_fx.index
        cum_fx = np.zeros((n_scenarios, len(fx_names)))
    else:
        n_fx = 0

    if resid_idio_std is None:
        idio_std = np.zeros(n_stocks)
    else:
        idio_std = np.nan_to_num(np.asarray(resid_idio_std.reindex(asset_names).to_numpy(), dtype=float), nan=0.0)

    if dynamics in ("var1", "block_bootstrap"):
        if _boot_idx is None:
            boot_idx = stationary_bootstrap_index_matrix(n_innov, n_scenarios, horizon_bdays, avg_block_length, rng)
        else:
            boot_idx = np.asarray(_boot_idx, dtype=int)
    if use_hier:
        # v24: one index matrix per column group; group 0 is boot_idx itself (main RNG stream), the
        # others come from their own seeded streams, so nothing else in this function moves.
        hier_M = np.asarray(hier["M"], dtype=float)                   # (k_drivers, k_x): d - mean = M x
        hier_groups = [np.asarray(g_, dtype=int) for g_ in hier["groups"]]
        hier_idx = [boot_idx] + [
            stationary_bootstrap_index_matrix(n_innov, n_scenarios, horizon_bdays, avg_block_length,
                                              np.random.default_rng([int(rng_seed), 9100 + gi]))
            for gi in range(1, len(hier_groups))]

    if use_v25:
        v25_groups = [np.asarray(g_, dtype=int) for g_ in v25_structure["groups"]]
        v25_idx = [boot_idx] + [
            stationary_bootstrap_index_matrix(n_innov, n_scenarios, horizon_bdays, avg_block_length,
                                              np.random.default_rng([int(rng_seed), 9250 + gi]))
            for gi in range(1, len(v25_groups))]
        _rf = v25_structure["raw_fhs"]
        v25_omega = np.asarray(_rf["omega"], dtype=float)[None, :]
        v25_alpha = np.asarray(_rf["alpha"], dtype=float)
        v25_beta = np.asarray(_rf["beta"], dtype=float)
        v25_gamma = np.asarray(_rf.get("gamma", 0.0), dtype=float)
        v25_sigma2 = np.tile(np.asarray(_rf["cond_var_now"], dtype=float), (n_scenarios, 1))
        v25_gjr = bool(np.any(v25_gamma != 0.0))
        if v25_structure["kind"] in ("kalman_hierarchical", "pit_kalman_hierarchical"):
            v25_gstate = np.tile(np.asarray(v25_structure["g_last"], dtype=float), (n_scenarios, 1))

    # Filtered Historical Simulation (conditional volatility) -- only with the block bootstrap. The
    # resampled draw is a STANDARDIZED residual that is re-scaled by a per-scenario, per-factor
    # conditional volatility starting at today's level and rolled forward by the GARCH recursion, so
    # volatility clusters along each path and mean-reverts to the unconditional level over the horizon.
    use_fhs = (fhs_filter is not None) and (dynamics == "block_bootstrap") and (not use_v25)
    if use_fhs:
        z_arr = np.asarray(fhs_filter["standardized_residuals"], dtype=float)
        fhs_omega = np.asarray(fhs_filter["omega"], dtype=float)[None, :]
        fhs_alpha = np.asarray(fhs_filter["alpha"], dtype=float)    # scalar (fixed) or (k,) per-factor
        fhs_beta = np.asarray(fhs_filter["beta"], dtype=float)      # broadcasting handles both
        fhs_gamma = np.asarray(fhs_filter.get("gamma", 0.0), dtype=float)
        _fhs_gjr = bool(np.any(fhs_gamma != 0.0))                   # leverage term active (estimated_gjr)
        sigma2 = np.tile(np.asarray(fhs_filter["cond_var_now"], dtype=float), (n_scenarios, 1))   # (S, k)

    state = np.tile(np.asarray(last_state, dtype=float), (n_scenarios, 1))                 # (S, k)
    key_rate_path = np.tile(current_key_rates.to_numpy().astype(float), (n_scenarios, 1))  # (S, n_key_rates)
    cum_excess = np.zeros((n_scenarios, n_stocks))
    cum_log = np.zeros((n_scenarios, n_stocks))

    for h in range(horizon_bdays):
        if dynamics == "var1":
            innov = resid_hist_arr[boot_idx[:, h]]                    # (S, k)
        elif dynamics == "block_bootstrap":
            if use_v25:
                _xdraw = np.empty((n_scenarios, dd_arr.shape[1]))
                for gi, gcols in enumerate(v25_groups):
                    _xdraw[:, gcols] = dd_arr[v25_idx[gi][:, h][:, None], gcols[None, :]]
                _kind25 = v25_structure["kind"]
                if _kind25 == "standardized_hierarchical":
                    _zc25 = _xdraw @ np.asarray(v25_structure["M"], dtype=float).T
                    _zscale25 = np.asarray(v25_structure.get("z_reconstruction_scale", np.ones(_zc25.shape[1])), dtype=float)
                    _zdrv = (np.asarray(v25_structure["z_mean"], dtype=float)[None, :]
                             + _zc25 / _zscale25[None, :])
                else:
                    _kg25 = int(v25_structure["k_global"])
                    _eta25 = _xdraw[:, :_kg25]
                    _u25 = _xdraw[:, _kg25:]
                    v25_gstate = v25_gstate @ np.asarray(v25_structure["A"], dtype=float).T + _eta25
                    _obs25 = (np.asarray(v25_structure["y_mean"], dtype=float)[None, :] +
                              v25_gstate @ np.asarray(v25_structure["loadings"], dtype=float).T + _u25)
                    if _kind25 == "pit_kalman_hierarchical":
                        _zdrv = _v25_inverse_empirical_pit(_obs25, v25_structure["sorted_z"],
                                                           clip=float(v25_structure.get("pit_clip", 1e-6)))
                    else:
                        _zdrv = _obs25
                innov = _zdrv * np.sqrt(v25_sigma2)
                _v25_innov_sq = innov ** 2
                if v25_gjr:
                    _v25_innov_neg = innov < 0.0
            elif use_hier and len(hier_groups) > 1:
                _src = z_arr if use_fhs else dd_arr                   # v24: each group on its own dates
                innov = np.empty((n_scenarios, _src.shape[1]))
                for gi, gcols in enumerate(hier_groups):
                    innov[:, gcols] = _src[hier_idx[gi][:, h][:, None], gcols[None, :]]
                if use_fhs:
                    innov = innov * np.sqrt(sigma2)
            elif use_fhs:
                innov = z_arr[boot_idx[:, h]] * np.sqrt(sigma2)       # revol standardized residual by today's vol
            else:
                innov = dd_arr[boot_idx[:, h]]                        # (S, k)
        else:  # t_copula: draw a fresh tail-dependent joint innovation (empirical margins + t copula)
            innov = sample_t_copula_innovations(copula, n_scenarios, rng)   # (S, k), demeaned

        if use_fhs:
            _fhs_innov_sq = innov ** 2                                # pre-crisis innovation drives the vol recursion
            if _fhs_gjr:
                _fhs_innov_neg = (innov < 0.0)                        # sign unchanged by the (positive) crisis scale

        if use_crisis:
            is_crisis = rng.random(n_scenarios) < crisis_prob         # (S,) per-scenario stress flag
            if crisis_mode == "regime_jump":
                cscale = np.where(is_crisis, crisis_scale, 1.0)
            else:  # moody_backbone: draw a real historical crisis severity for stressed scenarios
                cscale = np.ones(n_scenarios)
                n_c = int(is_crisis.sum())
                if n_c > 0:
                    cscale[is_crisis] = crisis_pool_arr[rng.integers(0, crisis_pool_arr.size, n_c)]
            innov = innov * cscale[:, None]                           # amplify the WHOLE joint innovation

        if use_fhs:
            # roll conditional variance forward (GARCH(1,1), mean-reverting); the crisis overlay is an
            # exogenous jump kept orthogonal to the endogenous vol clustering, so it does not feed back.
            if _fhs_gjr:
                sigma2 = fhs_omega + (fhs_alpha + fhs_gamma * _fhs_innov_neg) * _fhs_innov_sq + fhs_beta * sigma2
            else:
                sigma2 = fhs_omega + fhs_alpha * _fhs_innov_sq + fhs_beta * sigma2


        if use_v25:
            # Same convention as the base engine: the exogenous crisis multiplier does not feed back
            # into the endogenous GJR recursion.  The recurrence therefore uses the pre-crisis raw
            # innovation reconstructed from standardized dependence × current marginal volatility.
            if v25_gjr:
                v25_sigma2 = v25_omega + (v25_alpha + v25_gamma * _v25_innov_neg) * _v25_innov_sq + v25_beta * v25_sigma2
            else:
                v25_sigma2 = v25_omega + v25_alpha * _v25_innov_sq + v25_beta * v25_sigma2


        if dynamics == "var1":
            state = intercept[None, :] + state @ coef.T + innov      # (S, k)
        else:  # empirical innovation around the ORIGINAL driver mean; v25 already rebuilt raw driver shocks
            if use_v25:
                state = dmean_arr[None, :] + innov
            else:
                state = dmean_arr[None, :] + (innov @ hier_M.T if use_hier else innov)   # (S, k)

        eq_state = state[:, :n_eq]
        resid_state = state[:, n_eq:n_eq + n_resid]
        rate_state = state[:, n_eq + n_resid:n_eq + n_resid + n_rate]

        dy = mu_dy[None, :] + rate_state @ rate_load.T               # (S, n_key_rates)
        if ou_rates is not None:
            # Vasicek/OU pull toward the estimated long-run level, per tenor and per scenario. Gated
            # tenors replace the RW drift with the OU deterministic part (no double counting);
            # non-gated tenors have kappa = 0 and keep the random walk exactly.
            dy = (dy + ou_rates["kappa"][None, :] * (ou_rates["theta"][None, :] - key_rate_path)
                  - ou_rates["gated"][None, :] * mu_dy[None, :])
        key_rate_path = np.clip(key_rate_path + dy, rate_floor, None)

        if use_credit:
            credit_state = state[:, n_eq + n_resid + n_rate:n_eq + n_resid + n_rate + n_credit]
            dspread = mu_dspread[None, :] + credit_state @ credit_load.T   # (S, n_ratings)
            if ou_credit is not None:
                dspread = (dspread + ou_credit["kappa"][None, :] * (ou_credit["theta"][None, :] - spread_path)
                           - ou_credit["gated"][None, :] * mu_dspread[None, :])
            spread_path = np.clip(spread_path + dspread, credit_floor, None)

        if use_fx:
            base = n_eq + n_resid + n_rate + n_credit
            fx_state = state[:, base:base + n_fx]
            daily_fx = mu_fx[None, :] + fx_state @ fx_load.T              # (S, n_pairs) daily FX log-returns
            cum_fx += daily_fx

        idio = resid_state @ resid_load.T                            # (S, n_stocks)
        if idio_std.any():
            idio = idio + rng.normal(0.0, 1.0, size=(n_scenarios, n_stocks)) * idio_std[None, :]

        daily_excess = mu_excess[None, :] + eq_state @ beta.T + idio  # (S, n_stocks)
        daily_log = daily_excess + daily_rf                          # constant rf add-back (issue #6)
        cum_excess += daily_excess
        cum_log += daily_log

    price_scenarios = current_prices.to_numpy()[None, :] * np.exp(cum_log)

    result = {
        "stock_excess_return_scenarios": pd.DataFrame(cum_excess, columns=asset_names),
        "stock_log_return_scenarios": pd.DataFrame(cum_log, columns=asset_names),
        "stock_price_scenarios": pd.DataFrame(price_scenarios, columns=asset_names),
        "key_rate_scenarios": pd.DataFrame(key_rate_path, columns=rate_names),
    }
    if use_credit:
        result["credit_spread_scenarios"] = pd.DataFrame(spread_path, columns=spread_names)
    if use_fx:
        fx_spot_scenarios = current_fx.to_numpy()[None, :] * np.exp(cum_fx)
        result["fx_log_return_scenarios"] = pd.DataFrame(cum_fx, columns=fx_names)
        result["fx_spot_scenarios"] = pd.DataFrame(fx_spot_scenarios, columns=fx_names)
    return _restore_credit_residual(_apply_credit_bridge(result, credit_bridge, current_spreads, credit_floor),
                                        credit_block, credit_bridge, horizon_bdays, rng_seed, credit_floor)


def make_zero_curve_from_key_rates(key_tenors_years, key_rates):
    x = np.array([0.0] + list(key_tenors_years), dtype=float)
    y = np.array([key_rates[0]] + list(key_rates), dtype=float)
    def zero_rate(t):
        t = float(max(t, 0.0))
        return float(np.interp(t, x, y))
    return zero_rate


def generate_coupon_schedule(issue_date, maturity_date, freq=2):
    months_step = 12 // freq
    dates = []
    current = pd.Timestamp(issue_date) + pd.DateOffset(months=months_step)
    while current < pd.Timestamp(maturity_date):
        dates.append(current)
        current = current + pd.DateOffset(months=months_step)
    dates.append(pd.Timestamp(maturity_date))
    return pd.DatetimeIndex(dates)


def price_coupon_bond_from_curve(eval_date, payment_dates, key_rate_series, tenor_years,
                                 coupon_rate=0.02, face_value=100.0, freq=2, spread=0.0):
    """Dirty present value of a coupon bond off an interpolated zero curve.

    Issue #8: the tenor grid is now passed in (`tenor_years`) instead of hard-coding [2, 5, 10],
    so the pricer works with any set of FRED key rates.

    Credit block: an optional flat `spread` (the bond's option-adjusted credit spread, in decimals)
    is added to every interpolated zero rate, so a corporate bond is discounted at Treasury-zero +
    OAS. With spread=0.0 this is exactly the Treasury pricer (backward compatible).

    Issue #10 (declared approximations): the FRED CMT series (DGS*) are *par* bond-equivalent
    yields, not zero rates. Here we (a) linearly interpolate them across tenor and (b) treat the
    interpolated yields as *continuously-compounded zero rates* (exp(-z t) discounting). This
    conflates par-yield ~= zero-rate and bond-equivalent ~= continuous compounding and introduces a
    small systematic bias. It is acceptable for a prototype; a production version would bootstrap a
    zero curve from the par yields and respect the day-count / compounding convention. The OAS is
    likewise added as a flat continuously-compounded spread, consistent with that approximation.

    Returns the *dirty* (full) present value of all remaining cashflows; see `accrued_interest`
    for the clean/dirty split.
    """
    remaining_dates = payment_dates[payment_dates > pd.Timestamp(eval_date)]
    if len(remaining_dates) == 0:
        return float(face_value)
    times = np.array([(d - pd.Timestamp(eval_date)).days / 365.25 for d in remaining_dates], dtype=float)
    zero_curve = make_zero_curve_from_key_rates(tenor_years, np.asarray(key_rate_series, dtype=float))
    zero_rates = np.array([zero_curve(t) for t in times], dtype=float) + float(spread)
    discount_factors = np.exp(-zero_rates * times)
    cashflows = np.full(len(times), coupon_rate * face_value / freq, dtype=float)
    cashflows[-1] += face_value
    return float(np.sum(cashflows * discount_factors))


def reprice_corporate_bond_scenarios(eval_date, payment_dates, key_rate_scenarios, tenor_years,
                                     spread_scenarios, coupon_rate, face_value=100.0, freq=2):
    """Per-scenario dirty PV of a corporate bond, discounting at the simulated Treasury curve plus
    the simulated OAS for the bond's rating bucket.

    `key_rate_scenarios` is (S, n_key_rates) and `spread_scenarios` is the (S,) OAS path for the
    bond's rating; row s uses curve s and spread s jointly, so a scenario where equities fall and
    spreads widen together (the equity-credit link carried by the joint bootstrap) hits the
    corporate bond through both the rate and the spread channel. Light Python loop over scenarios:
    the pricer is a cheap interpolation, so this is not the simulation bottleneck.
    """
    curves = np.asarray(key_rate_scenarios, dtype=float)
    spreads = np.asarray(spread_scenarios, dtype=float)
    prices = np.empty(curves.shape[0], dtype=float)
    for s in range(curves.shape[0]):
        prices[s] = price_coupon_bond_from_curve(
            eval_date, payment_dates, curves[s], tenor_years,
            coupon_rate=coupon_rate, face_value=face_value, freq=freq, spread=float(spreads[s]),
        )
    return pd.Series(prices)


def standard_1y_transition_matrix(ratings=("AAA", "AA", "A", "BBB", "BB", "B", "CCC")):
    """A representative one-year rating transition matrix (rows = current rating, columns = horizon
    rating including 'D' = default), in the style of published S&P/Moody's global average matrices.

    These are illustrative average through-the-cycle values (e.g. ~0.3% one-year default for BBB,
    ~1.2% for BB, ~4.5% for B, ~26% for CCC); swap in your preferred matrix if you have one. Used as
    is at 1Y, and scaled to 1M/3M through its generator (transition_matrix_for_horizon) to draw each
    corporate's horizon rating; unused at 1W (the one-week migration probability is negligible).
    """
    pct = {
        "AAA": [90.00, 9.00, 0.60, 0.15, 0.15, 0.05, 0.03, 0.02],
        "AA":  [0.60, 90.00, 8.00, 0.80, 0.30, 0.20, 0.05, 0.05],
        "A":   [0.05, 2.00, 91.00, 5.50, 0.90, 0.40, 0.05, 0.10],
        "BBB": [0.02, 0.25, 4.50, 89.00, 4.50, 1.00, 0.40, 0.33],
        "BB":  [0.02, 0.08, 0.40, 6.00, 82.00, 8.00, 2.30, 1.20],
        "B":   [0.00, 0.08, 0.25, 0.40, 6.50, 82.00, 6.27, 4.50],
        "CCC": [0.00, 0.00, 0.20, 0.40, 1.50, 12.00, 59.90, 26.00],
    }
    cols = list(ratings) + ["D"]
    m = pd.DataFrame({r: pct[r] for r in ratings}, index=cols).T
    m = m.div(m.sum(axis=1), axis=0)            # normalize rows to 1 (defensive)
    return m


def transition_matrix_for_horizon(transition_matrix, horizon_years):
    """Rating transition matrix over `horizon_years`, from a ONE-YEAR matrix, via its generator.

    A time-homogeneous Markov chain observed yearly has P(h) = exp(h Q) with Q = log P(1). Empirical
    one-year matrices rarely embed exactly: log P typically has a few small NEGATIVE off-diagonal
    rates, which is not a valid generator (the embedding problem). The standard repair (Israel,
    Rosenthal & Wei 2001) sets those rates to zero and restores zero row sums on the diagonal; its
    price is a small mismatch exp(Q) != P(1), reported as `embedding_error`. Columns that are not rows
    (default, 'D') are treated as absorbing. At exactly one year the input is returned UNCHANGED (no
    round trip through log/exp), so the one-year engine reprices exactly as before.

    Returns (matrix over the horizon, shaped like the input; info dict).
    """
    m = transition_matrix
    rows, cols = list(m.index), list(m.columns)
    info = {"horizon_years": float(horizon_years), "method": "one-year matrix as is",
            "n_negative_rates": 0, "worst_negative_rate": 0.0, "embedding_error": 0.0}
    if abs(float(horizon_years) - 1.0) < 1e-12:
        return m, info
    n = len(cols)
    P = np.eye(n)                                    # absorbing by default (rows missing from input)
    for r in rows:
        P[cols.index(r), :] = m.loc[r].to_numpy(dtype=float)
    Q = np.real(logm(P))
    off = ~np.eye(n, dtype=bool)
    neg = off & (Q < 0.0)
    info["n_negative_rates"] = int((off & (Q < -1e-12)).sum())
    info["worst_negative_rate"] = float(Q[neg].min()) if neg.any() else 0.0
    Q[neg] = 0.0
    np.fill_diagonal(Q, 0.0)
    np.fill_diagonal(Q, -Q.sum(axis=1))
    info["embedding_error"] = float(np.abs(expm(Q) - P).max())
    Ph = np.clip(expm(float(horizon_years) * Q), 0.0, None)
    Ph = Ph / Ph.sum(axis=1, keepdims=True)
    out = pd.DataFrame([Ph[cols.index(r), :] for r in rows], index=rows, columns=cols)
    info["method"] = "generator, exp(h Q), negative rates zeroed"
    return out, info


def build_migration_thresholds(transition_matrix):
    """Latent-variable thresholds for the Gaussian-copula (CreditMetrics) migration model.

    For each current rating, a standard-normal latent Y determines the horizon rating: ordering the
    horizon states from worst (default) to best (AAA), the default band is the lowest Phi^{-1}(p_D) of
    the latent, the next band CCC, and so on. Returns, per current rating, the ascending thresholds
    and the worst->best ordered horizon-rating labels they separate.
    """
    cols = list(transition_matrix.columns)              # [AAA..CCC, D]
    worst_to_best = list(reversed(cols))                # [D, CCC, B, ..., AAA]
    thresholds = {}
    for cur in transition_matrix.index:
        p = transition_matrix.loc[cur, worst_to_best].to_numpy(dtype=float)
        cum = np.cumsum(p)[:-1]                          # exclude the last (=1 -> +inf)
        cum = np.clip(cum, 1e-12, 1 - 1e-12)
        thresholds[cur] = norm.ppf(cum)                 # ascending thresholds (len = n_states - 1)
    return {"thresholds": thresholds, "worst_to_best": worst_to_best}


def simulate_horizon_ratings(current_rating, systematic_z, loading, migration_thresholds, rng):
    """Draw each scenario's horizon rating for a name currently at `current_rating`.

    Latent Y = loading * M + sqrt(1 - loading^2) * eps, where M = `systematic_z` is the shared
    systematic credit factor per scenario (oriented so that a credit blow-out is a LOW M, pushing Y
    down into the downgrade/default bands) and eps is name-specific idiosyncratic noise. Because every
    name shares M, downgrades and defaults cluster in the bad-credit scenarios (wrong-way dependence).
    Returns the horizon rating label per scenario and the boolean default mask.
    """
    M = np.asarray(systematic_z, dtype=float)
    n = M.shape[0]
    eps = rng.standard_normal(n)
    Y = loading * M + np.sqrt(max(1.0 - loading ** 2, 0.0)) * eps
    thr = migration_thresholds["thresholds"][current_rating]
    order = migration_thresholds["worst_to_best"]
    band = np.searchsorted(thr, Y)                       # 0..len(order)-1, low Y -> worst band
    labels = np.array(order, dtype=object)[band]
    return labels, (labels == "D")


def reprice_corporate_bond_with_migration(eval_date, payment_dates, key_rate_scenarios, tenor_years,
                                          all_spread_scenarios, horizon_ratings, coupon_rate,
                                          recovery_rate=0.40, face_value=100.0, freq=2):
    """Per-scenario horizon value of a corporate bond WITH rating migration / default (1Y horizon).

    Each scenario s reprices the bond on the simulated Treasury curve plus the simulated OAS of its
    *horizon* rating (so a downgraded name is discounted at the wider migrated-rating spread, an
    upgraded name at the tighter one); a defaulted name is worth `recovery_rate` * face. The migrated
    spread is read from `all_spread_scenarios` (the simulated OAS path for every rating bucket), so the
    migration P&L and the spread-MtM P&L share the same simulated credit state.
    """
    curves = np.asarray(key_rate_scenarios, dtype=float)
    S = curves.shape[0]
    ratings = np.asarray(horizon_ratings, dtype=object)
    spreads_by_rating = {r: np.asarray(all_spread_scenarios[r], dtype=float) for r in all_spread_scenarios.columns}
    values = np.empty(S, dtype=float)
    for s in range(S):
        r = ratings[s]
        if r == "D":
            values[s] = recovery_rate * face_value
        else:
            sp = float(spreads_by_rating[r][s]) if r in spreads_by_rating else 0.0
            values[s] = price_coupon_bond_from_curve(
                eval_date, payment_dates, curves[s], tenor_years,
                coupon_rate=coupon_rate, face_value=face_value, freq=freq, spread=sp,
            )
    return pd.Series(values)


def coupons_paid_between(issue_date, horizon_date, payment_dates, coupon_rate, face_value=100.0, freq=2):
    """Total coupon cash paid between issue and horizon (no reinvestment), for horizon total return.

    Over a one-week horizon this is ~0 (no coupon usually falls in the window); over a one-year horizon
    it is roughly one full year of coupon, a material part of a bond's total return. Every horizon value
    adds it to the repriced PV (in-sample AND out-of-sample), so the total-return basis never differs.
    """
    issue_date = pd.Timestamp(issue_date); horizon_date = pd.Timestamp(horizon_date)
    paid = payment_dates[(payment_dates > issue_date) & (payment_dates <= horizon_date)]
    return float(len(paid) * coupon_rate * face_value / freq)


def accrued_interest(eval_date, payment_dates, coupon_rate=0.02, face_value=100.0, freq=2):
    """Accrued interest at eval_date (actual/actual within the current coupon period).

    Issue #10: included so portfolio P&L can be reported on a carry-consistent (clean vs dirty)
    basis, with clean = dirty - accrued. Over a one-week horizon with semiannual coupons this is
    immaterial (order 0.03 on a 2% note) but it is the technically-correct decomposition.
    """
    eval_date = pd.Timestamp(eval_date)
    future = payment_dates[payment_dates > eval_date]
    if len(future) == 0:
        return 0.0
    next_cpn = future[0]
    months_step = 12 // freq
    prev_cpn = next_cpn - pd.DateOffset(months=months_step)
    period_days = (next_cpn - prev_cpn).days
    accrued_days = max((eval_date - prev_cpn).days, 0)
    frac = accrued_days / period_days if period_days > 0 else 0.0
    return float(coupon_rate * face_value / freq * frac)


def price_fx_forward_mtm(eval_date, maturity_date, spot, usd_key_rate_series, usd_tenor_years,
                         strike, foreign_funding_rate, notional_foreign=1.0):
    """USD mark-to-market of a long-foreign FX forward (receive 1 foreign unit, pay `strike` USD per
    unit, at maturity), valued at `eval_date`.

    Decomposed as a long foreign zero-coupon bond minus a short USD zero-coupon bond:

        MTM_USD = notional_foreign * [ spot * exp(-r_foreign * tau) - strike * DF_USD(tau) ]

    where DF_USD(tau) = exp(-z_USD(tau) * tau) comes from the (simulated) USD zero curve and
    r_foreign is the foreign funding rate; tau is the year-fraction to maturity. `spot` is USD per
    one foreign unit (FRED convention, e.g. USD per EUR), so the long-foreign value rises with spot.

    Declared one-week approximation (covered interest parity): r_foreign is held constant over the
    horizon, so the forward-points / carry term is fixed and the P&L is driven by the spot move and
    USD discounting. At a one-week horizon the carry is immaterial (order a few bp) next to weekly
    spot volatility. This reuses the same continuously-compounded zero-curve convention (and its
    declared par-yield ~= zero-rate approximation) as the bond pricer.
    """
    eval_date = pd.Timestamp(eval_date)
    tau = max((pd.Timestamp(maturity_date) - eval_date).days / 365.25, 0.0)
    zero_curve = make_zero_curve_from_key_rates(usd_tenor_years, np.asarray(usd_key_rate_series, dtype=float))
    df_usd = float(np.exp(-zero_curve(tau) * tau))
    df_for = float(np.exp(-float(foreign_funding_rate) * tau))
    return float(notional_foreign) * (float(spot) * df_for - float(strike) * df_usd)


def atm_forward_fx_strike(eval_date, maturity_date, spot, usd_key_rate_series, usd_tenor_years,
                          foreign_funding_rate):
    """At-the-money forward FX rate (USD per foreign unit) by covered interest parity, so a freshly
    struck forward has zero MTM at inception:  K = spot * DF_foreign(tau) / DF_USD(tau)."""
    eval_date = pd.Timestamp(eval_date)
    tau = max((pd.Timestamp(maturity_date) - eval_date).days / 365.25, 0.0)
    zero_curve = make_zero_curve_from_key_rates(usd_tenor_years, np.asarray(usd_key_rate_series, dtype=float))
    df_usd = float(np.exp(-zero_curve(tau) * tau))
    df_for = float(np.exp(-float(foreign_funding_rate) * tau))
    return float(spot) * df_for / df_usd


def reprice_fx_forward_scenarios(eval_date, maturity_date, spot_scenarios, usd_key_rate_scenarios,
                                 usd_tenor_years, strike, foreign_funding_rate, notional_foreign=1.0):
    """Per-scenario USD MTM of the FX forward, using the simulated spot (row s) jointly with the
    simulated USD curve (row s), so an FX move and a Treasury move hit the forward together. Light
    Python loop; each valuation is a couple of interpolations, not the simulation bottleneck.
    """
    spots = np.asarray(spot_scenarios, dtype=float)
    curves = np.asarray(usd_key_rate_scenarios, dtype=float)
    mtm = np.empty(spots.shape[0], dtype=float)
    for s in range(spots.shape[0]):
        mtm[s] = price_fx_forward_mtm(
            eval_date, maturity_date, spots[s], curves[s], usd_tenor_years,
            strike=strike, foreign_funding_rate=foreign_funding_rate, notional_foreign=notional_foreign,
        )
    return pd.Series(mtm)


def rolling_horizon_log_returns(log_returns, window):
    return log_returns.rolling(window).sum().dropna()


def rolling_horizon_yield_changes(yield_changes, window):
    return yield_changes.rolling(window).sum().dropna()


def portfolio_metrics_from_instrument_returns(weights, instrument_returns, alpha=0.05):
    w = pd.Series(weights, index=instrument_returns.columns, dtype=float)
    w = w / w.sum()
    portfolio_return = instrument_returns.mul(w, axis=1).sum(axis=1)
    metrics = loss_summary_from_pnl(portfolio_return, alpha=alpha)
    metrics["mean_abs_weight"] = w.abs().mean()
    return portfolio_return, metrics


def inverse_vol_weights_from_scenarios(instrument_returns):
    vol = instrument_returns.std()
    inv_vol = 1.0 / vol.replace(0, np.nan)
    w = inv_vol / inv_vol.sum()
    return w.fillna(0.0)


def tail_contribution_table(portfolio_return, instrument_returns, weights, alpha=0.05):
    w = pd.Series(weights, index=instrument_returns.columns, dtype=float)
    w = w / w.sum()
    threshold = portfolio_return.quantile(alpha)
    tail_mask = portfolio_return <= threshold
    if tail_mask.sum() == 0:
        return pd.DataFrame()

    weighted_return_contrib = instrument_returns.mul(w, axis=1)
    avg_tail_return_contrib = weighted_return_contrib.loc[tail_mask].mean()
    avg_tail_loss_contrib = -avg_tail_return_contrib
    portfolio_es_loss = float(avg_tail_loss_contrib.sum())
    share = avg_tail_loss_contrib / portfolio_es_loss if portfolio_es_loss != 0 else np.nan

    return pd.DataFrame({
        "weight": w,
        "avg_tail_return_contribution": avg_tail_return_contrib,
        "avg_tail_loss_contribution": avg_tail_loss_contrib,
        "share_of_portfolio_ES_loss": share,
    }).sort_values("avg_tail_loss_contribution", ascending=False)


def search_long_only_min_cvar_portfolio(instrument_returns, alpha=0.05, max_weight=None,
                                        n_draws=None, seed=None, chunk_size=None):
    """Exact long-only minimum-CVaR portfolio via the Rockafellar-Uryasev LP (issue #3).

    Replaces the previous random simplex search with the convex linear program

        min_{w, zeta, u}   zeta + 1/(alpha * S) * sum_s u_s
        s.t.   u_s >= -(r_s . w) - zeta,   u_s >= 0
               sum_i w_i = 1,   0 <= w_i <= max_weight

    where r_s is the instrument-return vector in scenario s and S the number of scenarios. Solved
    with scipy.optimize.linprog(method='highs'). This returns the true optimum, is deterministic
    and reproducible, scales with the number of assets, and does not discard most draws under a
    tight max-weight cap (the old rejection sampler did). The legacy `n_draws / seed / chunk_size`
    arguments are accepted but ignored for backward compatibility.
    """
    R = instrument_returns.to_numpy()
    S, n = R.shape
    names = instrument_returns.columns.tolist()

    c = np.concatenate([np.zeros(n), [1.0], np.full(S, 1.0 / (alpha * S))])

    # inequality block:  -(R w) - zeta - u <= 0   (one row per scenario)
    A_ub = sparse.hstack([
        sparse.csr_matrix(-R),
        sparse.csr_matrix(-np.ones((S, 1))),
        -sparse.identity(S, format="csr"),
    ], format="csr")
    b_ub = np.zeros(S)

    # equality block:  sum w = 1
    A_eq = sparse.hstack([
        sparse.csr_matrix(np.ones((1, n))),
        sparse.csr_matrix((1, 1)),
        sparse.csr_matrix((1, S)),
    ], format="csr")
    b_eq = np.array([1.0])

    w_upper = max_weight if max_weight is not None else 1.0
    bounds = [(0.0, w_upper)] * n + [(None, None)] + [(0.0, None)] * S

    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    if not res.success:
        raise RuntimeError(f"min-CVaR LP failed to solve: {res.message}")

    w = np.clip(res.x[:n], 0.0, None)
    w = w / w.sum()
    port = R @ w
    q = float(np.quantile(port, alpha))
    tail = port[port <= q]
    es = float(-tail.mean()) if tail.size else np.nan

    weights = pd.Series(w, index=names, name="min_cvar_long_only")
    diagnostics = pd.Series({
        "n_scenarios": float(S),
        "VaR_95_loss": float(-q),
        "ES_95_loss": es,
        "mean": float(port.mean()),
        "std": float(port.std(ddof=1)),
    }, name="min_cvar_lp_diagnostics")
    return weights, diagnostics


def kupiec_pof_test(n_violations, n_obs, alpha):
    """Kupiec proportion-of-failures (unconditional coverage) LR test.

    H0: the VaR violation rate equals `alpha`. Returns (LR_stat, p_value); LR_stat ~ chi2(1).
    The expected number of violations is alpha * n_obs.
    """
    x, N, p = int(n_violations), int(n_obs), float(alpha)
    if N == 0:
        return np.nan, np.nan
    pi_hat = x / N
    if x == 0:
        lr = -2.0 * (N * np.log(1 - p))
    elif x == N:
        lr = -2.0 * (N * np.log(p))
    else:
        ll_null = x * np.log(p) + (N - x) * np.log(1 - p)
        ll_alt = x * np.log(pi_hat) + (N - x) * np.log(1 - pi_hat)
        lr = -2.0 * (ll_null - ll_alt)
    return float(lr), float(chi2.sf(lr, df=1))


def christoffersen_independence_test(violations):
    """Christoffersen independence LR test on a 0/1 violation sequence.

    H0: violations are serially independent (no clustering of breaches). Returns (LR_stat, p_value);
    LR_stat ~ chi2(1).
    """
    v = np.asarray(violations, dtype=int)
    if len(v) < 2:
        return np.nan, np.nan
    n00 = n01 = n10 = n11 = 0
    for prev, cur in zip(v[:-1], v[1:]):
        if prev == 0 and cur == 0:
            n00 += 1
        elif prev == 0 and cur == 1:
            n01 += 1
        elif prev == 1 and cur == 0:
            n10 += 1
        else:
            n11 += 1
    denom01 = n00 + n01
    denom11 = n10 + n11
    pi01 = n01 / denom01 if denom01 > 0 else 0.0
    pi11 = n11 / denom11 if denom11 > 0 else 0.0
    pi = (n01 + n11) / (n00 + n01 + n10 + n11)

    def _safe(p):
        return min(max(p, 1e-12), 1 - 1e-12)

    ll_null = (n00 + n10) * np.log(1 - _safe(pi)) + (n01 + n11) * np.log(_safe(pi))
    ll_alt = (n00 * np.log(1 - _safe(pi01)) + n01 * np.log(_safe(pi01))
              + n10 * np.log(1 - _safe(pi11)) + n11 * np.log(_safe(pi11)))
    lr = -2.0 * (ll_null - ll_alt)
    return float(lr), float(chi2.sf(lr, df=1))


def contiguous_window_start(index, tol_days):
    """Start date after the last internal calendar gap longer than `tol_days` (None if no such gap).

    A day-to-day change taken across a multi-year data hole (the known case: the 30-year Treasury
    discontinuation 2002-02 -> 2006-02) would enter the bootstrap pool as ONE fake extreme
    observation -- poison for a tail engine. The main data cell and the walk-forward backtest's
    per-as-of refit must apply the SAME rule, or an early backtest window could silently contain a
    hole the main window trims away. Returns (start_or_None, n_gaps, max_gap_days).
    """
    idx = pd.DatetimeIndex(index)
    if len(idx) < 2:
        return None, 0, 0
    gaps = idx.to_series().diff().dt.days
    big = gaps[gaps > float(tol_days)]
    if len(big) == 0:
        return None, 0, int(np.nanmax(gaps.to_numpy()[1:])) if len(idx) > 1 else 0
    return big.index[-1], int(len(big)), int(np.nanmax(gaps.to_numpy()[1:]))


def christoffersen_conditional_coverage_test(violations, alpha):
    """Christoffersen (1998) JOINT conditional-coverage LR test on a 0/1 violation sequence.

    H0: violations occur at rate `alpha` AND are serially independent. The statistic decomposes
    exactly as LR_cc = LR_uc (Kupiec) + LR_ind (Christoffersen independence), so it is chi2(2).
    This is the test the two separate p-values do not give: a VaR can pass unconditional coverage
    while its breaches cluster (a model that is right on average and wrong exactly when it matters),
    or pass independence while systematically over/under-covering. Returns (LR_stat, p_value).
    """
    v = np.asarray(violations, dtype=int)
    if len(v) < 2:
        return np.nan, np.nan
    lr_uc, _ = kupiec_pof_test(int(v.sum()), int(len(v)), float(alpha))
    lr_ind, _ = christoffersen_independence_test(v)
    if not np.isfinite(lr_uc) or not np.isfinite(lr_ind):
        return np.nan, np.nan
    lr_cc = float(max(lr_uc, 0.0) + max(lr_ind, 0.0))
    return lr_cc, float(chi2.sf(lr_cc, df=2))


def coverage_summary(violations, alpha):
    """Unconditional (Kupiec), independence and joint conditional-coverage results as one dict."""
    v = np.asarray(violations, dtype=int)
    n, x = int(len(v)), int(v.sum())
    lr_k, p_k = kupiec_pof_test(x, n, alpha) if n > 0 else (np.nan, np.nan)
    lr_i, p_i = christoffersen_independence_test(v) if n >= 2 else (np.nan, np.nan)
    lr_cc, p_cc = christoffersen_conditional_coverage_test(v, alpha) if n >= 2 else (np.nan, np.nan)
    return {"observations": n, "violations": x, "expected": alpha * n,
            "violation_rate": (x / n) if n else np.nan,
            "kupiec_LR": lr_k, "kupiec_p": p_k,
            "indep_LR": lr_i, "indep_p": p_i,
            "cc_LR": lr_cc, "cc_p": p_cc}


def sample_expected_shortfall(sample, alpha=0.05):
    """ES at level `alpha` as a POSITIVE loss: minus the mean of the sample below its alpha-quantile.

    Matches the VaR convention used by the engine (`VaR = -quantile(alpha)`), so ES >= VaR >= 0 for
    a loss-bearing distribution.
    """
    s = np.asarray(sample, dtype=float)
    if s.size == 0:
        return np.nan
    q = float(np.quantile(s, alpha))
    tail = s[s <= q]
    return float(-tail.mean()) if tail.size > 0 else float(-q)


def _as_z_statistics(X, V, E, alpha, which):
    """Vectorized Acerbi-Szekely Z1/Z2 over a matrix of return paths X (n_draws, T)."""
    ind = X < -V[None, :]
    contrib = np.where(ind, X / E[None, :], 0.0)
    if which == 2:
        T = X.shape[1]
        return contrib.sum(axis=1) / (T * alpha) + 1.0
    counts = ind.sum(axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        z1 = contrib.sum(axis=1) / np.maximum(counts, 1) + 1.0
    return np.where(counts > 0, z1, np.nan)


def acerbi_szekely_es_test(realized, var_levels, es_levels, scenario_matrix, alpha=0.05,
                           n_sim=2000, rng=None, which=2):
    """Acerbi-Szekely (2014) expected-shortfall backtest, Test 1 (which=1) or Test 2 (which=2).

    The portfolio layer minimizes CVaR, so validating only the VaR leaves the objective function
    itself untested. ES has no elicitability, hence no scoring-function backtest -- but it does admit
    these direct tests, because the engine emits the FULL predictive distribution at each as-of date,
    not just a quantile.

        I_t = 1{X_t < -VaR_t}
        Z2  = (1/(T*alpha)) * sum_t X_t * I_t / ES_t + 1     (unconditional: severity AND frequency)
        Z1  = (1/N_viol)    * sum_t X_t * I_t / ES_t + 1     (conditional: severity GIVEN a breach)

    Under H0 (the predictive distribution is correct) E[Z] = 0. A NEGATIVE realized Z means losses in
    the tail were worse than the model's own ES -- the ES is understated. The null distribution is
    obtained by resampling, for each date t, a return from the SAME predictive scenario set the engine
    produced at t (rows of `scenario_matrix`), holding VaR_t and ES_t fixed: no distributional
    assumption is imposed. The p-value is one-sided on the left (P[Z_sim <= Z_obs]), the direction
    that matters for risk. Z1 is undefined with zero breaches (nothing to condition on); Z2 is not,
    and with zero breaches Z2 = 1 > 0, i.e. evidence of a conservative, not an understated, ES.
    """
    X = np.asarray(realized, dtype=float)
    V = np.asarray(var_levels, dtype=float)
    E = np.asarray(es_levels, dtype=float)
    S = np.asarray(scenario_matrix, dtype=float)
    T = X.size
    out = {"statistic": np.nan, "p_value": np.nan, "n_obs": T, "n_violations": int((X < -V).sum()),
           "null_mean": np.nan, "n_sim": int(n_sim), "which": int(which)}
    if T == 0 or S.ndim != 2 or S.shape[0] != T or np.any(~np.isfinite(E)) or np.any(E <= 0):
        return out
    z_obs = float(_as_z_statistics(X[None, :], V, E, alpha, which)[0])
    if not np.isfinite(z_obs):
        return out                                   # Z1 with zero breaches: undefined by construction
    rng = np.random.default_rng(0) if rng is None else rng
    idx = rng.integers(0, S.shape[1], size=(int(n_sim), T))      # (n_sim, T) column-wise resample
    Xs = S[np.arange(T)[None, :], idx]                           # draw t-th return from t-th scenario set
    z_sim = _as_z_statistics(Xs, V, E, alpha, which)
    z_sim = z_sim[np.isfinite(z_sim)]
    if z_sim.size == 0:
        return out
    out["statistic"] = z_obs
    out["p_value"] = float((np.sum(z_sim <= z_obs) + 1.0) / (z_sim.size + 1.0))
    out["null_mean"] = float(z_sim.mean())
    return out


FF_PRICED_FACTORS = ("MKT_RF", "SMB", "HML", "MOM")


def apply_drift_mode(equity_block, rates_block, credit_block, fx_block, mode, ff_long=None,
                     asof=None, ff_window=None):
    """Set the engine's EXPECTED daily changes according to DRIFT_MODE (v22).

    The simulator adds a constant daily mean to every block: `mean_excess_returns` (equities),
    `mean_dy` (key rates), `mean_dspread` (credit) and `mean_fx_logret` (FX); the factor drivers are
    centered, so nothing else carries a drift. Up to v21 all four were the SAMPLE means of the
    estimation window. This function swaps them, returning shallow copies (the inputs are untouched):

      "sample" -- the inputs, unchanged (byte-identical);
      "factor" -- mu_i = beta_i' lambda with lambda the long-run Fama-French daily premia from `ff_long`
                  up to `asof` (point-in-time), alpha = 0; a factor outside the Fama-French set is
                  priced through its market beta (OLS slope on MKT_RF over the window x lambda_MKT).
                  Rates, credit and FX drift at zero;
      "zero"   -- every drift zero (equities then earn the risk-free rate only).

    Returns (equity_block, rates_block, credit_block, fx_block, info).
    """
    if mode == "sample":
        return equity_block, rates_block, credit_block, fx_block, {"mode": "sample"}
    if mode not in ("factor", "zero"):
        raise ValueError(f"Unknown DRIFT_MODE {mode!r}; expected 'sample', 'factor' or 'zero'")

    def _zeroed(block, key):
        if block is None:
            return None
        out = dict(block)
        out[key] = block[key] * 0.0
        return out

    rb = _zeroed(rates_block, "mean_dy")
    cb = _zeroed(credit_block, "mean_dspread")
    fb = _zeroed(fx_block, "mean_fx_logret")
    eb = dict(equity_block)
    info = {"mode": mode, "sample_mean_excess": equity_block["mean_excess_returns"].copy(), "lambda": None}
    if mode == "zero":
        eb["mean_excess_returns"] = equity_block["mean_excess_returns"] * 0.0
    else:
        if ff_long is None:
            raise ValueError("DRIFT_MODE='factor' needs the long Fama-French history (ff_long)")
        hist = ff_long.loc[:pd.Timestamp(asof)] if asof is not None else ff_long
        lam_ff = hist[[c for c in FF_PRICED_FACTORS if c in hist.columns]].mean()
        betas = equity_block["betas"]
        lam, src = {}, {}
        for col in betas.columns:
            if col in lam_ff.index:
                lam[col], src[col] = float(lam_ff[col]), "Fama-French long-run mean"
            else:
                f = equity_block["factors"][col]
                mkt = ff_window["MKT_RF"].reindex(f.index)
                ok = f.notna() & mkt.notna()
                b = float(np.cov(f[ok], mkt[ok])[0, 1] / np.var(mkt[ok], ddof=1))
                lam[col], src[col] = b * float(lam_ff["MKT_RF"]), f"market beta {b:.2f} x MKT_RF premium"
        lam = pd.Series(lam)
        eb["mean_excess_returns"] = pd.Series(betas.to_numpy() @ lam.reindex(betas.columns).to_numpy(),
                                              index=betas.index)
        info.update({"lambda": lam, "lambda_source": pd.Series(src),
                     "history_start": hist.index[0], "history_end": hist.index[-1]})
    info["drift_excess"] = eb["mean_excess_returns"]
    return eb, rb, cb, fb, info


# ---- v26: regime-adaptive within-block factor geometry ----------------------------------------------
# The v24 hierarchy already estimates its cross-asset transmission structure with exponentially
# weighted moments.  v26 gives the BLOCKWISE engine the same class of current-regime information
# INSIDE each asset block: weighted means, weighted OLS betas, and weighted PCA bases.  Historical
# shocks are still projected through today's basis and the stationary bootstrap remains empirical;
# no dynamic-correlation/DCC layer is introduced in this version.

def _v26_decay_weights(n, halflife_days):
    """Normalized exponential weights, newest observation last. None -> equal weights."""
    n = int(n)
    if n < 1:
        raise ValueError("need at least one observation")
    if halflife_days is None or not np.isfinite(float(halflife_days)):
        return np.full(n, 1.0 / n)
    h = float(halflife_days)
    if h <= 0:
        raise ValueError("halflife_days must be positive")
    w = 0.5 ** ((n - 1 - np.arange(n, dtype=float)) / h)
    return w / w.sum()


def _v26_weighted_mean_frame(df, w):
    A = np.asarray(df, dtype=float)
    return pd.Series((A * np.asarray(w)[:, None]).sum(axis=0), index=df.columns)


def _v26_weighted_cov_array(A, w, center=None):
    A = np.asarray(A, dtype=float)
    w = np.asarray(w, dtype=float)
    if center is None:
        center = (A * w[:, None]).sum(axis=0)
    X = A - np.asarray(center, dtype=float)[None, :]
    return (X * w[:, None]).T @ X


def _v26_weighted_std_frame(df, w):
    mu = _v26_weighted_mean_frame(df, w).to_numpy()
    v = np.diag(_v26_weighted_cov_array(df.to_numpy(), w, center=mu))
    return pd.Series(np.sqrt(np.clip(v, 0.0, None)), index=df.columns)


def _v26_weighted_pca(changes, explained_var_target, prefix, min_factors=1, max_factors=None,
                      mean_key="mean", halflife_days=378.0):
    """Weighted covariance PCA; every historical row is projected on the CURRENT weighted basis."""
    X = changes.copy().dropna()
    w = _v26_decay_weights(len(X), halflife_days)
    mu = _v26_weighted_mean_frame(X, w)
    C = _v26_weighted_cov_array(X.to_numpy(), w, center=mu.to_numpy())
    ev, V = np.linalg.eigh(np.atleast_2d(C))
    order = np.argsort(ev)[::-1]
    ev = np.clip(ev[order], 0.0, None)
    V = V[:, order]
    total = float(ev.sum())
    ratio_full = ev / total if total > 0 else np.r_[1.0, np.zeros(max(len(ev)-1, 0))]
    cum = np.cumsum(ratio_full)
    k = int(np.searchsorted(cum, float(explained_var_target)) + 1)
    k = max(int(min_factors), min(k, X.shape[1]))
    if max_factors is not None:
        k = max(int(min_factors), min(k, int(max_factors)))
    load = V[:, :k]
    centered = X - mu
    scores = pd.DataFrame(centered.to_numpy() @ load, index=X.index,
                          columns=[f"{prefix}_pc{i+1}" for i in range(k)])
    loadings = pd.DataFrame(load, index=X.columns, columns=scores.columns)
    ratio = pd.Series(ratio_full[:k], index=scores.columns, name="explained_variance_ratio")
    residuals = centered - scores.to_numpy() @ loadings.to_numpy().T
    return {
        "scores": scores,
        "loadings": loadings,
        mean_key: mu,
        "explained_ratio": ratio,
        "cum_explained_ratio": ratio.cumsum(),
        "n_factors": k,
        "pca": None,
        "residuals": residuals,
        "v26_weights": pd.Series(w, index=X.index),
        "v26_halflife_days": halflife_days,
        "v26_eigenvalues": pd.Series(ev[:k], index=scores.columns),
    }



def _v26_compress_residual_block(residuals, resid_explained_var_target=0.95, halflife_days=None):
    out = _v26_weighted_pca(residuals, resid_explained_var_target, prefix="resid",
                            min_factors=1, max_factors=max(1, residuals.shape[1] - 1),
                            mean_key="_resid_mean", halflife_days=halflife_days)
    # Residuals are reconstructed in the weighted-current basis.  The omitted variance re-injected
    # by the simulator is also measured with the same current-regime weights.
    w = out["v26_weights"].to_numpy()
    leftover = out["residuals"]
    idio_std = _v26_weighted_std_frame(leftover, w)
    return {
        "resid_scores": out["scores"],
        "resid_loadings": out["loadings"],
        "resid_explained_ratio": out["explained_ratio"].rename("resid_explained_variance_ratio"),
        "resid_cum_explained_ratio": out["explained_ratio"].cumsum().rename("resid_cum_explained_ratio"),
        "resid_idiosyncratic_std": idio_std.rename("resid_idiosyncratic_std"),
        "resid_pca": None,
        "v26_resid_weights": out["v26_weights"],
        "v26_halflife_days": halflife_days,
    }


def build_factor_driven_equity_block_weighted(excess_returns, factor_df, block_name,
                                              resid_explained_var_target=0.95, halflife_days=378.0):
    common_dates = excess_returns.index.intersection(factor_df.index)
    y = excess_returns.loc[common_dates].copy()
    f = factor_df.loc[common_dates].copy()
    w = _v26_decay_weights(len(common_dates), halflife_days)
    mu_excess = _v26_weighted_mean_frame(y, w)
    mu_factors = _v26_weighted_mean_frame(f, w)
    centered_y = y - mu_excess
    centered_f = f - mu_factors
    sw = np.sqrt(w)[:, None]
    X = centered_f.to_numpy()
    Y = centered_y.to_numpy()
    beta_array = np.linalg.lstsq(X * sw, Y * sw, rcond=None)[0].T
    betas = pd.DataFrame(beta_array, index=y.columns, columns=centered_f.columns)
    fitted = pd.DataFrame(X @ beta_array.T, index=y.index, columns=y.columns)
    residuals = centered_y - fitted
    sse = (residuals.to_numpy() ** 2 * w[:, None]).sum(axis=0)
    sst = (centered_y.to_numpy() ** 2 * w[:, None]).sum(axis=0)
    asset_r2 = pd.Series(1.0 - sse / np.maximum(sst, 1e-300), index=y.columns)
    residual_block = _v26_compress_residual_block(residuals, resid_explained_var_target,
                                                   halflife_days=halflife_days)
    return {
        "engine_name": block_name,
        "factors": centered_f,
        "factor_means": mu_factors,
        "betas": betas,
        "residuals": residuals,
        "asset_r2": asset_r2,
        "mean_excess_returns": mu_excess,
        "excess_returns": y,
        "v26_weights": pd.Series(w, index=common_dates),
        "v26_halflife_days": halflife_days,
        **residual_block,
    }


def build_latent_statistical_equity_block_weighted(log_returns, rf_daily,
                                                   resid_explained_var_target=0.95,
                                                   halflife_days=378.0):
    excess_returns, short_rate_daily = compute_excess_log_returns_daily(log_returns, rf_daily)
    w = _v26_decay_weights(len(excess_returns), halflife_days)
    mu = _v26_weighted_mean_frame(excess_returns, w)
    X = excess_returns - mu
    C = _v26_weighted_cov_array(X.to_numpy(), w, center=np.zeros(X.shape[1]))
    ev, V = np.linalg.eigh(C)
    v = V[:, np.argmax(ev)]
    # deterministic sign: positive exposure to the equal-weight equity direction
    if float(v.sum()) < 0:
        v = -v
    factor_scores = pd.DataFrame(X.to_numpy() @ v[:, None], index=X.index,
                                 columns=["latent_equity_factor"])
    block = build_factor_driven_equity_block_weighted(
        excess_returns, factor_scores, "latent_statistical",
        resid_explained_var_target=resid_explained_var_target, halflife_days=halflife_days)
    block["short_rate_daily"] = short_rate_daily.reindex(block["excess_returns"].index)
    block["latent_explained_variance"] = float(np.max(ev) / max(float(np.sum(ev)), 1e-300))
    return block


def build_proxy_market_equity_block_weighted(log_returns, market_proxy_prices, rf_daily,
                                             resid_explained_var_target=0.95,
                                             halflife_days=378.0):
    excess_returns, short_rate_daily = compute_excess_log_returns_daily(log_returns, rf_daily)
    market_excess = compute_market_excess_log_returns_daily(market_proxy_prices, rf_daily)
    block = build_factor_driven_equity_block_weighted(
        excess_returns, market_excess.to_frame(name="market_erp_proxy"), "proxy_market",
        resid_explained_var_target=resid_explained_var_target, halflife_days=halflife_days)
    block["short_rate_daily"] = short_rate_daily.reindex(block["excess_returns"].index)
    block["market_excess"] = market_excess.reindex(block["excess_returns"].index)
    return block


def build_apt_equity_block_weighted(log_returns, rf_daily, ff_factor_panel, selected_factors, block_name,
                                    resid_explained_var_target=0.95, halflife_days=378.0):
    excess_returns, short_rate_daily = compute_excess_log_returns_daily(log_returns, rf_daily)
    block = build_factor_driven_equity_block_weighted(
        excess_returns, ff_factor_panel[selected_factors].copy(), block_name,
        resid_explained_var_target=resid_explained_var_target, halflife_days=halflife_days)
    block["short_rate_daily"] = short_rate_daily.reindex(block["excess_returns"].index)
    block["selected_factors"] = list(selected_factors)
    return block


def build_rates_factor_model_weighted(yield_changes, explained_var_target=0.995, halflife_days=378.0):
    return _v26_weighted_pca(yield_changes, explained_var_target, "rate", min_factors=2,
                             mean_key="mean_dy", halflife_days=halflife_days)


def build_credit_factor_model_weighted(spread_changes, explained_var_target=0.99, max_factors=None,
                                       halflife_days=378.0):
    return _v26_weighted_pca(spread_changes, explained_var_target, "credit", min_factors=2,
                             max_factors=max_factors, mean_key="mean_dspread",
                             halflife_days=halflife_days)


def build_fx_factor_block_weighted(fx_log_returns, explained_var_target=0.99, max_factors=None,
                                   halflife_days=378.0):
    return _v26_weighted_pca(fx_log_returns, explained_var_target, "fx", min_factors=1,
                             max_factors=max_factors, mean_key="mean_fx_logret",
                             halflife_days=halflife_days)


def v26_subspace_angle_deg(loadings_a, loadings_b):
    """Largest principal angle between two loading subspaces (degrees)."""
    A = np.asarray(loadings_a, dtype=float)
    B = np.asarray(loadings_b, dtype=float)
    qa, _ = np.linalg.qr(A)
    qb, _ = np.linalg.qr(B)
    s = np.linalg.svd(qa.T @ qb, compute_uv=False)
    s = np.clip(s, 0.0, 1.0)
    return float(np.degrees(np.arccos(np.min(s))))


# ---- v26: common half-life selector ---------------------------------------------------------------
# Headline principle: tune MEMORY, not ARCHITECTURE.  At every as-of date and horizon, one half-life
# is selected on a tiny model-agnostic panel of raw cross-asset observables, using ONLY earlier
# outcomes.  The SAME selected H is then handed to blockwise_adaptive and hierarchical_adaptive.
# Thus H cannot make one architecture win simply because it was tuned separately.

def _v26_halflife_label(days):
    if days is None or not np.isfinite(float(days)):
        return "static"
    return f"{float(days)/252.0:.2g}y"


def _v26_regularized_cov(C, ridge=0.05):
    C = np.asarray(C, dtype=float)
    d = np.clip(np.diag(C), 1e-12, None)
    D = np.diag(d)
    C = (1.0 - float(ridge)) * C + float(ridge) * D
    # numerical floor relative to average variance
    eps = max(float(np.mean(d)) * 1e-8, 1e-14)
    return C + eps * np.eye(C.shape[0])


def _v26_gaussian_cov_loss(y, mu, C):
    C = _v26_regularized_cov(C)
    d = np.asarray(y, dtype=float) - np.asarray(mu, dtype=float)
    sign, ldet = np.linalg.slogdet(C)
    if sign <= 0 or not np.isfinite(ldet):
        return np.inf
    try:
        q = float(d @ np.linalg.solve(C, d))
    except np.linalg.LinAlgError:
        q = float(d @ np.linalg.pinv(C) @ d)
    return float(ldet + q)


def _v26_model_agnostic_panel(lr, dy, ds):
    """Three raw observables: EW equity log-return, DGS10 change, representative credit-spread change."""
    eq = lr.mean(axis=1).rename("equity_ew")
    rcol = "DGS10" if "DGS10" in dy.columns else dy.columns[len(dy.columns)//2]
    cands = [c for c in ("Baa_spr", "BBB", "Baa", "BB") if c in ds.columns]
    ccol = cands[0] if cands else ds.columns[-1]
    out = pd.concat([eq, dy[rcol].rename("rates"), ds[ccol].rename("credit")], axis=1).dropna()
    return out


def v26_select_shared_halflife(panel, horizon_bdays, candidate_days, fallback_days=378.0,
                               min_train_days=504, max_origins=60, one_se=True):
    """Nested, point-in-time, model-agnostic half-life selection for one forecast horizon.

    For every inner historical origin s, estimate the h-day covariance from overlapping h-day
    aggregates ending no later than s, with candidate exponential memory H.  Score the NEXT realized
    h-day aggregate by Gaussian covariance loss.  Inner origins are h days apart, so target windows
    do not overlap.  The final choice uses the one-standard-error rule and therefore chooses the
    LONGEST memory statistically indistinguishable from the best mean loss -- deliberate protection
    against noisy short-memory overfit.  No observation after the outer as-of date enters.
    """
    X = panel.dropna().copy()
    h = int(horizon_bdays)
    n = len(X)
    if n < max(int(min_train_days), 4*h) + h + 5:
        return {"halflife_days": float(fallback_days), "n_origins": 0,
                "fallback": True, "scores": pd.DataFrame()}
    A = X.to_numpy(dtype=float)
    # prefix sums make each forward h-day target exact and cheap
    cs = np.vstack([np.zeros((1, A.shape[1])), np.cumsum(A, axis=0)])
    first = max(int(min_train_days), 4*h)
    origins = np.arange(first, n-h, h, dtype=int)
    if len(origins) > int(max_origins):
        origins = origins[-int(max_origins):]
    rows = []
    losses = {c: [] for c in candidate_days}
    for p in origins:
        # h-day historical aggregates ending on indices h-1 ... p
        Yhist = cs[h:p+2] - cs[:p+2-h]
        # there are p-h+2 aggregate rows, newest ends at p
        for cand in candidate_days:
            if cand is None:
                w = np.full(len(Yhist), 1.0/len(Yhist))
            else:
                w = _v26_decay_weights(len(Yhist), float(cand))
            mu = (Yhist * w[:, None]).sum(axis=0)
            C = _v26_weighted_cov_array(Yhist, w, center=mu)
            y_next = cs[p+h+1] - cs[p+1]
            losses[cand].append(_v26_gaussian_cov_loss(y_next, mu, C))
    for cand in candidate_days:
        arr = np.asarray(losses[cand], dtype=float)
        arr = arr[np.isfinite(arr)]
        rows.append({"halflife_days": np.inf if cand is None else float(cand),
                     "label": _v26_halflife_label(cand),
                     "mean_loss": float(arr.mean()) if len(arr) else np.inf,
                     "se_loss": float(arr.std(ddof=1)/np.sqrt(len(arr))) if len(arr) > 1 else np.inf,
                     "n_origins": int(len(arr))})
    tab = pd.DataFrame(rows).sort_values("halflife_days")
    finite = tab[np.isfinite(tab["mean_loss"])]
    if len(finite) == 0:
        return {"halflife_days": float(fallback_days), "n_origins": 0,
                "fallback": True, "scores": tab}
    ibest = finite["mean_loss"].idxmin()
    best = finite.loc[ibest]
    cutoff = float(best["mean_loss"] + (best["se_loss"] if one_se and np.isfinite(best["se_loss"]) else 0.0))
    elig = finite[finite["mean_loss"] <= cutoff]
    # one-SE: longest admissible memory; inf/static wins only if genuinely indistinguishable
    chosen_row = elig.sort_values("halflife_days").iloc[-1]
    hd = chosen_row["halflife_days"]
    if not np.isfinite(hd):
        hd = None
    return {"halflife_days": hd, "n_origins": int(best["n_origins"]),
            "fallback": False, "scores": tab, "best_mean_loss": float(best["mean_loss"]),
            "cutoff": cutoff}



## 1. Load and align market data

The equity universe is now intentionally broader than the earlier tech-only prototype. The cross-asset input layer includes:

- a **multi-sector US equity sample**,
- a **market proxy**,
- **Treasury key rates**,
- and daily **observable factor data** used by the APT engine.


In [ ]:
# Download and align the data (equity, Treasury curve, Fama-French factors, and the credit curve)
# Per-block sample starts. SAMPLE_MODE="short" reproduces the single START_DATE window exactly;
# "long" lets each block bring its full history -- the aligned window is whatever the data jointly
# support, and the diagnostic below shows which block binds.
def _block_start(block):
    return BLOCK_START_DATES.get(block, START_DATE) if SAMPLE_MODE == "long" else START_DATE


stock_prices_raw, market_proxy_raw, rates_raw = download_market_data(
    tickers=TICKERS,
    market_proxy=MARKET_PROXY,
    rate_tickers=RATE_TICKERS,
    start_date=_block_start("equity"),
    download_end=DOWNLOAD_END,
)

ff_factor_panel_raw = download_fama_french_daily_factors(
    start_date=_block_start("equity"),
    end_date=DOWNLOAD_END,
)
# Long-run factor premia for DRIFT_MODE="factor" (v22): the full Ken French daily history, sliced at
# each as-of date downstream so every premium estimate is point-in-time. Not downloaded otherwise.
ff_long_history = (download_fama_french_daily_factors(start_date=FF_LONG_RUN_START, end_date=DOWNLOAD_END)
                   if DRIFT_MODE == "factor" else None)

# Credit block: ICE BofA OAS by rating bucket (see note in the config cell on the FRED ~3y window).
credit_spreads_raw = download_credit_oas(
    credit_oas_series=CREDIT_OAS_SERIES,
    start_date=_block_start("credit"),
    download_end=DOWNLOAD_END,
)

# FX block: currency-pair spot from FRED (long history; see config note). One pair (EUR/USD) here.
fx_rates_raw = download_fx_rates(
    fx_series=FX_SERIES,
    start_date=_block_start("fx"),
    download_end=DOWNLOAD_END,
)

# Crisis backbone: long-history Moody's Baa/Aaa yields (decades incl. every postwar crisis). Pulled
# from MOODYS_START_DATE (independent of the short main-sample window) to calibrate the crisis tail.
moodys_raw = download_moodys_credit(
    moodys_series=MOODYS_SERIES,
    start_date=MOODYS_START_DATE,
    download_end=DOWNLOAD_END,
)

# Credit DRIVER panel per CREDIT_BLOCK_SOURCE. "oas_buckets": the 7-bucket OAS panel drives the
# credit factors (v18 behaviour; FRED serves a rolling ~3y of it). "moody_2pt": a real, never-
# truncated 2-point curve -- Moody's seasoned Aaa/Baa yields minus the 10Y Treasury, both decimals,
# daily since the 1980s -- drives them instead; the bucket panel is kept aside as today's repricing
# ANCHOR and for the sensitivity BRIDGE (built in the joint-state section).
credit_bucket_panel = credit_spreads_raw.dropna(how="any")
if CREDIT_BLOCK_SOURCE == "moody_2pt":
    _m10 = rates_raw["DGS10"].dropna()
    credit_driver_raw = pd.DataFrame({
        "Aaa_spr": moodys_raw["Aaa"] - _m10,
        "Baa_spr": moodys_raw["Baa"] - _m10,
    }).dropna(how="any")
else:
    credit_driver_raw = credit_spreads_raw

effective_date = choose_effective_asof(
    stock_prices_raw,
    market_proxy_raw,
    rates_raw,
    ff_factor_panel_raw,
    AS_OF_DATE,
    credit=credit_driver_raw,
    fx=fx_rates_raw,
)

stock_prices, market_proxy_prices, key_rates, ff_factor_panel, credit_spreads, fx_rates = align_market_data(
    stock_prices_raw,
    market_proxy_raw,
    rates_raw,
    ff_factor_panel_raw,
    effective_date,
    credit=credit_driver_raw,
    fx=fx_rates_raw,
)

# Long-window contiguity guard (LONG_GAP_TOLERANCE_DAYS in the config cell): a day-to-day change
# taken across a multi-year data hole -- the known case is the DGS30 discontinuation 2002-02 ->
# 2006-02 -- would enter the bootstrap pool as ONE fake extreme observation, poison for a tail
# engine. If the aligned window contains such a gap, trim it to start after the last one. The
# walk-forward backtest applies the same rule to every refitted window (same helper).
_win_start, _n_gaps, _max_gap = contiguous_window_start(stock_prices.index, LONG_GAP_TOLERANCE_DAYS)
if _win_start is not None:
    print(f"NOTE: aligned window contains {_n_gaps} internal gap(s) longer than "
          f"{LONG_GAP_TOLERANCE_DAYS} calendar days (largest: {_max_gap}d); trimming the "
          f"window start to {_win_start.date()} so no daily change spans a data hole.\n")
    stock_prices = stock_prices.loc[_win_start:]
    market_proxy_prices = market_proxy_prices.loc[_win_start:]
    key_rates = key_rates.loc[_win_start:]
    ff_factor_panel = ff_factor_panel.loc[_win_start:]
    credit_spreads = credit_spreads.loc[_win_start:]
    fx_rates = fx_rates.loc[_win_start:]

sector_table = pd.Series(SECTOR_MAP, name="sector").to_frame()

_credit_lbl = ("credit 2pt (Moody's - DGS10)" if CREDIT_BLOCK_SOURCE == "moody_2pt"
               else "credit OAS (all buckets)")
_first_usable = {
    "equity (all tickers)": stock_prices_raw.dropna(how="any").index.min(),
    "rates (all tenors)": rates_raw.dropna(how="any").index.min(),
    "FF factors": ff_factor_panel_raw.dropna(how="any").index.min(),
    _credit_lbl: credit_driver_raw.dropna(how="any").index.min(),
    "fx": fx_rates_raw.dropna(how="any").index.min(),
}
_binding = max(_first_usable, key=lambda kk: _first_usable[kk])
print(f"Sample mode               : {SAMPLE_MODE}   |   credit block source: {CREDIT_BLOCK_SOURCE}")
print("First usable date per block (all series present):")
for _k, _v in _first_usable.items():
    print(f"  {_k:28s}: {_v.date()}" + ("   <- binds the joint window" if _k == _binding else ""))
if CREDIT_BLOCK_SOURCE == "moody_2pt":
    print(f"  {'bucket OAS (anchor/bridge)':28s}: {credit_bucket_panel.index.min().date()} "
          "(kept OUT of the joint alignment; anchors repricing and calibrates the bridge)")
if SAMPLE_MODE == "long" and CREDIT_BLOCK_SOURCE == "oas_buckets":
    print("WARNING: SAMPLE_MODE='long' with CREDIT_BLOCK_SOURCE='oas_buckets' -- the FRED OAS "
          "panel's rolling ~3y truncation binds the joint window, so the long mode is inert "
          "(the v18 lesson). Set CREDIT_BLOCK_SOURCE='moody_2pt' to unlock the pre-2023 history.")
print(f"Aligned joint window      : {stock_prices.index.min().date()} -> {effective_date.date()}  "
      f"({len(stock_prices)} business days)")
print()
print(f"Configured AS_OF_DATE      : {AS_OF_DATE.date()}")
print(f"Effective evaluation date : {effective_date.date()}")
print(f"Equity sample shape       : {stock_prices.shape}")
print(f"Market proxy sample shape : {market_proxy_prices.shape}")
print(f"Rates sample shape        : {key_rates.shape}")
print(f"FF factor sample shape    : {ff_factor_panel.shape}")
print(f"Credit driver panel shape : {credit_spreads.shape}  (columns: {list(credit_spreads.columns)})")
print(f"Credit driver date range  : {credit_spreads.index.min().date()} -> {credit_spreads.index.max().date()}")
print(f"FX sample shape           : {fx_rates.shape}  (pairs: {list(fx_rates.columns)})")
print(f"FX spot at effective date : " + ", ".join(f"{p}={fx_rates[p].iloc[-1]:.4f}" for p in fx_rates.columns))
print(f"Active ERP engine         : {ACTIVE_EQUITY_ENGINE}")

print()
print("Equity universe and sectors:")
display(sector_table)

print("Latest credit-spread curve at the effective date (bps):")
_curve_now = credit_bucket_panel.loc[:effective_date].tail(1)
display((_curve_now.T * 1e4).rename(columns={_curve_now.index[-1]: "OAS_bps"}))

print("Latest cross-asset state at the effective date:")
display(pd.concat([stock_prices.tail(1).T, market_proxy_prices.tail(1).T, key_rates.tail(1).T], axis=0))


In [ ]:
# Visual inspection of raw state variables
plot_time_series(stock_prices, "Adjusted-close prices — broad US equity prototype universe", "Price")
plot_time_series(market_proxy_prices, f"Adjusted-close price — market proxy ({MARKET_PROXY})", "Price")
plot_time_series(key_rates * 100.0, "Treasury key rates — full key-rate grid (%)", "Yield (%)")
plot_time_series(ff_factor_panel[[c for c in ["MKT_RF", "SMB", "HML", "MOM"] if c in ff_factor_panel.columns]],
                 "Daily observable equity factor panel (log-return approximations)",
                 "Daily factor return")


## 2. Risk-driver design and modular ERP engines

The equity block is now **plug-and-play**. The notebook builds several alternative engines and lets the user choose the active one:

- **latent_statistical**: latent common factor extracted from equity excess returns,
- **proxy_market**: observed market ERP proxy via the broad market index,
- **apt_ff3**: observable factor engine based on MKT-RF, SMB, HML,
- **apt_carhart**: observable factor engine based on MKT-RF, SMB, HML, MOM.

Each engine is paired with a compressed idiosyncratic residual block.

What the engine choice changes is the **dependence structure** (betas, residual factors). The **expected return** is set separately by `DRIFT_MODE` (v22). Under `"sample"` (the default) every engine carries each stock's estimation-window mean, so the four engines differ in risk, not in premium. Under `"factor"` each stock earns β′λ on long-run Fama–French premia with zero alpha, and the engine then determines the premium too.

Alongside equities, two curve blocks share the same construction: the **Treasury key-rate curve** (daily yield changes → PCA into level/slope/curvature factors) and, new in v7, the **credit-spread curve**. The credit block takes the ICE BofA option-adjusted spreads (OAS) across rating buckets (AAA→CCC), differences them, and compresses them by PCA into a few **credit factors** (overall credit-risk level, IG-vs-HY quality slope, curvature). All blocks — equity factors, equity residual PCs, rates PCs, credit PCs — are stacked into a single joint state, so the contemporaneous **equity-credit dependence** (spreads widening as equity falls) is captured by the joint innovation bootstrap rather than by an imposed correlation.


In [ ]:
# Build transformed risk drivers and alternative equity-premium engines
equity_log_returns = compute_equity_log_returns(stock_prices)
yield_changes = compute_yield_changes(key_rates)
market_log_returns = compute_equity_log_returns(market_proxy_prices)
spread_changes = compute_spread_changes(credit_spreads)
fx_log_returns = compute_fx_log_returns(fx_rates)

common_dates = (
    equity_log_returns.index
    .intersection(yield_changes.index)
    .intersection(market_log_returns.index)
    .intersection(ff_factor_panel.index)
    .intersection(spread_changes.index)
    .intersection(fx_log_returns.index)
)
equity_log_returns = equity_log_returns.loc[common_dates]
yield_changes = yield_changes.loc[common_dates]
spread_changes = spread_changes.loc[common_dates]
fx_log_returns = fx_log_returns.loc[common_dates]
key_rates = key_rates.loc[common_dates]
credit_spreads = credit_spreads.loc[common_dates]
fx_rates = fx_rates.loc[common_dates]
market_proxy_prices = market_proxy_prices.loc[common_dates]
ff_factor_panel = ff_factor_panel.loc[common_dates]

# Issue #6: single risk-free convention. The daily Fama-French RF (already downloaded) is the same
# risk-free that MKT_RF embeds, so we use it to define equity excess returns instead of DGS2/252.
rf_daily = ff_factor_panel["RF"]

latent_equity_block = build_latent_statistical_equity_block(
    log_returns=equity_log_returns,
    rf_daily=rf_daily,
    resid_explained_var_target=TARGET_RESID_EXPLAINED_VAR,
)

proxy_equity_block = build_proxy_market_equity_block(
    log_returns=equity_log_returns,
    market_proxy_prices=market_proxy_prices,
    rf_daily=rf_daily,
    resid_explained_var_target=TARGET_RESID_EXPLAINED_VAR,
)

apt_ff3_equity_block = build_apt_equity_block(
    log_returns=equity_log_returns,
    rf_daily=rf_daily,
    ff_factor_panel=ff_factor_panel,
    selected_factors=["MKT_RF", "SMB", "HML"],
    block_name="apt_ff3",
    resid_explained_var_target=TARGET_RESID_EXPLAINED_VAR,
)

apt_carhart_equity_block = build_apt_equity_block(
    log_returns=equity_log_returns,
    rf_daily=rf_daily,
    ff_factor_panel=ff_factor_panel,
    selected_factors=["MKT_RF", "SMB", "HML", "MOM"],
    block_name="apt_carhart",
    resid_explained_var_target=TARGET_RESID_EXPLAINED_VAR,
)

equity_engines = {
    "latent_statistical": latent_equity_block,
    "proxy_market": proxy_equity_block,
    "apt_ff3": apt_ff3_equity_block,
    "apt_carhart": apt_carhart_equity_block,
}

if ACTIVE_EQUITY_ENGINE not in equity_engines:
    raise ValueError(f"Unknown ACTIVE_EQUITY_ENGINE: {ACTIVE_EQUITY_ENGINE}")

equity_block = equity_engines[ACTIVE_EQUITY_ENGINE]

rates_block = build_rates_factor_model(
    yield_changes=yield_changes,
    explained_var_target=TARGET_RATE_EXPLAINED_VAR,
)

# Credit block: PCA of the rating-bucket OAS curve. Its factors join the joint state, so the
# equity-credit (Merton-type) co-movement is carried by the joint innovation bootstrap.
credit_block = build_credit_factor_model(
    spread_changes=spread_changes,
    explained_var_target=TARGET_CREDIT_EXPLAINED_VAR,
    max_factors=CREDIT_MAX_FACTORS,
)

# Sensitivity bridge (moody_2pt only): per-bucket OLS of daily bucket-OAS changes on the 2-point
# spread changes over the overlap window. It converts SIMULATED 2-point moves into bucket space for
# bond repricing -- sensitivities plus today's real anchor, no fabricated bucket history. The
# walk-forward backtest never touches it (its metric is equity VaR on the native 2-point engine).
credit_bridge = None
if CREDIT_BLOCK_SOURCE == "moody_2pt":
    if CREDIT_BRIDGE_FREQUENCY not in ("daily", "horizon"):
        raise ValueError(f"Unknown CREDIT_BRIDGE_FREQUENCY {CREDIT_BRIDGE_FREQUENCY!r}; expected 'daily' or 'horizon'")
    credit_bridge = estimate_credit_bucket_bridge(
        bucket_panel=credit_bucket_panel,
        spread_changes_2pt=spread_changes,
        anchor_date=effective_date,
        horizon_bdays=(HORIZON_BDAYS if CREDIT_BRIDGE_FREQUENCY == "horizon" else 1),
    )

# FX block: PCA of the currency-pair daily log-returns. Its factors join the joint state (after the
# credit PCs), so the equity-FX and rates-FX co-movements are carried by the same joint innovation
# bootstrap. With a single pair this is one factor; the block compresses if more pairs are added.
fx_block = build_fx_factor_block(
    fx_log_returns=fx_log_returns,
    explained_var_target=TARGET_FX_EXPLAINED_VAR,
    max_factors=FX_MAX_FACTORS,
)

# v22 -- credit variance the compression drops. "restore" attaches the daily compression residuals
# (PCA truncation for the bucket panel, or the 2-point -> bucket bridge residual) so the simulator adds
# their h-day sums back at the horizon; "drop" attaches nothing (byte-identical to v21).
if CREDIT_RESIDUAL_MODE == "restore":
    _pool_owner = credit_bridge if credit_bridge is not None else credit_block
    _pool_owner["residual_pool"] = _pool_owner["residuals"]
    _res = _pool_owner["residuals"]
    _full = (credit_bucket_panel.diff() if credit_bridge is not None else spread_changes).reindex(_res.index)
    _bf_h = int(credit_bridge.get("frequency_bdays", 1)) if credit_bridge is not None else 1
    if _bf_h > 1:
        # v24: with the bridge fitted on h-day changes, the DAILY share is the wrong yardstick (the h-day
        # gamma applied to daily moves fits daily noise poorly by design); report it at the fit frequency.
        _lost = (_res.rolling(_bf_h).sum().var() / _full[_res.columns].rolling(_bf_h).sum().var()).dropna()
        _lost_lab = f"{_bf_h}-day"
    else:
        _lost = (_res.var() / _full[_res.columns].var()).dropna()
        _lost_lab = "DAILY"
    print("Credit residual restored at the horizon ("
          + ("2-point -> bucket bridge residual" if credit_bridge is not None
             else f"PCA truncation, {credit_block['n_factors']} of {spread_changes.shape[1]} factors kept")
          + f"). Share of each bucket's {_lost_lab} variance the compression had dropped: "
          + ", ".join(f"{b} {v:.0%}" for b, v in _lost.items()) + ".")
elif CREDIT_RESIDUAL_MODE != "drop":
    raise ValueError(f"Unknown CREDIT_RESIDUAL_MODE {CREDIT_RESIDUAL_MODE!r}; expected 'drop' or 'restore'")

# v22 -- expected returns / changes over the horizon (DRIFT_MODE). "sample" leaves every block's mean
# at its estimation-window average (v21); "factor" prices equities on long-run Fama-French premia
# (alpha = 0) with zero drift elsewhere; "zero" removes every drift.
equity_block, rates_block, credit_block, fx_block, drift_info = apply_drift_mode(
    equity_block, rates_block, credit_block, fx_block, mode=DRIFT_MODE,
    ff_long=ff_long_history, asof=effective_date, ff_window=ff_factor_panel,
)
if DRIFT_MODE != "sample":
    _drift_tab = pd.DataFrame({
        "sample_mean_excess_ann": drift_info["sample_mean_excess"] * 252.0,
        f"{DRIFT_MODE}_drift_excess_ann": drift_info["drift_excess"] * 252.0,
    })
    print(f"\nDRIFT_MODE={DRIFT_MODE!r}: annualized expected excess return per stock, the estimation-window "
          "sample mean (v21) against the drift now simulated (rates, spreads and FX now drift at zero):")
    display(_drift_tab.round(4))
    if drift_info["lambda"] is not None:
        print(f"Long-run premia (annualized), Fama-French daily history {drift_info['history_start'].date()} -> "
              f"{drift_info['history_end'].date()}:")
        display(pd.DataFrame({"lambda_ann": drift_info["lambda"] * 252.0,
                              "source": drift_info["lambda_source"]}).round(4))

# Crisis-tail calibration from the long Moody's Baa-Aaa history (decades incl. every postwar crisis).
# This characterizes how often / how severely credit blows out vs the short calm window; the regime
# overlay in the simulator uses it. Computed regardless of CRISIS_MODE (informative even when off).
crisis_calibration = compute_credit_crisis_calibration(
    moodys=moodys_raw,
    horizon_bdays=HORIZON_BDAYS,
    calm_sigma_threshold=CRISIS_SIGMA_THRESHOLD,
)
# Effective regime parameters: config overrides win, else the Moody's-calibrated values.
crisis_prob_eff = CRISIS_PROB if CRISIS_PROB is not None else crisis_calibration["crisis_prob"]
crisis_scale_eff = CRISIS_SCALE if CRISIS_SCALE is not None else crisis_calibration["crisis_scale"]
crisis_pool_eff = crisis_calibration["stress_magnitude_pool"]
print(
    f"Crisis backbone (Moody's Baa-Aaa, {crisis_calibration['history_start'].date()} -> "
    f"{crisis_calibration['history_end'].date()}, {crisis_calibration['n_total_days']} days): "
    f"{crisis_calibration['n_stress_days']} stress days -> crisis_prob={crisis_prob_eff:.4f}/day, "
    f"crisis_scale={crisis_scale_eff:.2f}x, tail dof={crisis_calibration['dof']:.1f}. "
    f"Active CRISIS_MODE={CRISIS_MODE!r}."
)

curve_lsc_levels = build_curve_lsc_factors(key_rates)
curve_lsc_changes = curve_lsc_levels.diff().dropna()

credit_lsc_levels = build_credit_lsc_factors(credit_spreads)
credit_lsc_changes = credit_lsc_levels.diff().dropna()

engine_summary = pd.concat([
    summarize_equity_engine(block) for block in equity_engines.values()
], axis=1).T.sort_values(["mean_asset_R2", "median_asset_R2"], ascending=False)

print(f"Active equity factor block: {equity_block['engine_name']}")
print(f"Rates factors retained   : {rates_block['n_factors']} (from {key_rates.shape[1]} tenors)")
print(f"Credit factors retained  : {credit_block['n_factors']} (from {credit_spreads.shape[1]} "
      + ("Moody's 2-point spread series)" if CREDIT_BLOCK_SOURCE == "moody_2pt" else "rating buckets)"))
print(f"FX factors retained      : {fx_block['n_factors']} (from {fx_rates.shape[1]} currency pair(s))")
print(f"Daily risk-free source   : Fama-French RF (mean {rf_daily.mean():.6f}/day)")
print("Equity engine comparison:")
display(engine_summary)

print("Credit-spread PCA explained variance:")
display(credit_block["explained_ratio"].to_frame("credit_explained_variance_ratio"))

# Economic interpretation of the credit PCs via level / quality-slope / curvature changes
credit_interpretation = pd.concat([credit_block["scores"], credit_lsc_changes], axis=1).dropna().corr()
credit_interpretation = credit_interpretation.loc[credit_block["scores"].columns, credit_lsc_changes.columns]
print("Interpretation of credit PCA scores (corr with level / quality-slope / curvature changes):")
display(credit_interpretation)

if credit_bridge is not None:
    _bfreq = int(credit_bridge.get("frequency_bdays", 1))
    print(f"\nBucket sensitivity bridge (interceptless OLS on {credit_bridge['n_overlap']} overlap days; "
          + ("gamma maps daily 2-point spread changes to bucket-OAS changes; used for repricing only):" if _bfreq == 1
             else f"gamma fitted on overlapping {_bfreq}-day 2-point and bucket-OAS changes "
                  "(CREDIT_BRIDGE_FREQUENCY='horizon'), r2 at that frequency; used for repricing only):"))
    _btab = credit_bridge["gamma"].copy()
    _btab["r2"] = credit_bridge["r2"]
    _btab["anchor_bps"] = credit_bridge["current_buckets"] * 1e4
    display(_btab.round({"Aaa_spr": 2, "Baa_spr": 2, "r2": 2, "anchor_bps": 0}))
    print("Read: a +1bp move of the simulated Baa spread widens each bucket by ~gamma_Baa bps around "
          "its real anchor. Declared limitation: a linear calm-window map understates crisis "
          "decompression on the HY end (CCC convexity) -- a repricing-granularity caveat; the credit "
          "DYNAMICS themselves are the real long 2-point history.")


In [ ]:

# Diagnostics on engine structure and factor interpretation
active_beta_table = equity_block["betas"].copy()
active_beta_table.index.name = "asset"

rate_interpretation = pd.concat([rates_block["scores"], curve_lsc_changes], axis=1).dropna().corr()
rate_interpretation = rate_interpretation.loc[rates_block["scores"].columns, curve_lsc_changes.columns]

compare_factor_table = pd.concat([
    proxy_equity_block["factors"].rename(columns={proxy_equity_block["factors"].columns[0]: "market_erp_proxy"}),
    latent_equity_block["factors"].rename(columns={latent_equity_block["factors"].columns[0]: "latent_equity_factor"}),
    ff_factor_panel[[c for c in ["MKT_RF", "MOM"] if c in ff_factor_panel.columns]],
], axis=1).dropna().corr()

print("Active engine beta matrix:")
display(active_beta_table)

print("Active engine asset-level R^2:")
display(equity_block["asset_r2"].to_frame("asset_R2"))

print("Residual equity PCA explained variance (active engine):")
display(equity_block["resid_explained_ratio"].to_frame("resid_explained_variance_ratio"))
display(equity_block["resid_cum_explained_ratio"].to_frame("resid_cum_explained_ratio"))

print("Comparison across observable and latent equity factors:")
display(compare_factor_table)

print("Economic interpretation of rates PCA scores via level / slope / curvature changes:")
display(rate_interpretation)

plot_time_series(
    pd.concat([equity_block["factors"], rates_block["scores"]], axis=1),
    f"Active ERP engine factors and rates PCA scores — {ACTIVE_EQUITY_ENGINE}",
    "Factor value",
)



## 3. Joint state representation and hybrid scenario engine

The active joint state vector stacks:

- the chosen **ERP engine factors**,
- compressed **equity residual factors**,
- compressed **risk-free curve factors**.

Dynamics are modelled through a **VAR(1)**, while innovations are generated with **stationary bootstrap** to avoid relying purely on Gaussian i.i.d. shocks.

**v24 — a cross-asset factor layer on top of the blocks.** The blocks above are compressed separately and stacked, so on their own they contain no cross-asset factor: the dependence between equities, rates, credit and FX lives only in the co-dated rows the bootstrap resamples. The cell below also estimates a **hierarchical** structure on the same drivers, `d = Λg + u`:

- **Global factors `g`.** These are the linear combinations of the drivers whose removal leaves the least cross-block covariance.
- **Block-specific parts `u`.** These are what each block keeps for itself.
- **Pre-test, printed in every mode.** It reports the canonical correlations between blocks, how many global factors it takes before no cross-block pair keeps material correlation, and the global, block and idiosyncratic share of every observed series. It also shows the transmission of each global factor before and after 2022.

With `FACTOR_STRUCTURE="hierarchical"` the simulator resamples `[g, u]` instead:

- the global factors and each block's local part get their own date streams;
- each global factor carries one variance state;
- with `GLOBAL_STRUCTURE_HALFLIFE_YEARS`, the loadings are those of the current regime.


In [ ]:
# Joint state vector and dynamic model
drivers = build_joint_driver_matrix(equity_block, rates_block, credit_block, fx_block)
print(f"Joint driver matrix shape: {drivers.shape}")
print("Driver columns:")
print(drivers.columns.tolist())
display(drivers.head())

var_res = build_var1_model(drivers)
print("VAR(1) fitted successfully.")
print(f"Selected lag order: {var_res.k_ar}")
print(f"Number of state variables: {var_res.neqs}")
display(var_res.params)

try:
    print(var_res.summary())
except Exception as err:
    print("VAR summary could not be rendered cleanly.")
    print(f"Reason: {type(err).__name__}: {err}")

# Issue #9: stability check (statsmodels convention: stable iff ALL companion roots have modulus > 1)
print("Companion roots (modulus should be > 1 for stability):")
print(np.abs(var_res.roots))
print(f"VAR(1) is_stable(): {var_res.is_stable()}")
print(f"Min |root|: {np.min(np.abs(var_res.roots)):.4f}")

# Issue #9: is the VAR(1) doing real work, or is risk coming from the bootstrapped innovations?
# Per-equation one-step in-sample R^2 vs a constant-only baseline.
one_step_r2 = var_one_step_r2(var_res, drivers)
print()
print("VAR(1) one-step-ahead in-sample R^2 per driver:")
display(one_step_r2.to_frame("one_step_R2"))
print(
    f"Mean one-step R^2 across drivers: {one_step_r2.mean():.4f}. "
    "If these are near zero, the VAR mean dynamics add little predictive content and the simulated "
    "risk is effectively driven by the bootstrapped joint innovations + factor loadings; the model "
    "could then be simplified (block-bootstrap the demeaned innovations) or regularized."
)

# Option A: precompute the block-bootstrap innovation source (demeaned joint drivers + their mean),
# used by the simulator when DRIVER_DYNAMICS == "block_bootstrap". This drops the VAR(1) mean
# dynamics and resamples the demeaned drivers (equity factors + resid PCs + rate PCs + credit PCs +
# FX PCs) in stationary blocks around their mean.
driver_mean = drivers.mean()
demeaned_drivers = drivers - driver_mean

_k = drivers.shape[1]
_var_params = _k + _k * _k          # k intercepts + k*k AR coefficients
print()
print("Joint-state dynamics (Option A):")
_dyn_compare = pd.Series({
    "active_DRIVER_DYNAMICS": DRIVER_DYNAMICS,
    "state_dimension_k": _k,
    "var1_dynamics_parameters": _var_params,
    "block_bootstrap_dynamics_parameters": 0,
    "mean_one_step_R2": round(float(one_step_r2.mean()), 4),
})
display(_dyn_compare.to_frame("value"))
print(
    f"Active mode: {DRIVER_DYNAMICS}. 'block_bootstrap' resamples the demeaned joint innovations in "
    f"stationary blocks and removes the {_var_params} VAR(1) dynamics parameters; with a mean one-step "
    f"R^2 of {one_step_r2.mean():.4f} the VAR's mean dynamics contribute little, so the two modes give "
    "near-identical one-week distributions. Set DRIVER_DYNAMICS='var1' in the config to recover the "
    "VAR(1) engine and compare the tail metrics."
)

# t-copula (v11): fit a Student-t copula to the joint innovations. Fitted regardless of the active
# mode so the tail-dependence diagnostic is always available; only used by the simulator when
# DRIVER_DYNAMICS == "t_copula". The dof nu controls tail dependence (low nu -> joint extremes
# co-occur; nu -> inf is the Gaussian / linear-correlation case the bootstrap implicitly assumes).
t_copula_model = fit_t_copula(drivers)
print()
print("Joint-innovation tail dependence (Student-t copula, v11):")
_cop = pd.Series({
    "fitted_dof_nu": round(t_copula_model["nu"], 2),
    "avg_abs_pairwise_corr": round(t_copula_model["avg_abs_corr"], 4),
    "avg_pairwise_tail_dependence": round(t_copula_model["avg_tail_dependence"], 4),
    "copula_loglik": round(t_copula_model["loglik"], 1),
    "gaussian_copula_loglik": round(t_copula_model["gaussian_loglik"], 1),
})
display(_cop.to_frame("value"))
print(
    f"Fitted dof nu={t_copula_model['nu']:.1f} (lower = heavier joint tails). The t-copula gains "
    f"{t_copula_model['loglik'] - t_copula_model['gaussian_loglik']:+.0f} log-likelihood over a "
    f"Gaussian copula (nu->inf), and implies an average pairwise tail-dependence coefficient of "
    f"{t_copula_model['avg_tail_dependence']:.3f} vs 0 for the Gaussian case -- i.e. drivers are "
    "jointly extreme more often than the block bootstrap's resampling of observed vectors implies. "
    "Set DRIVER_DYNAMICS='t_copula' in the config to simulate from it (empirical margins, t dependence)."
)

# Asset-level copula (v12): fit the t-copula on the OBSERVABLES (equity log-returns + key-rate changes
# + spread changes + FX log-return) instead of the orthogonal PCA factors. The factor copula above
# had almost no correlation to work with (the PCs are orthogonal by construction), so its tail
# dependence was negligible; the observables carry the real cross-sectional correlation (the curve
# moves together, the credit buckets move together, equities co-move), so this is where tail
# dependence can actually bite. Used by the simulator only when DRIVER_DYNAMICS == "asset_copula".
asset_copula_model = fit_asset_copula(equity_log_returns, yield_changes, spread_changes, fx_log_returns)
print()
print("Factor-level vs asset-level copula (v12):")
_cop_compare = pd.DataFrame({
    "factor_copula": {
        "dimension": t_copula_model["dim"], "fitted_dof_nu": round(t_copula_model["nu"], 1),
        "avg_abs_corr": round(t_copula_model["avg_abs_corr"], 3),
        "avg_tail_dependence": round(t_copula_model["avg_tail_dependence"], 3),
    },
    "asset_copula": {
        "dimension": asset_copula_model["dim"], "fitted_dof_nu": round(asset_copula_model["nu"], 1),
        "avg_abs_corr": round(asset_copula_model["avg_abs_corr"], 3),
        "avg_tail_dependence": round(asset_copula_model["avg_tail_dependence"], 3),
    },
}).T
display(_cop_compare)
print(
    f"The asset-level copula sees average |correlation| {asset_copula_model['avg_abs_corr']:.3f} "
    f"(vs {t_copula_model['avg_abs_corr']:.3f} for the orthogonal factors) and an implied average "
    f"tail dependence of {asset_copula_model['avg_tail_dependence']:.3f} (vs "
    f"{t_copula_model['avg_tail_dependence']:.3f}). This is the reduced-form generator (no factor "
    "structure); set DRIVER_DYNAMICS='asset_copula' to simulate from it and compare the equity "
    "co-exceedance in the validation cell. The validation cell's tail-dependence panel pits the "
    "historical, block-bootstrap, factor-copula and asset-copula joints against each other."
)

# Conditional-volatility (FHS) filter: standardize each factor by its EWMA/GARCH conditional vol so
# the simulator can re-scale by today's vol. Fitted whenever CONDITIONAL_VOL == "ewma_fhs"; the
# current-vol ratio reads off whether the market is calm or stressed RIGHT NOW relative to its history.
if CONDITIONAL_VOL == "ewma_fhs":
    fhs_filter = compute_ewma_fhs_filter(
        drivers.to_numpy(), driver_mean.to_numpy(),
        ewma_lambda=VOL_EWMA_LAMBDA, reversion_halflife=VOL_REVERSION_HALFLIFE_DAYS,
        params_mode=VOL_PARAMS_MODE,
    )
    if fhs_filter.get("per_factor") is not None:
        _pf = pd.DataFrame(fhs_filter["per_factor"], index=drivers.columns)
        _pf["uncond_var_share"] = fhs_filter["uncond_var"] / fhs_filter["uncond_var"].sum()
        # Clustering flag reflects the model ACTUALLY used: a factor conditions on its own vol only
        # if its fitted persistence > 0. A factor gated to constant variance (news weight below the
        # impact gate) keeps persistence 0 and runs flat, even if its in-sample LR is large -- so the
        # flag must key off persistence, not off lr_vs_const, or the count over-states clustering.
        _pf["clustering"] = _pf["persistence"] > 0.0
        _pf = _pf.sort_values("uncond_var_share", ascending=False)
        print()
        print("Per-factor GARCH(1,1) estimates (QML with variance targeting; LR = likelihood-ratio vs "
              "constant variance, indicative threshold ~6; no-evidence factors fall back to constant "
              "variance, i.e. the filter conditions only where the data supports it):")
        _pf_round = {c: d for c, d in {"alpha": 3, "gamma": 3, "persistence": 4,
                                       "half_life_days": 0, "lr_vs_const": 1,
                                       "lr_asym": 1, "uncond_var_share": 3, "neg_share": 3,
                                       "v22_longrun_var_inflation": 2}.items()
                     if c in _pf.columns}
        display(_pf.head(12).round(_pf_round))
        if "gamma" in _pf.columns:
            _nl = int((_pf["gamma"] > 0).sum())
            print(f"{_nl} of {len(_pf)} factors show a leverage effect (gamma > 0, LR_asym >= 3.84): "
                  "for these, negative shocks raise next-day variance by alpha + gamma, positive "
                  "shocks by alpha alone. Factors without asymmetry evidence keep the symmetric fit.")
            _gj = _pf[_pf["gamma"] > 0] if "neg_share" in _pf.columns else _pf.iloc[0:0]
            if len(_gj):
                print(f"Variance targeting (v23): on the {len(_gj)} leverage factors the down-day share of the "
                      f"squared standardized residuals is {_gj['neg_share'].min():.3f}-{_gj['neg_share'].max():.3f} "
                      "(1/2 would be symmetric). Up to v22 the targeting assumed 1/2; at these persistences that "
                      f"put the simulated long-run variance up to {_gj['v22_longrun_var_inflation'].max():.1f}x its "
                      f"target ({_gj['v22_longrun_var_inflation'].idxmax()}) -- the drift that made long-horizon risk "
                      "grow faster than sqrt(t) in the v22 run. The targeting now uses the measured share, so the "
                      "simulated variance reverts to the unconditional level.")
        _nc = int(_pf["clustering"].sum())
        _ngate = int(((_pf["persistence"] <= 0.0) & (_pf["lr_vs_const"] >= 6.0)).sum())
        print(f"{_nc} of {len(_pf)} factors show volatility clustering (fitted persistence > 0, so the "
              "filter conditions on them); the rest run at constant variance. This is the per-factor "
              "heterogeneity that one fixed half-life (and, before it, one block length) could not "
              "represent -- each clustered factor now decays at its own estimated speed."
              + (f" {_ngate} factor(s) carry a large in-sample LR yet were gated to constant variance: "
                 "their estimated news weight alpha + gamma/2 sits below the impact gate -- "
                 "statistically detectable but too small to be worth modelling. The LR column keeps "
                 "that evidence visible while the clustering flag reports the model actually run."
                 if _ngate else ""))
    _vr = pd.Series(fhs_filter["current_vol_ratio"], index=drivers.columns)
    _eq_factor_cols = [c for c in drivers.columns if str(c).startswith(("MKT", "equity", "Mkt", "eq_"))] or list(drivers.columns[:1])
    print()
    print("Conditional volatility (FHS) active. Current conditional vol vs unconditional, per factor "
          f"(>1 = market currently MORE volatile than its history; news weight {1-VOL_EWMA_LAMBDA:.2f}, "
          f"vol half-life {VOL_REVERSION_HALFLIFE_DAYS:.0f} business days):")
    display(_vr.to_frame("current_vol_ratio").round(3))
    print(
        f"The equity market factor is currently at {_vr.iloc[0]:.2f}x its unconditional volatility, so "
        f"the {HORIZON_LABEL} risk will be scaled {'UP' if _vr.iloc[0] > 1 else 'DOWN'} relative to the "
        "unconditional bootstrap. Over the one-year horizon this conditioning mean-reverts away (the "
        "variance reverts to its unconditional level over the half-life), so the 1Y risk is close to "
        "unconditional. Asset correlations are not fitted: they rise in stress automatically because "
        "the shared factor's vol scales every loading on it together."
    )
else:
    fhs_filter = None

# Politis-White automatic block-length selection. The diagnostic makes the block/FHS interaction
# explicit: on the LEVELS the optimal block is short (returns are nearly serially uncorrelated), but
# on the ABSOLUTE series it is long for the raw innovations (that is the volatility clustering the
# block must otherwise carry) and short again for the FHS standardized residuals (the GARCH filter has
# already absorbed the clustering). So the right data-driven block is short *when FHS is on* and long
# when it is off -- two routes to capturing the same dependence.
def _pw_median(mat):
    return float(np.median(politis_white_block_length(np.asarray(mat))))

_dd = demeaned_drivers.to_numpy()
_pw = {"raw innovations (levels)": _pw_median(_dd),
       "raw innovations |.| (vol clustering)": _pw_median(np.abs(_dd))}
if CONDITIONAL_VOL == "ewma_fhs" and fhs_filter is not None:
    _z = fhs_filter["standardized_residuals"]
    _pw["FHS standardized residuals (levels)"] = _pw_median(_z)
    _pw["FHS standardized |.| (vol clustering)"] = _pw_median(np.abs(_z))

if BOOTSTRAP_BLOCK_MODE == "politis_white":
    if CONDITIONAL_VOL == "ewma_fhs" and fhs_filter is not None:
        effective_block_length = max(1, int(round(_pw["FHS standardized residuals (levels)"])))
        _blk_src = "Politis-White on FHS standardized residuals"
    else:
        effective_block_length = max(1, int(round(_pw["raw innovations |.| (vol clustering)"])))
        _blk_src = "Politis-White on |raw innovations| (captures vol clustering; no FHS active)"
else:
    effective_block_length = BOOTSTRAP_EXPECTED_BLOCK
    _blk_src = f"fixed BOOTSTRAP_EXPECTED_BLOCK={BOOTSTRAP_EXPECTED_BLOCK}"

print()
print("Politis-White optimal expected block length (median across factors):")
display(pd.Series(_pw, name="optimal_block_bdays").round(2).to_frame())
print(
    f"Block length used by the bootstrap: {effective_block_length} business days ({_blk_src}). "
    "On the levels the optimum is short because returns are nearly serially uncorrelated; the long "
    "block on the |raw innovations| is the volatility clustering, which the FHS filter absorbs -- "
    "hence the FHS standardized residuals admit a short block. Set BOOTSTRAP_BLOCK_MODE='politis_white' "
    "to use the data-driven value; 'fixed' keeps the hard-coded block."
)

# ---- Level mean reversion (OU / Vasicek) on rate and spread levels --------
# Estimated on the same sample the bootstrap resamples from. Where a series is
# gated in, the simulator replaces the bootstrapped random-walk drift with an
# explicit daily pull kappa * (theta - level); non-gated series keep the
# random walk exactly, so reversion is applied only where the data supports it.
ou_rates = None
ou_credit = None
if LEVEL_MEAN_REVERSION == "ou":
    ou_rates = estimate_ou_level_params(key_rates.to_numpy(), tstat_gate=OU_TSTAT_GATE)
    if credit_spreads is not None and credit_spreads.shape[1] > 0:
        ou_credit = estimate_ou_level_params(credit_spreads.to_numpy(), tstat_gate=OU_TSTAT_GATE)
    print()
    print("Ornstein-Uhlenbeck level mean reversion (gate: OLS slope < 0 AND t-stat <= "
          f"{OU_TSTAT_GATE:.2f}, the Dickey-Fuller 5% critical value -- under the unit-root "
          "null the OLS t-stat is not t-distributed, so an ordinary -2 gate would find "
          "spurious reversion on short samples). Non-gated series keep the random walk:")
    _ou_tabs = [("rates", ou_rates, key_rates.columns)]
    if ou_credit is not None:
        _ou_tabs.append(("credit spreads", ou_credit, credit_spreads.columns))
    for _nm, _ou, _cols in _ou_tabs:
        _t = pd.DataFrame(_ou, index=_cols)[["kappa", "theta", "half_life_days", "tstat", "gated"]]
        _ng = int(_t["gated"].sum())
        print(f"\n{_nm}: {_ng} of {len(_t)} series gated into mean reversion "
              "(theta in the same decimal units as the levels, e.g. 0.043 = 4.3%; half-life in "
              "business days):")
        display(_t.round({"kappa": 4, "theta": 3, "half_life_days": 0, "tstat": 2, "gated": 0}))

# ---- v24: cross-asset factor structure ---------------------------------------------------------------
# A pre-test printed in every mode (cheap, no random numbers): do the blocks share common factors, how
# many does it take before no cross-block pair keeps material correlation, how is each observed series
# split between global, block and idiosyncratic risk, and does the transmission of the global factors
# change across regimes? With FACTOR_STRUCTURE="hierarchical" the same structure drives the simulation.
_hier_labels = driver_block_labels(drivers.columns)
_hier_hl = None if GLOBAL_STRUCTURE_HALFLIFE_YEARS is None else 252.0 * float(GLOBAL_STRUCTURE_HALFLIFE_YEARS)
hier_structure = build_hierarchical_structure(
    drivers, _hier_labels, n_global=GLOBAL_FACTORS, max_global=GLOBAL_FACTORS_MAX,
    resid_corr_tol=GLOBAL_RESID_CORR_TOL, halflife_days=_hier_hl, local_resampling=HIER_LOCAL_RESAMPLING,
)
print()
print("Cross-asset factor structure (v24 pre-test). Leading canonical correlations between the driver blocks "
      "(near 0: nothing shared; near 1: a shared factor; sampling floor about sqrt((p_a + p_b) / T)):")
display(block_canonical_correlations(drivers, _hier_labels).round(3))
_kt = hier_structure["k_table"].copy()
_hier_wlab = ("equal weights" if _hier_hl is None else
              f"weights with a {GLOBAL_STRUCTURE_HALFLIFE_YEARS:g}-year half-life, effective sample {hier_structure['ess']:.0f} days")
print(f"Global factors k_G and what they leave behind: share of the cross-block covariance explained (block-normalized), "
      f"and the largest / RMS cross-block correlation the block-specific parts u still carry ({_hier_wlab}). "
      f"Selected k_G = {hier_structure['k_global']} "
      + (f"(GLOBAL_FACTORS='auto': smallest k_G <= {GLOBAL_FACTORS_MAX} with no pair above {hier_structure['resid_corr_tol']:.3f})."
         if GLOBAL_FACTORS == "auto" else f"(GLOBAL_FACTORS={GLOBAL_FACTORS})."))
display(_kt.round(4))
if GLOBAL_FACTORS == "auto" and float(_kt.loc[hier_structure["k_global"], "resid_max_abs_corr"]) > hier_structure["resid_corr_tol"]:
    print(f"NOTE: even {hier_structure['k_global']} global factors leave a cross-block pair with "
          f"{float(_kt.loc[hier_structure['k_global'], 'resid_max_abs_corr']):.3f} unexplained correlation "
          f"({_kt.loc[hier_structure['k_global'], 'resid_max_pair']}): with "
          "HIER_LOCAL_RESAMPLING='independent' that dependence is dropped (raise GLOBAL_FACTORS_MAX, or use 'joint').")
_rep_cr = REPRESENTATIVE_RATING if REPRESENTATIVE_RATING in spread_changes.columns else spread_changes.columns[-1]
_hier_obs = pd.DataFrame({
    "EW equity": equity_log_returns[TICKERS].mean(axis=1),
    "DGS10 change": yield_changes["DGS10"],
    f"{_rep_cr} change": spread_changes[_rep_cr],
    f"{fx_log_returns.columns[0]} return": fx_log_returns.iloc[:, 0],
})
global_transmission = global_transmission_table(hier_structure, _hier_obs, BACKTEST_REGIME_SPLIT)
print("Transmission: daily correlation of each global factor with the observables, whole window and before / from "
      f"{pd.Timestamp(BACKTEST_REGIME_SPLIT).date()} (the factors are defined once, so a sign change is a change of "
      "loadings, i.e. of transmission):")
display(global_transmission.round(3))
_eq_excess = equity_block["excess_returns"] if "excess_returns" in equity_block else equity_log_returns
global_variance_split = hierarchical_variance_decomposition(hier_structure, {
    "equity": _eq_excess[TICKERS], "rates": yield_changes, "credit": spread_changes, "fx": fx_log_returns})
print("Variance of each observed daily series split into global / own-block common / idiosyncratic shares "
      "(sequential R^2; the 'global' column is the analogue of a one-factor asset correlation rho):")
display(global_variance_split.round(3))
_gv = global_variance_split.groupby(level="block")["global"].mean()
print("Average global share by block: " + ", ".join(f"{b} {v:.0%}" for b, v in _gv.items()) + ".")

hier_sim_spec = None
hier_fhs_filter = None
hier_demeaned_drivers = None
if FACTOR_STRUCTURE == "hierarchical":
    _X = hier_structure["X"]
    hier_demeaned_drivers = _X - _X.mean()
    if CONDITIONAL_VOL == "ewma_fhs":
        hier_fhs_filter = compute_fhs_filter_for_matrix(
            _X.to_numpy(), ewma_lambda=VOL_EWMA_LAMBDA, reversion_halflife=VOL_REVERSION_HALFLIFE_DAYS,
            params_mode=VOL_PARAMS_MODE)
    hier_sim_spec = hierarchical_sim_spec(hier_structure)
    print()
    print(f"FACTOR_STRUCTURE='hierarchical' ACTIVE: the simulator resamples [g, u] ({_X.shape[1]} columns: "
          f"{hier_structure['k_global']} global + {drivers.shape[1]} block-specific) in "
          f"{len(hier_sim_spec['groups'])} date stream(s) ({', '.join(hier_sim_spec['group_labels'])}) and rebuilds the "
          f"drivers as d = mean + Lambda g + u"
          + ("" if _hier_hl is None else f", Lambda estimated on the current regime ({GLOBAL_STRUCTURE_HALFLIFE_YEARS:g}-year half-life)")
          + ".")
    if hier_fhs_filter is not None:
        _gvr = pd.Series(np.asarray(hier_fhs_filter["current_vol_ratio"])[:hier_structure["k_global"]],
                         index=hier_structure["g"].columns)
        print("Global factors' current conditional vol vs unconditional (one variance state each): "
              + ", ".join(f"{j} {v:.2f}" for j, v in _gvr.items()) + ".")
active_demeaned_drivers = hier_demeaned_drivers if FACTOR_STRUCTURE == "hierarchical" else demeaned_drivers
active_fhs_filter = hier_fhs_filter if FACTOR_STRUCTURE == "hierarchical" else fhs_filter

# ---- v25 challenger pre-test -----------------------------------------------------------------------
v25_challenger_specs = {}
if V25_BUILD_CHALLENGERS:
    if fhs_filter is None:
        print("v25 challengers skipped: they are defined on original-driver GJR standardized residuals, so set CONDITIONAL_VOL='ewma_fhs'.")
    else:
        for _nm25 in V25_CHALLENGER_NAMES:
            try:
                _sp25 = build_v25_challenger_structure(
                    drivers, fhs_filter, kind=_nm25, labels=_hier_labels,
                    n_global=GLOBAL_FACTORS, max_global=GLOBAL_FACTORS_MAX,
                    resid_corr_tol=GLOBAL_RESID_CORR_TOL, halflife_days=_hier_hl,
                    local_resampling=HIER_LOCAL_RESAMPLING,
                    kalman_refinements=V25_KALMAN_REFINEMENTS, ar_cap=V25_KALMAN_AR_CAP,
                    r_floor=V25_KALMAN_R_FLOOR, pit_clip=V25_PIT_CLIP)
                v25_challenger_specs[_nm25] = _sp25
            except Exception as _err25:
                print(f"[{_nm25} unavailable] {type(_err25).__name__}: {_err25}")
        if v25_challenger_specs:
            _tab25 = pd.DataFrame({k: v25_challenger_diagnostics(v) for k, v in v25_challenger_specs.items()}).T
            print()
            print("v25c standardized-hierarchy pre-test on the SAME GJR-standardized original drivers.  Residual "
                  "cross-block dependence uses the SAME denominator convention as v24.  The raw reconstructed-z "
                  "std range is printed before the unit-variance guard applied to the original-driver GJR recursion:")
            display(_tab25.round(4))


In [ ]:
# Simulate joint scenarios over the configured horizon (HORIZON_MODE) under the active ERP engine
current_prices = stock_prices.loc[effective_date]
current_key_rates = key_rates.loc[effective_date]
current_spreads = credit_spreads.loc[effective_date]
# The anchor in BUCKET space for downstream cells: with the bridge active the simulator returns
# bucket scenarios around the real bucket anchor, so "scenario minus current" must use that anchor.
bucket_anchor_spreads = credit_bridge["current_buckets"] if credit_bridge is not None else current_spreads
current_fx = fx_rates.loc[effective_date]

# Issue #6: constant daily risk-free for the excess->total reconstruction, taken as the latest
# Fama-French RF (same convention used to define the excess returns). Issue #11: re-inject the
# idiosyncratic residual variance dropped by the PCA compression as iid noise. Credit block: the
# rating-bucket OAS curve is simulated jointly and returned as credit_spread_scenarios.
current_daily_rf = float(ff_factor_panel["RF"].iloc[-1])

# The keyword set is kept as `main_sim_kwargs` so diagnostics further down (the v21 risk term
# structure) re-simulate EXACTLY this engine, changing only the horizon, size and seed. v24: with
# FACTOR_STRUCTURE="hierarchical" the resampled matrix, its FHS filter and the hierarchical map are the
# active ones (active_* are the blockwise objects otherwise, so the blockwise engine is unchanged).
main_sim_kwargs = dict(
    var_res=var_res,
    resid_hist=var_res.resid,
    last_state=drivers.iloc[-1].to_numpy(),
    equity_block=equity_block,
    rates_block=rates_block,
    current_prices=current_prices,
    current_key_rates=current_key_rates,
    horizon_bdays=HORIZON_BDAYS,
    n_scenarios=N_SCENARIOS,
    avg_block_length=effective_block_length,
    rng_seed=RNG_SEED,
    daily_rf=current_daily_rf,
    resid_idio_std=equity_block.get("resid_idiosyncratic_std"),
    credit_block=credit_block,
    current_spreads=current_spreads,
    fx_block=fx_block,
    current_fx=current_fx,
    dynamics=DRIVER_DYNAMICS,
    demeaned_drivers=active_demeaned_drivers,
    driver_mean=driver_mean,
    crisis_mode=CRISIS_MODE,
    crisis_prob=crisis_prob_eff,
    crisis_scale=crisis_scale_eff,
    crisis_pool=crisis_pool_eff,
    copula=t_copula_model,
    asset_copula=asset_copula_model,
    fhs_filter=active_fhs_filter,
    ou_rates=ou_rates, ou_credit=ou_credit,
    credit_bridge=credit_bridge,
    hier=hier_sim_spec,
)
sim = simulate_joint_horizon(**main_sim_kwargs)

stock_excess_return_scenarios = sim["stock_excess_return_scenarios"]
stock_log_return_scenarios = sim["stock_log_return_scenarios"]
stock_price_scenarios = sim["stock_price_scenarios"]
key_rate_scenarios = sim["key_rate_scenarios"]
credit_spread_scenarios = sim["credit_spread_scenarios"]
fx_spot_scenarios = sim["fx_spot_scenarios"]


# ---- v25: same-date paired challenger comparison ----------------------------------------------------
# This is a diagnostic panel only: the headline `sim` above remains the configured v24 engine.  Each
# challenger uses the same seed / scenario count and the same repricer; only the dependence state changes.
if v25_challenger_specs and DRIVER_DYNAMICS == "block_bootstrap":
    _v25_rows = []
    _v25_current_sims = {}
    for _nm25, _sp25 in v25_challenger_specs.items():
        try:
            _kw25 = dict(main_sim_kwargs)
            _kw25.update({"demeaned_drivers": _sp25["X"], "fhs_filter": None, "hier": None,
                          "v25_structure": _sp25, "rng_seed": RNG_SEED})
            _s25 = simulate_joint_horizon(**_kw25)
            _v25_current_sims[_nm25] = _s25
            _eq25 = _s25["stock_log_return_scenarios"][TICKERS].mean(axis=1).to_numpy(dtype=float)
            _d1025 = _s25["key_rate_scenarios"]["DGS10"].to_numpy(dtype=float) - float(current_key_rates["DGS10"])
            _crcol25 = (REPRESENTATIVE_RATING if REPRESENTATIVE_RATING in _s25["credit_spread_scenarios"].columns
                        else _s25["credit_spread_scenarios"].columns[-1])
            _dc25 = (_s25["credit_spread_scenarios"][_crcol25].to_numpy(dtype=float)
                     - float(bucket_anchor_spreads[_crcol25]))
            _v25_rows.append({"structure": _nm25, "k_global": _sp25["k_global"],
                              "eq_VaR95": float(-np.quantile(_eq25, 0.05)),
                              "eq_ES95": sample_expected_shortfall(_eq25, 0.05),
                              "corr_eq_DGS10": float(np.corrcoef(_eq25, _d1025)[0, 1]),
                              "corr_eq_credit": float(np.corrcoef(_eq25, _dc25)[0, 1]),
                              **_sp25["resid_metrics"]})
        except Exception as _err25:
            print(f"[current-date {_nm25} comparison skipped] {type(_err25).__name__}: {_err25}")
    if _v25_rows:
        print("\nv25 same-date challenger comparison (same seed, same physical support size, same repricer):")
        display(pd.DataFrame(_v25_rows).set_index("structure").round(4))


In [ ]:

# Marginal forecast views
plot_histogram(
    stock_excess_return_scenarios[REPRESENTATIVE_EQUITY],
    f"Forecast distribution — {REPRESENTATIVE_EQUITY} {HORIZON_LABEL} excess log-return",
    f"{HORIZON_LABEL.capitalize()} excess log-return",
)

plot_histogram(
    key_rate_scenarios["DGS10"] * 100.0,
    "Forecast distribution — 10Y Treasury yield at horizon",
    "Yield (%)",
)

display(loss_summary_from_pnl(stock_excess_return_scenarios[REPRESENTATIVE_EQUITY]).to_frame(f"{REPRESENTATIVE_EQUITY}_1w_excess_return"))
display(distribution_summary(key_rate_scenarios["DGS10"]).to_frame("DGS10_horizon_yield"))


In [ ]:
# Distributional and dependence validation (issues #1a, #1c; plus equity-credit dependence)
# A joint scenario engine must be judged on its dependence and tail calibration, not only on a
# single-name standard deviation. We compare simulated horizon behaviour with historical rolling windows
# of the same length (5 business days at 1W, 21 at 1M, ...)
# behaviour along seven dimensions (risk scaling, kurtosis, cross-equity correlation, equity-rates
# correlation, equity-credit correlation, credit-spread scaling, and FX risk-scaling / equity-FX
# correlation); the VaR coverage / PIT calibration is handled separately by the walk-forward
# backtest below.

# Column / label stems follow the horizon (identical to v20 at 1W: 'sim_1w', 'hist_5d', '5-day').
_SIM, _HIST, _HD = f"sim_{HORIZON_MODE.lower()}", f"hist_{HORIZON_BDAYS}d", f"{HORIZON_BDAYS}-day"
hist_5d_stock = rolling_horizon_log_returns(equity_log_returns, HORIZON_BDAYS)
hist_5d_yield = rolling_horizon_yield_changes(yield_changes, HORIZON_BDAYS)
sim_d10 = key_rate_scenarios["DGS10"] - current_key_rates["DGS10"]

# (1) Risk scaling: simulated horizon std vs historical rolling same-length std
scaling_table = pd.DataFrame({
    f"{_HIST}_std": [hist_5d_stock[REPRESENTATIVE_EQUITY].std(), hist_5d_yield["DGS10"].std()],
    f"{_SIM}_std": [stock_log_return_scenarios[REPRESENTATIVE_EQUITY].std(), sim_d10.std()],
}, index=[f"{REPRESENTATIVE_EQUITY}_return", "DGS10_change"])
scaling_table["sim/hist"] = scaling_table[f"{_SIM}_std"] / scaling_table[f"{_HIST}_std"]
print(f"Risk scaling (simulated {HORIZON_LABEL} vs historical {_HD} std):")
display(scaling_table)

# (2) Tail behaviour: excess kurtosis (pandas .kurt() is the Fisher / excess kurtosis)
def _exkurt(x):
    return float(pd.Series(x).kurt())

kurt_table = pd.DataFrame({
    f"{_HIST}_excess_kurtosis": [_exkurt(hist_5d_stock[REPRESENTATIVE_EQUITY]), _exkurt(hist_5d_yield["DGS10"])],
    f"{_SIM}_excess_kurtosis": [_exkurt(stock_log_return_scenarios[REPRESENTATIVE_EQUITY]), _exkurt(sim_d10)],
}, index=[f"{REPRESENTATIVE_EQUITY}_return", "DGS10_change"])
print(f"Tail behaviour (excess kurtosis, simulated vs historical {_HD}):")
display(kurt_table)

# (3) Cross-equity dependence: simulated vs historical 5-day correlation matrices
sim_corr = stock_log_return_scenarios.corr()
hist_corr = hist_5d_stock.corr()

def _avg_offdiag(corr_df):
    arr = corr_df.to_numpy().copy()
    np.fill_diagonal(arr, np.nan)
    return float(np.nanmean(arr))

cross_equity_compare = pd.Series({
    "sim_avg_cross_equity_corr": _avg_offdiag(sim_corr),
    "hist_avg_cross_equity_corr": _avg_offdiag(hist_corr),
    "max_abs_pairwise_corr_diff": float(np.nanmax(np.abs(sim_corr.to_numpy() - hist_corr.to_numpy()))),
})
print(f"Cross-equity dependence (simulated vs historical {_HD}):")
display(cross_equity_compare.to_frame("value"))

# (4) Equity-vs-rates dependence: corr(equity 5-day return, 5-day change in DGS10)
sim_eq_rate_corr = pd.Series(
    {t: float(np.corrcoef(stock_log_return_scenarios[t], sim_d10)[0, 1]) for t in TICKERS}
)
hist_joint = pd.concat([hist_5d_stock, hist_5d_yield["DGS10"].rename("DGS10_chg")], axis=1).dropna()
hist_eq_rate_corr = pd.Series({t: float(hist_joint[t].corr(hist_joint["DGS10_chg"])) for t in TICKERS})
# v23: the same correlation over the most recent ~3 years only. The stock-bond link changes SIGN across
# regimes (negative return correlation 2000-2021, positive since 2022); a bootstrap over a long window
# mixes them, so this column shows how far today's regime is from the window average the engine draws on.
_recent = hist_joint.iloc[-756:]
hist_eq_rate_corr_recent = pd.Series({t: float(_recent[t].corr(_recent["DGS10_chg"])) for t in TICKERS})
eq_rate_compare = pd.DataFrame({"sim": sim_eq_rate_corr, "hist": hist_eq_rate_corr,
                                "hist_last_3y": hist_eq_rate_corr_recent})
eq_rate_compare.loc["AVERAGE"] = eq_rate_compare.mean()
print(f"Equity vs DGS10-change correlation (simulated vs historical {_HD}; last column: last ~3 years only):")
display(eq_rate_compare)
_er = eq_rate_compare.loc["AVERAGE"]
_er_recent_side = abs(_er["sim"] - _er["hist_last_3y"]) < abs(_er["sim"] - _er["hist"])
if (np.sign(_er["hist_last_3y"]) != np.sign(_er["hist"]) and len(hist_joint) > 2 * 756
        and min(abs(_er["hist_last_3y"]), abs(_er["hist"])) > 0.05 and _er_recent_side):
    # v24: with the current-regime hierarchical structure the engine can sit on the recent side
    print(f"The stock-yield link has changed sign over the window ({_er['hist']:+.3f} on average, "
          f"{_er['hist_last_3y']:+.3f} over the last 3 years) and the engine gives {_er['sim']:+.3f}: it follows the "
          "recent regime, not the window average.")
elif (np.sign(_er["hist_last_3y"]) != np.sign(_er["hist"]) and len(hist_joint) > 2 * 756
        and min(abs(_er["hist_last_3y"]), abs(_er["hist"])) > 0.05):
    print(f"The stock-yield link has changed sign over the window ({_er['hist']:+.3f} on average, "
          f"{_er['hist_last_3y']:+.3f} over the last 3 years) and the engine, drawing on the whole window, "
          f"gives {_er['sim']:+.3f}: it imports the older regime into today's scenarios, which matters for "
          "any Treasury hedge. v24: FACTOR_STRUCTURE='hierarchical' with GLOBAL_STRUCTURE_HALFLIFE_YEARS transmits the "
          "global shocks through the current regime's loadings; the walk-forward 'stock_bond_40_60' book tests it.")

# (5) Equity-credit dependence (the Merton-type link): corr(equity 5-day return, 5-day change in
# the representative-rating OAS). The joint innovation bootstrap should reproduce the empirical
# (negative) co-movement: when equity falls, credit spreads widen.
hist_5d_spread = rolling_horizon_yield_changes(spread_changes, HORIZON_BDAYS)
# Validate the engine against ITS OWN history, i.e. in the space it actually simulates: with the
# bucket bridge active the native credit panel is the Moody's 2-point curve (preserved by the
# simulator under "credit_spread_scenarios_raw"); the bridged bucket scenarios are a repricing view.
_val_credit_scen = sim.get("credit_spread_scenarios_raw", credit_spread_scenarios)
_val_rep = REPRESENTATIVE_RATING if REPRESENTATIVE_RATING in _val_credit_scen.columns else _val_credit_scen.columns[-1]
sim_dspread_rep = _val_credit_scen[_val_rep] - current_spreads[_val_rep]

sim_eq_credit_corr = pd.Series(
    {t: float(np.corrcoef(stock_log_return_scenarios[t], sim_dspread_rep)[0, 1]) for t in TICKERS}
)
hist_ec = pd.concat(
    [hist_5d_stock, hist_5d_spread[_val_rep].rename("spread_chg")], axis=1
).dropna()
hist_eq_credit_corr = pd.Series({t: float(hist_ec[t].corr(hist_ec["spread_chg"])) for t in TICKERS})
eq_credit_compare = pd.DataFrame({"sim": sim_eq_credit_corr, "hist": hist_eq_credit_corr})
eq_credit_compare.loc["AVERAGE"] = eq_credit_compare.mean()
print(f"Equity vs {_val_rep} credit-change correlation (simulated vs historical {_HD}):")
display(eq_credit_compare)

# (6) Credit-spread risk scaling: simulated one-week vs historical 5-day std of the OAS change
credit_scaling = pd.DataFrame({
    f"{_HIST}_std": [hist_5d_spread[r].std() for r in _val_credit_scen.columns],
    f"{_SIM}_std": [(_val_credit_scen[r] - current_spreads[r]).std() for r in _val_credit_scen.columns],
}, index=list(_val_credit_scen.columns))
credit_scaling["sim/hist"] = credit_scaling[f"{_SIM}_std"] / credit_scaling[f"{_HIST}_std"]
print(f"Credit-spread risk scaling (simulated {HORIZON_LABEL} vs historical {_HD} std, by rating):")
display(credit_scaling)

# (6b) Bucket view of the 2-point -> bucket bridge (v22, moody_2pt only). The table above validates the
# engine in the 2-point space it simulates, which hides what the bridge does to the BUCKETS the bonds
# are priced on. Measured on the overlap itself (no simulation, so no calm-vs-24-year regime mismatch):
# the share of each bucket's h-day variance the bridged 2-point move reproduces, and the residual share
# that CREDIT_RESIDUAL_MODE="restore" adds back at the horizon.
if credit_bridge is not None:
    _ov = credit_bridge["residuals"].index
    _dbk = credit_bucket_panel.diff().reindex(_ov)
    _fit = pd.DataFrame(spread_changes.reindex(_ov).to_numpy() @ credit_bridge["gamma"].to_numpy().T,
                        index=_ov, columns=credit_bridge["gamma"].index)
    _hb = _dbk.rolling(HORIZON_BDAYS).sum().dropna()
    _hf = _fit.rolling(HORIZON_BDAYS).sum().dropna()
    _hr = credit_bridge["residuals"].rolling(HORIZON_BDAYS).sum().dropna()
    _bf = int(credit_bridge.get("frequency_bdays", 1))
    bridge_bucket_view = pd.DataFrame({
        ("daily_r2" if _bf == 1 else f"fit_r2_{_bf}d"): credit_bridge["r2"],
        f"bridged_share_of_{_HD}_var": _hf.var() / _hb.var(),
        f"residual_share_of_{_HD}_var": _hr.var() / _hb.var(),
    }).reindex(list(credit_bridge["gamma"].index))
    print(f"Bridge, bucket view on the {len(_ov)}-day overlap ({_ov[0].date()} -> {_ov[-1].date()}): share of each "
          f"bucket's {_HD} OAS-change variance carried by the bridged 2-point move, and the residual share "
          f"the bridge drops (CREDIT_RESIDUAL_MODE={CREDIT_RESIDUAL_MODE!r}: "
          + ("added back at the horizon" if CREDIT_RESIDUAL_MODE == "restore" else "currently dropped") + "):")
    display(bridge_bucket_view.round(3))

# (7) FX block: simulated one-week vs historical 5-day std of the FX log-return, plus the equity-FX
# dependence (corr of the equity 5-day return with the 5-day FX log-return). Same sim-vs-hist logic
# as the other blocks; for a single pair this checks that the FX factor is scaled correctly and that
# its (weak) co-movement with equities is reproduced.
hist_5d_fx = rolling_horizon_log_returns(fx_log_returns, HORIZON_BDAYS)
sim_fx_ret = {p: np.log(fx_spot_scenarios[p] / current_fx[p]) for p in fx_spot_scenarios.columns}

fx_scaling = pd.DataFrame({
    f"{_HIST}_std": [hist_5d_fx[p].std() for p in fx_spot_scenarios.columns],
    f"{_SIM}_std": [sim_fx_ret[p].std() for p in fx_spot_scenarios.columns],
}, index=list(fx_spot_scenarios.columns))
fx_scaling["sim/hist"] = fx_scaling[f"{_SIM}_std"] / fx_scaling[f"{_HIST}_std"]
print(f"FX risk scaling (simulated {HORIZON_LABEL} vs historical {_HD} std of the FX log-return):")
display(fx_scaling)

rep_fx = list(fx_spot_scenarios.columns)[0]
sim_eq_fx_corr = pd.Series(
    {t: float(np.corrcoef(stock_log_return_scenarios[t], sim_fx_ret[rep_fx])[0, 1]) for t in TICKERS}
)
hist_ef = pd.concat([hist_5d_stock, hist_5d_fx[rep_fx].rename("fx_ret")], axis=1).dropna()
hist_eq_fx_corr = pd.Series({t: float(hist_ef[t].corr(hist_ef["fx_ret"])) for t in TICKERS})
eq_fx_compare = pd.DataFrame({"sim": sim_eq_fx_corr, "hist": hist_eq_fx_corr})
eq_fx_compare.loc["AVERAGE"] = eq_fx_compare.mean()
print(f"Equity vs {rep_fx} {_HD}-return correlation (simulated vs historical {_HD}):")
display(eq_fx_compare)

# (8) Crisis-tail overlay effect. The seven checks above validate the engine against the CALM
# 3-year sample. The crisis overlay is an opt-in stress layer that deliberately fattens the tail
# beyond that calm sample (the sample contains no real crisis). Here we quantify it: re-simulate the
# equal-weight equity one-week loss under the calm engine and under both crisis modes, calibrated
# from the long Moody's Baa-Aaa history, and compare the 95% ES and the excess kurtosis. This is a
# demonstration of the overlay, independent of the active CRISIS_MODE used elsewhere in the notebook.
def _eq_loss_tail(crisis_mode):
    if FACTOR_STRUCTURE == "hierarchical":       # v24: the active engine (blockwise call below unchanged)
        s = simulate_joint_horizon(**{**main_sim_kwargs, "n_scenarios": 8000, "rng_seed": RNG_SEED,
                                      "crisis_mode": crisis_mode})
        port = s["stock_log_return_scenarios"].mean(axis=1)
        loss = -port
        thr = loss.quantile(0.95)
        return {"std": float(port.std()), "ES_95_loss": float(loss[loss >= thr].mean()),
                "excess_kurtosis": float(port.kurtosis()), "worst_1pct": float(loss.quantile(0.99))}
    s = simulate_joint_horizon(
        var_res=var_res, resid_hist=var_res.resid, last_state=drivers.iloc[-1].to_numpy(),
        equity_block=equity_block, rates_block=rates_block,
        current_prices=current_prices, current_key_rates=current_key_rates,
        horizon_bdays=HORIZON_BDAYS, n_scenarios=8000, avg_block_length=effective_block_length,
        rng_seed=RNG_SEED, daily_rf=current_daily_rf,
        resid_idio_std=equity_block.get("resid_idiosyncratic_std"),
        credit_block=credit_block, current_spreads=current_spreads,
        fx_block=fx_block, current_fx=current_fx,
        dynamics=DRIVER_DYNAMICS, demeaned_drivers=demeaned_drivers, driver_mean=driver_mean,
        crisis_mode=crisis_mode, crisis_prob=crisis_prob_eff, crisis_scale=crisis_scale_eff,
        crisis_pool=crisis_pool_eff, copula=t_copula_model, asset_copula=asset_copula_model, fhs_filter=fhs_filter,
        ou_rates=ou_rates, ou_credit=ou_credit,
    )
    port = s["stock_log_return_scenarios"].mean(axis=1)   # equal-weight equity horizon return
    loss = -port
    thr = loss.quantile(0.95)
    return {"std": float(port.std()), "ES_95_loss": float(loss[loss >= thr].mean()),
            "excess_kurtosis": float(port.kurtosis()),
            "worst_1pct": float(loss.quantile(0.99))}

crisis_compare = pd.DataFrame({m: _eq_loss_tail(m) for m in ["none", "regime_jump", "moody_backbone"]}).T
crisis_compare["ES_uplift_vs_calm"] = crisis_compare["ES_95_loss"] / crisis_compare.loc["none", "ES_95_loss"]
print(
    f"Crisis-tail overlay effect on the equal-weight equity {HORIZON_LABEL} loss "
    f"(Moody's-calibrated: crisis_prob={crisis_prob_eff:.4f}/day, scale={crisis_scale_eff:.2f}x):"
)
display(crisis_compare)

# (9) Tail dependence: do equities go to their joint extremes together more than linear correlation
# implies? We measure the lower-tail co-exceedance -- the probability two names are BOTH below their
# 5% quantile, averaged over pairs, expressed as P(both in 5% tail) / 0.05 (independence -> 0.05,
# comonotone -> 1) -- for the historical 5-day returns, the block-bootstrap sim, and the t-copula sim
# (both crisis off). The block bootstrap resamples observed joint vectors, so it tracks the empirical
# dependence; the t-copula fits a parametric tail-dependent structure that can extrapolate joint
# extremes beyond those observed in the short sample.
def _lower_tail_coexceedance(stock_returns_df, q=0.05):
    R = stock_returns_df.rank(pct=True).to_numpy()
    below = (R < q).astype(float)
    co = (below.T @ below) / below.shape[0]
    iu = np.triu_indices(below.shape[1], k=1)
    return float(co[iu].mean() / q)

def _sim_equity(dyn):
    s = simulate_joint_horizon(
        var_res=var_res, resid_hist=var_res.resid, last_state=drivers.iloc[-1].to_numpy(),
        equity_block=equity_block, rates_block=rates_block, current_prices=current_prices,
        current_key_rates=current_key_rates, horizon_bdays=HORIZON_BDAYS, n_scenarios=8000,
        avg_block_length=effective_block_length, rng_seed=RNG_SEED, daily_rf=current_daily_rf,
        resid_idio_std=equity_block.get("resid_idiosyncratic_std"), credit_block=credit_block,
        current_spreads=current_spreads, fx_block=fx_block, current_fx=current_fx,
        dynamics=dyn, demeaned_drivers=demeaned_drivers, driver_mean=driver_mean,
        crisis_mode="none", copula=t_copula_model, asset_copula=asset_copula_model, fhs_filter=fhs_filter,
        ou_rates=ou_rates, ou_credit=ou_credit,
    )
    return s["stock_log_return_scenarios"]

tail_dep = pd.Series({
    f"historical_{HORIZON_BDAYS}d": _lower_tail_coexceedance(hist_5d_stock[TICKERS]),
    "block_bootstrap": _lower_tail_coexceedance(_sim_equity("block_bootstrap")),
    "factor_t_copula": _lower_tail_coexceedance(_sim_equity("t_copula")),
    "asset_copula": _lower_tail_coexceedance(_sim_equity("asset_copula")),
})
if FACTOR_STRUCTURE == "hierarchical":           # v24: the active engine, same size / seed / no crisis
    tail_dep["hierarchical (active)"] = _lower_tail_coexceedance(simulate_joint_horizon(
        **{**main_sim_kwargs, "n_scenarios": 8000, "rng_seed": RNG_SEED, "crisis_mode": "none"}
    )["stock_log_return_scenarios"][TICKERS])
print(
    f"Lower-tail co-exceedance of equities (P(pair both in 5% tail)/0.05; 0.05 = independence) -- "
    f"factor copula nu={t_copula_model['nu']:.1f}, asset copula nu={asset_copula_model['nu']:.1f}:"
)
display(tail_dep.to_frame("coexceedance"))
print(
    "The block bootstrap and the asset-level copula are both expected to track the historical "
    "equity co-tail (the bootstrap resamples observed joint vectors; the asset copula fits the "
    "dependence where the real correlation lives), whereas the factor-level copula -- fit on the "
    "orthogonal PCs -- understates it because the equity co-tail is a shared-loadings effect, not a "
    "factor-innovation-copula effect. The asset copula can also extrapolate joint extremes beyond "
    "the observed sample, which the bootstrap cannot."
)

# Marginal distribution shape, historical vs simulated
plot_histogram(
    hist_5d_stock[REPRESENTATIVE_EQUITY],
    f"Historical rolling {_HD} {REPRESENTATIVE_EQUITY} log-returns",
    f"{_HD} log-return",
)
plot_histogram(
    stock_log_return_scenarios[REPRESENTATIVE_EQUITY],
    f"Simulated {HORIZON_LABEL} {REPRESENTATIVE_EQUITY} log-returns",
    f"{HORIZON_LABEL} log-return",
)


# Conditional-correlation payoff (FHS only): the headline claim is that asset correlations rise in
# stress *for free*, because the shared SYSTEMATIC factors' conditional vol scales every loading on
# them together -- no asset correlation matrix is fitted. To isolate that mechanism we scale only the
# systematic equity factors' (market/size/value/momentum) current conditional variance, holding the
# idiosyncratic residual PCs and the name-specific noise at their current level: a real crisis is
# precisely the common factor blowing out relative to idiosyncratic risk, which is what lifts the
# systematic share of every name's return and pulls correlations up. (Scaling *all* factors uniformly
# would understate this, since it scales idiosyncratic risk in lockstep and only shrinks the small
# fixed noise term.)
if CONDITIONAL_VOL == "ewma_fhs" and fhs_filter is not None:
    _sys_eq_idx = [i for i, c in enumerate(drivers.columns)
                   if not str(c).startswith(("resid_pc", "rate_pc", "credit_pc", "fx"))]
    _sys_eq_names = [str(drivers.columns[i]) for i in _sys_eq_idx]

    def _sim_equity_vol(sys_scale):
        f2 = dict(fhs_filter)
        cv = np.array(fhs_filter["cond_var_now"], dtype=float).copy()
        cv[_sys_eq_idx] = cv[_sys_eq_idx] * sys_scale      # scale ONLY the systematic equity factors
        f2["cond_var_now"] = cv
        s = simulate_joint_horizon(
            var_res=var_res, resid_hist=var_res.resid, last_state=drivers.iloc[-1].to_numpy(),
            equity_block=equity_block, rates_block=rates_block, current_prices=current_prices,
            current_key_rates=current_key_rates, horizon_bdays=HORIZON_BDAYS, n_scenarios=8000,
            avg_block_length=effective_block_length, rng_seed=RNG_SEED, daily_rf=current_daily_rf,
            resid_idio_std=equity_block.get("resid_idiosyncratic_std"), credit_block=credit_block,
            current_spreads=current_spreads, fx_block=fx_block, current_fx=current_fx,
            dynamics="block_bootstrap", demeaned_drivers=demeaned_drivers, driver_mean=driver_mean,
            crisis_mode="none", fhs_filter=f2,
            ou_rates=ou_rates, ou_credit=ou_credit,
        )["stock_log_return_scenarios"]
        corr = s.corr().to_numpy()
        return float(corr[np.triu_indices(corr.shape[0], 1)].mean())
    _calm = _sim_equity_vol(0.25)      # systematic factors a quarter of their current variance
    _now = _sim_equity_vol(1.0)        # current systematic state
    _stress = _sim_equity_vol(9.0)     # systematic factors at a 3x-vol stress
    print()
    print(("(Blockwise engine, for reference.) " if FACTOR_STRUCTURE == "hierarchical" else "")
          + f"Conditional correlation (FHS): average cross-equity correlation as ONLY the systematic "
          f"equity factors {_sys_eq_names} are scaled from calm to stressed (idiosyncratic risk held "
          "fixed), with no correlation matrix anywhere in the model:")
    display(pd.Series({"calm (systematic x0.25 var)": _calm, "current state": _now,
                       "stressed (systematic x9 var)": _stress},
                      name="avg_cross_equity_corr").to_frame().round(3))
    print(
        f"Cross-equity correlation rises from {_calm:.3f} (calm) to {_stress:.3f} (stressed) purely "
        "because the shared systematic factors' higher conditional volatility lifts the systematic "
        "share of each name's return relative to its (fixed) idiosyncratic risk. This is the "
        "'correlations go to one in a crash' behaviour, reproduced structurally rather than by "
        "fitting a conditional correlation matrix on the asset pairs."
    )

    # v23 -- is the long-sample "understated correlation" a defect or today's state? Re-simulate with
    # EVERY factor at its long-run variance (no vol state) and compare with the current state and with
    # the historical average over the window (which pools calm and crisis periods).
    def _avg_corr_at(cond_var):
        f2 = dict(active_fhs_filter)
        f2["cond_var_now"] = np.asarray(cond_var, dtype=float).copy()
        s = simulate_joint_horizon(**{**main_sim_kwargs, "n_scenarios": 8000, "rng_seed": RNG_SEED,
                                      "crisis_mode": "none", "fhs_filter": f2})["stock_log_return_scenarios"][TICKERS]
        c = s.corr().to_numpy()
        return float(c[np.triu_indices(c.shape[0], 1)].mean())
    _corr_now = _avg_corr_at(active_fhs_filter["cond_var_now"])
    _corr_neutral = _avg_corr_at(active_fhs_filter["uncond_var"])
    _corr_hist = float(cross_equity_compare["hist_avg_cross_equity_corr"])
    print()
    display(pd.Series({"current state": _corr_now, "neutral state (every factor at long-run variance)": _corr_neutral,
                       f"historical {_HD} average over the window": _corr_hist},
                      name="avg_cross_equity_corr").to_frame().round(3))
    if abs(_corr_hist - _corr_now) < 0.02:
        print(f"Simulated ({_corr_now:.3f}) and historical ({_corr_hist:.3f}) cross-equity correlation already agree "
              f"within 0.02; the neutral state gives {_corr_neutral:.3f}.")
    else:
        _closed = (_corr_neutral - _corr_now) / (_corr_hist - _corr_now)
        print(f"Moving from today's state to the neutral state closes {_closed:.0%} of the gap between the simulated "
              f"({_corr_now:.3f}) and the historical ({_corr_hist:.3f}) cross-equity correlation. What it closes is the "
              "current vol state; what remains is structural (or the historical figure's crisis weighting: a pooled "
              "correlation is dominated by high-variance periods, where correlations are high).")


# (9b) Cross-asset dependence: today's state vs the neutral state (v24). The v23 run found the simulated
# cross-asset correlations at 0.37-0.61 of their historical values, and the neutral-state test showed that
# for the CROSS-EQUITY correlation 91% of the gap was today's vol state. The same test for the cross-asset
# pairs: re-simulate with every factor at its long-run variance and see how much of each gap closes. What
# does not close is structural -- or the historical figure's pooling of regimes (the 'last 3y' column).
# With FACTOR_STRUCTURE="hierarchical" the blockwise engine is shown alongside, same seed and size.
def _xasset_corrs(eq, dy, dcr, dfx):
    _e = pd.DataFrame(eq)[TICKERS]
    def _avg(x):
        return float(np.mean([np.corrcoef(_e[t], x)[0, 1] for t in TICKERS]))
    return {"equity / DGS10": _avg(dy), f"equity / {_val_rep}": _avg(dcr), f"equity / {rep_fx}": _avg(dfx),
            f"DGS10 / {_val_rep}": float(np.corrcoef(dy, dcr)[0, 1]),
            f"DGS10 / {rep_fx}": float(np.corrcoef(dy, dfx)[0, 1]),
            f"{_val_rep} / {rep_fx}": float(np.corrcoef(dcr, dfx)[0, 1])}

def _xasset_from_sim(s):
    _cr = s.get("credit_spread_scenarios_raw", s["credit_spread_scenarios"])
    return _xasset_corrs(s["stock_log_return_scenarios"],
                         (s["key_rate_scenarios"]["DGS10"] - current_key_rates["DGS10"]).to_numpy(),
                         (_cr[_val_rep] - current_spreads[_val_rep]).to_numpy(),
                         np.log(s["fx_spot_scenarios"][rep_fx] / current_fx[rep_fx]).to_numpy())

_hx = pd.concat([hist_5d_stock[TICKERS], hist_5d_yield["DGS10"].rename("_dy"),
                 hist_5d_spread[_val_rep].rename("_dcr"), hist_5d_fx[rep_fx].rename("_dfx")], axis=1).dropna()
_xa = {"hist": _xasset_corrs(_hx[TICKERS], _hx["_dy"], _hx["_dcr"], _hx["_dfx"])}
_hx3 = _hx.iloc[-756:]
_xa["hist_last_3y"] = _xasset_corrs(_hx3[TICKERS], _hx3["_dy"], _hx3["_dcr"], _hx3["_dfx"])
_XA_N = 8000
_xa["sim_current"] = _xasset_from_sim(simulate_joint_horizon(
    **{**main_sim_kwargs, "n_scenarios": _XA_N, "rng_seed": RNG_SEED, "crisis_mode": "none"}))
if CONDITIONAL_VOL == "ewma_fhs" and active_fhs_filter is not None:
    _f_neutral = {**active_fhs_filter, "cond_var_now": np.asarray(active_fhs_filter["uncond_var"], dtype=float).copy()}
    _xa["sim_neutral"] = _xasset_from_sim(simulate_joint_horizon(
        **{**main_sim_kwargs, "n_scenarios": _XA_N, "rng_seed": RNG_SEED, "crisis_mode": "none", "fhs_filter": _f_neutral}))
if FACTOR_STRUCTURE == "hierarchical":
    _bw_kw = {**main_sim_kwargs, "demeaned_drivers": demeaned_drivers, "fhs_filter": fhs_filter, "hier": None,
              "n_scenarios": _XA_N, "rng_seed": RNG_SEED, "crisis_mode": "none"}
    _xa["blockwise_current"] = _xasset_from_sim(simulate_joint_horizon(**_bw_kw))
    if fhs_filter is not None:
        _xa["blockwise_neutral"] = _xasset_from_sim(simulate_joint_horizon(
            **{**_bw_kw, "fhs_filter": {**fhs_filter, "cond_var_now": np.asarray(fhs_filter["uncond_var"], dtype=float).copy()}}))
cross_asset_state_compare = pd.DataFrame(_xa)
print()
print(f"Cross-asset correlation ({_HD} changes; equity rows are averages over the {len(TICKERS)} names): history, "
      "the last ~3 years, and the simulated engine from today's state and from the neutral state (every factor at "
      "its long-run variance)" + (", with the blockwise engine alongside:" if FACTOR_STRUCTURE == "hierarchical" else ":"))
display(cross_asset_state_compare.round(3))
_cx = cross_asset_state_compare
if FACTOR_STRUCTURE == "hierarchical" and GLOBAL_STRUCTURE_HALFLIFE_YEARS is not None:
    # The engine targets the CURRENT regime, so the pooled history is not its yardstick: count the pairs on
    # which each engine sits closer to the last ~3 years than to the window average.
    _sides = {}
    for _col in [c_ for c_ in ("sim_current", "blockwise_current") if c_ in _cx.columns]:
        _sides[_col] = int(((_cx[_col] - _cx["hist_last_3y"]).abs() < (_cx[_col] - _cx["hist"]).abs()).sum())
    print("Regime check (current-regime structure, GLOBAL_STRUCTURE_HALFLIFE_YEARS="
          f"{GLOBAL_STRUCTURE_HALFLIFE_YEARS:g}): pairs on which the engine sits closer to the last ~3 years than to the "
          "window average -- " + ", ".join(f"{k_} {v_} of {len(_cx)}" for k_, v_ in _sides.items()) + ".")
elif "sim_neutral" in _cx.columns:
    _gap = _cx["hist"] - _cx["sim_current"]
    _closed = (_cx["sim_neutral"] - _cx["sim_current"]) / _gap
    _closed = _closed[_gap.abs() >= 0.05]
    if len(_closed):
        print("Share of each gap (history minus today's simulation, pairs with a gap of at least 0.05) that the neutral "
              "state closes: " + ", ".join(f"{k_} {v_:.0%}" for k_, v_ in _closed.items()) + ". Close to 100%: today's "
              "vol state explains the gap; well below: structure, or the history pools regimes (compare 'hist_last_3y').")
    else:
        print("No pair is 0.05 or more away from its window average: nothing for the neutral state to explain.")


# (10) Term structure of risk (v21). The SAME engine (main_sim_kwargs), from the SAME current state,
# simulated at 1W, 1M, 3M and 1Y, against the square-root-of-time rule applied to the 1W number -- the
# way a one-week model is usually stretched to longer horizons, and the way FRTB's liquidity-horizon
# cascade scales a 10-day ES. Each "_vs_sqrt_t" column is the h-day figure over sqrt(h/5) times the 1W
# figure: above 1, risk grows FASTER than sqrt(t); below 1, slower. The equity vol ratio is decomposed:
#   * eq_vol_vs_sqrt_t          -- the engine from TODAY's state;
#   * eq_vol_neutral_vs_sqrt_t  -- (FHS only) the same engine with every factor's conditional variance
#                                  set to its long-run level: no vol state left, only the dynamics;
#   * eq_vol_hist_vs_sqrt_t     -- the sample's own variance ratio (rolling h-day vs 5-day sums of the
#                                  equal-weight portfolio over the aligned window).
# today / neutral is the STATE effect sqrt(t) cannot see (a calm state reverts UP, a stressed one DOWN);
# neutral vs hist says whether the bootstrap carries the sample's serial dependence (it carries lags up
# to about one block, effective_block_length days, and not beyond). The equity VaR ratio also carries
# the drift (the compounded risk premium lowers a long-horizon VaR). ES/VaR tracks the SHAPE of the
# tail: 1.254 for a normal law at 95%. Independent of HORIZON_MODE; its own seed, nothing else moves.
_TS_N = 4_000
_ts_neutral_fhs = None
if CONDITIONAL_VOL == "ewma_fhs" and active_fhs_filter is not None:
    _ts_neutral_fhs = {**active_fhs_filter, "cond_var_now": np.asarray(active_fhs_filter["uncond_var"], dtype=float).copy()}
_ts_cr = REPRESENTATIVE_RATING
_ts_ew_hist = equity_log_returns[TICKERS].mean(axis=1)


def _ts_sim(h, fhs=None):
    kw = {**main_sim_kwargs, "horizon_bdays": h, "n_scenarios": _TS_N, "rng_seed": RNG_SEED + 31}
    if fhs is not None:
        kw["fhs_filter"] = fhs
    return simulate_joint_horizon(**kw)


_ts_rows = {}
for _m, _h in HORIZON_BDAYS_BY_MODE.items():
    _s = _ts_sim(_h)
    _eq = _s["stock_log_return_scenarios"][TICKERS].mean(axis=1).to_numpy()
    _var = float(-np.quantile(_eq, 0.05))
    _es = sample_expected_shortfall(_eq, 0.05)
    _row = {"bdays": _h, "eq_mean": float(_eq.mean()), "eq_VaR_95": _var, "eq_ES_95": _es,
            "eq_ES/VaR": _es / _var if _var > 0 else np.nan,
            "_eq_vol": float(_eq.std()),
            "_eq_vol_hist": float(_ts_ew_hist.rolling(_h).sum().dropna().std()),
            "_DGS10": float((_s["key_rate_scenarios"]["DGS10"] - current_key_rates["DGS10"]).std())}
    _crs = _s.get("credit_spread_scenarios")
    if _crs is not None and _ts_cr in _crs.columns:
        _row["_credit"] = float((_crs[_ts_cr] - bucket_anchor_spreads[_ts_cr]).std())
    if _ts_neutral_fhs is not None:
        _sn = _ts_sim(_h, _ts_neutral_fhs)
        _row["_eq_vol_neutral"] = float(_sn["stock_log_return_scenarios"][TICKERS].mean(axis=1).std())
    _ts_rows[_m] = _row
_ts = pd.DataFrame(_ts_rows).T
_sqrt_t = np.sqrt(_ts["bdays"].astype(float) / float(HORIZON_BDAYS_BY_MODE["1W"]))
risk_term_structure = _ts[["bdays", "eq_mean", "eq_VaR_95", "eq_ES_95", "eq_ES/VaR"]].copy()
for _col, _lab in [("_eq_vol", "eq_vol_vs_sqrt_t"), ("_eq_vol_neutral", "eq_vol_neutral_vs_sqrt_t"),
                   ("_eq_vol_hist", "eq_vol_hist_vs_sqrt_t"), ("eq_VaR_95", "eq_VaR_vs_sqrt_t"),
                   ("_DGS10", "DGS10_vs_sqrt_t"), ("_credit", f"{_ts_cr}_vs_sqrt_t")]:
    if _col in _ts.columns:
        risk_term_structure[_lab] = _ts[_col] / (_sqrt_t * float(_ts.loc["1W", _col]))
print()
print("Term structure of risk from the current state (equal-weight equity, DGS10 change, "
      f"{_ts_cr} spread change): each horizon simulated directly, over the 1W figure scaled by sqrt(h/5):")
display(risk_term_structure.round(4))
_r = float(risk_term_structure.loc["3M", "eq_vol_vs_sqrt_t"])
_rh = float(risk_term_structure.loc["3M", "eq_vol_hist_vs_sqrt_t"])
_n3 = len(_ts_ew_hist) / 63.0
if "eq_vol_neutral_vs_sqrt_t" in risk_term_structure.columns:
    _rn = float(risk_term_structure.loc["3M", "eq_vol_neutral_vs_sqrt_t"])
    _state = _r / _rn
    _why = (f"Today's vol STATE accounts for a factor {_state:.2f}: "
            + ("the state is calmer than the long-run level and the filter reverts it UP, so risk grows "
               "faster than sqrt(t)" if _state > 1.01 else
               "the state is more stressed than the long-run level and the filter reverts it DOWN, so risk "
               "grows slower than sqrt(t)" if _state < 0.99 else "the state is close to the long-run level")
            + f". With the state set to its long-run level the ratio is {_rn:.2f}: that part is the serial "
            "dependence the block bootstrap carries.")
    _dyn = _rn
else:
    _why = ("There is no vol state (CONDITIONAL_VOL='none'), so the whole gap from 1 is the short-lag serial "
            f"dependence the block bootstrap carries from the sample, up to about one block ({effective_block_length} days).")
    _dyn = _r
print(
    f"At 3M the engine's equity vol is {_r:.2f}x the sqrt(t)-scaled 1W vol. {_why} The sample's own variance "
    f"ratio at 3M is {_rh:.2f}"
    + (f" -- noisy: a 3M window fits only about {_n3:.0f} times in this sample" if _n3 < 40 else "")
    + (". Engine and sample agree on the dynamics." if abs(_dyn - _rh) < 0.05 else
       f". The gap to the engine's {_dyn:.2f} is dependence at lags longer than a bootstrap block, which the "
       "engine does not carry.")
    + " A one-week model stretched by sqrt(t) misses the state effect entirely (Diebold, Hickman, Inoue & "
    "Schuermann: sqrt-h scaling exaggerates the swings of long-horizon volatility). BACKTEST_HORIZONS tests "
    "this term structure against realized returns."
)
_m1y, _v1y = float(risk_term_structure.loc["1Y", "eq_mean"]), float(_ts.loc["1Y", "_eq_vol"])
print(
    f"Drift (DRIFT_MODE={DRIFT_MODE!r}): the equal-weight book's expected one-year log return is {_m1y:+.1%} "
    f"against a one-year vol of {_v1y:.1%}, i.e. {_m1y / _v1y:+.2f} standard deviations -- the gap between "
    "eq_VaR_vs_sqrt_t and eq_vol_vs_sqrt_t. "
    + ("Under 'sample' it is the estimation window's own average return projected forward; 'factor' prices "
       "it on long-run premia, 'zero' removes it." if DRIFT_MODE == "sample" else
       "It now comes from long-run Fama-French premia, not from the window's own return." if DRIFT_MODE == "factor"
       else "With no drift, VaR scales with vol alone.")
)


### 3b. Walk-forward VaR backtest (coverage and calibration)

A single as-of date cannot validate a scenario engine. Here the engine is **refit on an expanding, point-in-time window** at a panel of past as-of dates. At each date we simulate the horizon distribution (one week by default) of an **equal-weight equity** portfolio, take its 95% VaR and ES, and compare them with the **realized** forward return over the same horizon. As-of dates are spaced at least one horizon apart, so the forecast windows **never overlap** and the tests' null distributions hold.

The tests:

- **Kupiec** proportion of failures (unconditional coverage).
- **Christoffersen** independence (no clustering of breaches) and their **joint conditional-coverage** test.
- The **Acerbi–Székely** expected-shortfall test. It checks the risk measure the portfolio layer actually minimizes.
- The **PIT / rank histogram**, which should be roughly uniform.

With `BACKTEST_MODE="full"` the panel spans the whole aligned window, and coverage is also sliced by volatility regime and by calendar year. With `BACKTEST_HORIZONS` the same refits are reused at several horizons, giving coverage as a function of the horizon.

**v24 — multi-asset books and paired structures.**
- **`BACKTEST_PORTFOLIOS`** adds three books to the equal-weight equity headline book:
  - a 40/60 stock–Treasury book;
  - a 40/20/40 stock–Treasury–corporate book;
  - the min-CVaR LP re-solved at every date. Its weights are chosen on half the scenarios and its VaR is read on the other half.

  Each book is priced identically on simulated and realized moves, and is reported before and from `BACKTEST_REGIME_SPLIT`, next to the engine's predicted stock–bond correlation and the realized one.
- **`BACKTEST_STRUCTURES`** runs several factor structures on the same refits, dates and seeds: blockwise; hierarchical; hierarchical with static loadings; hierarchical with joint resampling. Their difference is then attributable to one modelling choice at a time.

This is the most expensive cell because it refits the full pipeline at each date on live data. The defaults (`BACKTEST_N_DATES`, `BACKTEST_N_SCENARIOS`) are deliberately modest; increase them, or use `"full"`, for more statistical power.


In [ ]:
# Walk-forward VaR/ES backtest. Kupiec (unconditional coverage), Christoffersen (independence) and
# their JOINT conditional-coverage test, an Acerbi-Szekely expected-shortfall test, a PIT histogram,
# and -- with BACKTEST_MODE="full" -- coverage sliced by volatility regime and by calendar year. With
# BACKTEST_HORIZONS (v21) the same refits are reused at several horizons, each on non-overlapping
# windows, giving coverage as a function of the horizon. The engine is refit on an expanding,
# point-in-time window at each as-of date (no lookahead).

def _build_active_equity_block_v26(log_returns, market_prices, ff_panel, rf_d, halflife_days=None):
    """Static legacy builder when H=None; weighted current-regime builder otherwise."""
    if halflife_days is None:
        if ACTIVE_EQUITY_ENGINE == "latent_statistical":
            return build_latent_statistical_equity_block(log_returns, rf_d, TARGET_RESID_EXPLAINED_VAR)
        if ACTIVE_EQUITY_ENGINE == "proxy_market":
            return build_proxy_market_equity_block(log_returns, market_prices, rf_d, TARGET_RESID_EXPLAINED_VAR)
        if ACTIVE_EQUITY_ENGINE == "apt_ff3":
            return build_apt_equity_block(log_returns, rf_d, ff_panel, ["MKT_RF", "SMB", "HML"],
                                          "apt_ff3", TARGET_RESID_EXPLAINED_VAR)
        if ACTIVE_EQUITY_ENGINE == "apt_carhart":
            return build_apt_equity_block(log_returns, rf_d, ff_panel, ["MKT_RF", "SMB", "HML", "MOM"],
                                          "apt_carhart", TARGET_RESID_EXPLAINED_VAR)
    else:
        if ACTIVE_EQUITY_ENGINE == "latent_statistical":
            return build_latent_statistical_equity_block_weighted(
                log_returns, rf_d, TARGET_RESID_EXPLAINED_VAR, halflife_days=halflife_days)
        if ACTIVE_EQUITY_ENGINE == "proxy_market":
            return build_proxy_market_equity_block_weighted(
                log_returns, market_prices, rf_d, TARGET_RESID_EXPLAINED_VAR, halflife_days=halflife_days)
        if ACTIVE_EQUITY_ENGINE == "apt_ff3":
            return build_apt_equity_block_weighted(
                log_returns, rf_d, ff_panel, ["MKT_RF", "SMB", "HML"], "apt_ff3",
                TARGET_RESID_EXPLAINED_VAR, halflife_days=halflife_days)
        if ACTIVE_EQUITY_ENGINE == "apt_carhart":
            return build_apt_equity_block_weighted(
                log_returns, rf_d, ff_panel, ["MKT_RF", "SMB", "HML", "MOM"], "apt_carhart",
                TARGET_RESID_EXPLAINED_VAR, halflife_days=halflife_days)
    raise ValueError(f"Unknown ACTIVE_EQUITY_ENGINE: {ACTIVE_EQUITY_ENGINE}")


def _v26_prepare_asof(asof):
    """Raw point-in-time panel prepared ONCE; architecture-specific fits are built from this."""
    eff = choose_effective_asof(stock_prices_raw, market_proxy_raw, rates_raw, ff_factor_panel_raw,
                                asof, credit=credit_driver_raw)
    sp, mp, kr, ff, cr, _fx = align_market_data(
        stock_prices_raw, market_proxy_raw, rates_raw, ff_factor_panel_raw, eff, credit=credit_driver_raw)
    _ws, _, _ = contiguous_window_start(sp.index, LONG_GAP_TOLERANCE_DAYS)
    if _ws is not None:
        sp, mp, kr, ff, cr = sp.loc[_ws:], mp.loc[_ws:], kr.loc[_ws:], ff.loc[_ws:], cr.loc[_ws:]
    lr = compute_equity_log_returns(sp)
    dy = compute_yield_changes(kr)
    mlr = compute_equity_log_returns(mp)
    ds = compute_spread_changes(cr)
    cd = lr.index.intersection(dy.index).intersection(mlr.index).intersection(ff.index).intersection(ds.index)
    lr, dy, kr, mp, ff, ds, cr = (lr.loc[cd], dy.loc[cd], kr.loc[cd], mp.loc[cd],
                                   ff.loc[cd], ds.loc[cd], cr.loc[cd])
    return {"eff": eff, "sp": sp, "mp": mp, "kr": kr, "ff": ff, "cr": cr,
            "lr": lr, "dy": dy, "ds": ds, "rf_d": ff["RF"]}


def _v26_fit_engine(raw, halflife_days=None):
    """Fit either the legacy equal-weight block geometry or the v26 weighted geometry."""
    lr, dy, ds, mp, ff, rf_d = raw["lr"], raw["dy"], raw["ds"], raw["mp"], raw["ff"], raw["rf_d"]
    eb = _build_active_equity_block_v26(lr, mp, ff, rf_d, halflife_days=halflife_days)
    if halflife_days is None:
        rb = build_rates_factor_model(dy, TARGET_RATE_EXPLAINED_VAR)
        cb = build_credit_factor_model(ds, TARGET_CREDIT_EXPLAINED_VAR, max_factors=CREDIT_MAX_FACTORS)
    else:
        rb = build_rates_factor_model_weighted(dy, TARGET_RATE_EXPLAINED_VAR,
                                               halflife_days=halflife_days)
        cb = build_credit_factor_model_weighted(ds, TARGET_CREDIT_EXPLAINED_VAR,
                                                max_factors=CREDIT_MAX_FACTORS,
                                                halflife_days=halflife_days)
    eb, rb, cb, _, _ = apply_drift_mode(
        eb, rb, cb, None, DRIFT_MODE, ff_long=ff_long_history, asof=raw["eff"], ff_window=ff)
    drv = build_joint_driver_matrix(eb, rb, cb)
    vr = build_var1_model(drv)
    mean = drv.mean()
    fhs = (compute_ewma_fhs_filter(drv.to_numpy(), mean.to_numpy(),
                                   ewma_lambda=VOL_EWMA_LAMBDA,
                                   reversion_halflife=VOL_REVERSION_HALFLIFE_DAYS,
                                   params_mode=VOL_PARAMS_MODE)
           if CONDITIONAL_VOL == "ewma_fhs" else None)
    return {"eb": eb, "rb": rb, "cb": cb, "drv": drv, "vr": vr, "mean": mean, "fhs": fhs,
            "halflife_days": halflife_days}


def _v26_hier_setup(engine, halflife_days):
    hs = build_hierarchical_structure(
        engine["drv"], n_global=GLOBAL_FACTORS, max_global=GLOBAL_FACTORS_MAX,
        resid_corr_tol=GLOBAL_RESID_CORR_TOL, halflife_days=halflife_days,
        k_table=0, local_resampling="independent")
    X = hs["X"]
    fhs = (compute_fhs_filter_for_matrix(
        X.to_numpy(), ewma_lambda=VOL_EWMA_LAMBDA,
        reversion_halflife=VOL_REVERSION_HALFLIFE_DAYS, params_mode=VOL_PARAMS_MODE)
        if CONDITIONAL_VOL == "ewma_fhs" else None)
    return {"dd": X - X.mean(), "fhs": fhs, "hier": hierarchical_sim_spec(hs),
            "k_global": hs["k_global"]}


def _v26_structure_spec(structure, static_engine, adaptive_engine, halflife_days):
    if structure == "blockwise_static":
        e = static_engine
        return {"engine": e, "dd": e["drv"] - e["mean"], "fhs": e["fhs"],
                "hier": None, "k_global": np.nan}
    if structure == "blockwise_adaptive":
        e = adaptive_engine
        return {"engine": e, "dd": e["drv"] - e["mean"], "fhs": e["fhs"],
                "hier": None, "k_global": np.nan}
    if structure == "hierarchical_static":
        e = static_engine
        h = _v26_hier_setup(e, None)
        return {"engine": e, **h}
    if structure == "hierarchical_adaptive":
        e = adaptive_engine
        h = _v26_hier_setup(e, halflife_days)
        return {"engine": e, **h}
    raise ValueError(f"Unknown v26 structure {structure!r}")


def _bt_push(st, book, m, asof, scen, real):
    scen = np.asarray(scen, dtype=float)
    var95 = float(-np.quantile(scen, 0.05))
    bt_book_records.setdefault((st, book, m), []).append({
        "asof": asof, "VaR_95": var95, "ES_95": sample_expected_shortfall(scen, 0.05),
        "realized_return": float(real), "violation": int(real < -var95),
        "pit": float((scen <= real).mean())})
    bt_book_scen.setdefault((st, book, m), []).append(scen)


def _bt_record_books(st, m, eff, sim_bt, port_sim, realized, eff_pos, fwd_pos, h, kr, cr):
    """Every listed book, priced identically on simulated and realized horizon moves."""
    _bt_push(st, "equity_ew", m, eff, port_sim, realized)
    if len(_bt_books) == 1:
        return
    tau = h / 252.0
    fwd_date = stock_prices_raw.index[fwd_pos]
    y0 = float(kr.loc[eff, "DGS10"])
    y1 = float(rates_raw["DGS10"].loc[:fwd_date].dropna().iloc[-1])
    s0 = float(cr.loc[eff, _bt_spread_col])
    s1 = float(credit_driver_raw[_bt_spread_col].loc[:fwd_date].dropna().iloc[-1])
    dy_sim = sim_bt["key_rate_scenarios"]["DGS10"].to_numpy(dtype=float) - y0
    ds_sim = sim_bt["credit_spread_scenarios"][_bt_spread_col].to_numpy(dtype=float) - s0
    ust_sim = par_bond_horizon_return(y0, dy_sim, 10.0, tau)
    ust_real = float(par_bond_horizon_return(y0, y1 - y0, 10.0, tau))
    corp_sim = par_bond_horizon_return(y0 + s0, dy_sim + ds_sim, 10.0, tau)
    corp_real = float(par_bond_horizon_return(y0 + s0, (y1 - y0) + (s1 - s0), 10.0, tau))
    eq_sim = np.expm1(sim_bt["stock_log_return_scenarios"][TICKERS].to_numpy(dtype=float))
    eq_real = (stock_prices_raw[TICKERS].iloc[fwd_pos].to_numpy(dtype=float)
               / stock_prices_raw[TICKERS].iloc[eff_pos].to_numpy(dtype=float) - 1.0)
    eqw_sim, eqw_real = eq_sim.mean(axis=1), float(eq_real.mean())
    bt_stock_bond.setdefault((st, m), []).append(
        {"asof": eff, "pred_corr": float(np.corrcoef(eqw_sim, ust_sim)[0, 1]),
         "eq_real": eqw_real, "ust_real": ust_real})
    if "stock_bond_40_60" in _bt_books:
        _bt_push(st, "stock_bond_40_60", m, eff,
                 0.4 * eqw_sim + 0.6 * ust_sim, 0.4 * eqw_real + 0.6 * ust_real)
    if "credit_40_20_40" in _bt_books:
        _bt_push(st, "credit_40_20_40", m, eff,
                 0.4 * eqw_sim + 0.2 * ust_sim + 0.4 * corp_sim,
                 0.4 * eqw_real + 0.2 * ust_real + 0.4 * corp_real)
    if "min_cvar" in _bt_books:
        R = np.column_stack([eq_sim, ust_sim, corp_sim])
        try:
            if not np.all(np.isfinite(R)):
                raise ValueError(f"nonfinite scenario return matrix: {np.size(R) - np.isfinite(R).sum()} bad cells")
            w_, _ = search_long_only_min_cvar_portfolio(
                pd.DataFrame(R[0::2]), alpha=CVAR_ALPHA, max_weight=CVAR_MAX_WEIGHT)
            w_ = w_.to_numpy(dtype=float)
            _bt_push(st, "min_cvar", m, eff, R[1::2] @ w_,
                     float(np.r_[eq_real, ust_real, corp_real] @ w_))
            bt_mincvar_weights.setdefault((st, m), []).append(
                {"asof": eff, "equities": float(w_[:len(TICKERS)].sum()),
                 "UST_10Y": float(w_[-2]), "corporate": float(w_[-1])})
        except Exception as _err:
            bt_mincvar_failures.append(
                {"structure": st, "horizon": m, "asof": pd.Timestamp(eff),
                 "error_type": type(_err).__name__, "error": str(_err),
                 "max_abs_return": float(np.nanmax(np.abs(R))) if np.isfinite(R).any() else np.nan,
                 "nonfinite_cells": int(np.size(R) - np.isfinite(R).sum())})


def _trailing_annualized_vol(eff_pos, lookback=21):
    lo = max(0, eff_pos - lookback)
    px = stock_prices_raw[TICKERS].iloc[lo:eff_pos + 1]
    if len(px) < 6:
        return np.nan
    r = np.log(px / px.shift(1)).mean(axis=1).dropna()
    return float(r.std() * np.sqrt(252.0)) if len(r) >= 5 else np.nan


# ---- v26 architecture lab --------------------------------------------------------------------------
_bt_primary = V26_PRIMARY_STRUCTURE
_bt_structs = list(V26_STRUCTURES)
if _bt_primary not in _bt_structs:
    raise ValueError("V26_PRIMARY_STRUCTURE must be one of V26_STRUCTURES")
_bt_books = list(dict.fromkeys(BACKTEST_PORTFOLIOS))
_bt_extra = True
_bt_split = pd.Timestamp(BACKTEST_REGIME_SPLIT)
_bt_spread_col = ("Baa_spr" if "Baa_spr" in credit_driver_raw.columns else
                  REPRESENTATIVE_RATING if REPRESENTATIVE_RATING in credit_driver_raw.columns
                  else credit_driver_raw.columns[-1])
bt_book_records, bt_book_scen, bt_stock_bond, bt_mincvar_weights, bt_k_global = {}, {}, {}, {}, {}
bt_mincvar_failures, bt_halflife_records, bt_geometry_records = [], [], []

# Horizons and non-overlapping outer test grids: unchanged from v25c.
_bt_modes = [HORIZON_MODE] if BACKTEST_HORIZONS is None else list(dict.fromkeys(BACKTEST_HORIZONS))
for _m in _bt_modes:
    if _m not in HORIZON_BDAYS_BY_MODE:
        raise ValueError(f"Unknown horizon {_m!r} in BACKTEST_HORIZONS")
if HORIZON_MODE not in _bt_modes:
    raise ValueError("BACKTEST_HORIZONS must include HORIZON_MODE")
_bt_h = {m: HORIZON_BDAYS_BY_MODE[m] for m in _bt_modes}
_h_min = min(_bt_h.values())
_base_step = BACKTEST_STEP_BDAYS if BACKTEST_HORIZONS is None else max(5, _h_min)
_cal = stock_prices_raw.index
_elig = np.where(np.asarray(_cal <= AS_OF_DATE))[0]
_elig = _elig[_elig + _h_min < len(_cal)]
_thinned_by, _n_floor_skipped = 1, 0
_aligned = stock_prices.index
_need_rows = int(BACKTEST_MIN_TRAIN_DAYS) if BACKTEST_MODE == "full" else int(BACKTEST_MIN_REFIT_DAYS)
_backtest_skipped = _need_rows >= len(_aligned)
if _backtest_skipped:
    print(f"BACKTEST SKIPPED: need {_need_rows} training rows, have {len(_aligned)}.")
    _sel_positions = []
elif BACKTEST_MODE == "full":
    _first_ok = _aligned[min(int(BACKTEST_MIN_TRAIN_DAYS), len(_aligned) - 1)]
    _cand = [p for p in _elig if _cal[p] >= _first_ok][::_base_step]
    if BACKTEST_MAX_DATES is not None and len(_cand) > int(BACKTEST_MAX_DATES):
        _thinned_by = int(np.ceil(len(_cand) / float(BACKTEST_MAX_DATES)))
        _cand = _cand[::_thinned_by]
    _sel_positions = sorted(_cand)
else:
    _floor = _aligned[min(int(BACKTEST_MIN_REFIT_DAYS), len(_aligned) - 1)]
    _wanted = _elig[::-1][::_base_step][:BACKTEST_N_DATES]
    _elig_r = _elig[np.asarray(_cal[_elig] >= _floor)]
    _sel_positions = sorted(list(_elig_r[::-1][::_base_step][:BACKTEST_N_DATES]))
    _n_floor_skipped = int(len(_wanted) - len(_sel_positions))
backtest_asof_dates = [_cal[p] for p in _sel_positions]
_grid_spacing = _base_step * _thinned_by
_sub_grid, _sub_k = {}, {}
for _m, _h in _bt_h.items():
    _sub_k[_m] = max(1, int(np.ceil(_h / float(_grid_spacing))))
    _ok = [p for p in _sel_positions if p + _h < len(_cal)]
    _sub_grid[_m] = set(_ok[::-1][::_sub_k[_m]])

print(f"v26 fair architecture lab [{BACKTEST_MODE}]: {_bt_structs}")
print("Static vs adaptive block geometry crossed with blockwise vs hierarchical dependence.")
print(f"Half-life mode: {V26_HALFLIFE_MODE!r}. Adaptive models ALWAYS share the same H at a given as-of/horizon.")
if backtest_asof_dates:
    print(f"{len(backtest_asof_dates)} outer as-of dates, {backtest_asof_dates[0].date()} -> "
          f"{backtest_asof_dates[-1].date()}, {BACKTEST_N_SCENARIOS} scenarios per structure/horizon.")
if BACKTEST_HORIZONS is not None:
    print("Outer non-overlapping windows: " +
          "; ".join(f"{m}: {len(_sub_grid[m])}" for m in _bt_modes))
print()

_candidate_days = tuple(None if x is None else float(x) * 252.0 for x in V26_HALFLIFE_GRID_YEARS)
_fixed_days_by_h = {
    m: (None if V26_FIXED_HALFLIFE_YEARS_BY_HORIZON.get(m) is None
        else float(V26_FIXED_HALFLIFE_YEARS_BY_HORIZON[m]) * 252.0)
    for m in _bt_modes
}

bt_records_by_horizon = {m: [] for m in _bt_modes}
bt_scenarios_by_horizon = {m: [] for m in _bt_modes}

for j, asof in enumerate(backtest_asof_dates):
    _active = [m for m in _bt_modes if _sel_positions[j] in _sub_grid[m]]
    if not _active:
        continue
    try:
        raw = _v26_prepare_asof(asof)
        eff, sp, kr, cr = raw["eff"], raw["sp"], raw["kr"], raw["cr"]
        static_engine = _v26_fit_engine(raw, None)
        # OU is model-independent and refit on exactly the same point-in-time raw window.
        bt_ou_r = (estimate_ou_level_params(kr.to_numpy(), tstat_gate=OU_TSTAT_GATE)
                   if LEVEL_MEAN_REVERSION == "ou" else None)
        bt_ou_c = (estimate_ou_level_params(cr.to_numpy(), tstat_gate=OU_TSTAT_GATE)
                   if (LEVEL_MEAN_REVERSION == "ou" and cr is not None and cr.shape[1] > 0) else None)
        eff_pos = stock_prices_raw.index.get_loc(eff)
        _tv21 = _trailing_annualized_vol(eff_pos)
        calib_panel = _v26_model_agnostic_panel(raw["lr"], raw["dy"], raw["ds"])
        adaptive_cache = {}
        setup_cache = {}

        for _m in _active:
            _h = _bt_h[_m]
            if V26_HALFLIFE_MODE == "shared_fixed":
                _hl = _fixed_days_by_h[_m]
                _sel = {"halflife_days": _hl, "n_origins": 0, "fallback": False}
            elif V26_HALFLIFE_MODE == "shared_nested":
                _sel = v26_select_shared_halflife(
                    calib_panel, _h, _candidate_days,
                    fallback_days=(V26_FALLBACK_HALFLIFE_YEARS * 252.0),
                    min_train_days=V26_HALFLIFE_MIN_TRAIN_DAYS,
                    max_origins=V26_HALFLIFE_MAX_INNER_ORIGINS,
                    one_se=V26_HALFLIFE_ONE_SE_RULE)
                _hl = _sel["halflife_days"]
            else:
                raise ValueError("V26_HALFLIFE_MODE must be 'shared_fixed' or 'shared_nested'")

            bt_halflife_records.append({
                "asof": eff, "horizon": _m,
                "halflife_days": np.inf if _hl is None else float(_hl),
                "halflife_years": np.inf if _hl is None else float(_hl)/252.0,
                "label": _v26_halflife_label(_hl),
                "n_inner_origins": int(_sel.get("n_origins", 0)),
                "fallback": bool(_sel.get("fallback", False)),
            })

            _akey = "static" if _hl is None else round(float(_hl), 8)
            if _akey not in adaptive_cache:
                adaptive_cache[_akey] = _v26_fit_engine(raw, _hl)
            adaptive_engine = adaptive_cache[_akey]

            # How much does the current-regime basis actually rotate relative to the static block basis?
            bt_geometry_records.append({
                "asof": eff, "horizon": _m,
                "halflife_years": np.inf if _hl is None else float(_hl)/252.0,
                "rates_angle_deg": v26_subspace_angle_deg(
                    static_engine["rb"]["loadings"], adaptive_engine["rb"]["loadings"]),
                "credit_angle_deg": v26_subspace_angle_deg(
                    static_engine["cb"]["loadings"], adaptive_engine["cb"]["loadings"]),
                "equity_resid_angle_deg": v26_subspace_angle_deg(
                    static_engine["eb"]["resid_loadings"], adaptive_engine["eb"]["resid_loadings"]),
            })

            fwd_pos = eff_pos + _h
            if fwd_pos >= len(stock_prices_raw.index):
                continue
            p0 = stock_prices_raw[TICKERS].iloc[eff_pos]
            p1 = stock_prices_raw[TICKERS].iloc[fwd_pos]
            realized = float(np.log(p1 / p0).mean())

            # Structure setups are horizon-specific only through the shared adaptive half-life.
            specs = {}
            for _st in _bt_structs:
                _ck = (_st, _akey)
                if _ck not in setup_cache:
                    setup_cache[_ck] = _v26_structure_spec(
                        _st, static_engine, adaptive_engine, _hl)
                specs[_st] = setup_cache[_ck]
                if np.isfinite(specs[_st].get("k_global", np.nan)):
                    bt_k_global.setdefault(_st, []).append(specs[_st]["k_global"])

            for _st in _bt_structs:
                _k = specs[_st]
                eng = _k["engine"]
                drv, vr = eng["drv"], eng["vr"]
                sim_bt = simulate_joint_horizon(
                    var_res=vr, resid_hist=vr.resid, last_state=drv.iloc[-1].to_numpy(),
                    equity_block=eng["eb"], rates_block=eng["rb"],
                    current_prices=sp.loc[eff], current_key_rates=kr.loc[eff],
                    horizon_bdays=_h, n_scenarios=BACKTEST_N_SCENARIOS,
                    avg_block_length=effective_block_length,
                    rng_seed=RNG_SEED + j + 7919 * _bt_modes.index(_m),
                    daily_rf=float(raw["rf_d"].iloc[-1]),
                    resid_idio_std=eng["eb"].get("resid_idiosyncratic_std"),
                    credit_block=eng["cb"], current_spreads=cr.loc[eff],
                    dynamics=DRIVER_DYNAMICS, demeaned_drivers=_k["dd"], driver_mean=eng["mean"],
                    crisis_mode=CRISIS_MODE, crisis_prob=crisis_prob_eff, crisis_scale=crisis_scale_eff,
                    crisis_pool=crisis_pool_eff, copula=None, asset_copula=None, fhs_filter=_k["fhs"],
                    ou_rates=bt_ou_r, ou_credit=bt_ou_c, hier=_k["hier"], v25_structure=None)
                port_sim = sim_bt["stock_log_return_scenarios"][TICKERS].mean(axis=1)
                var95 = float(-port_sim.quantile(0.05))
                es95 = sample_expected_shortfall(port_sim.to_numpy(), 0.05)
                if _st == _bt_primary:
                    bt_records_by_horizon[_m].append({
                        "asof": eff, "VaR_95": var95, "ES_95": es95,
                        "realized_return": realized, "violation": int(realized < -var95),
                        "pit": float((port_sim <= realized).mean()),
                        "trailing_vol_21d": _tv21})
                    bt_scenarios_by_horizon[_m].append(port_sim.to_numpy(dtype=float))
                try:
                    _bt_record_books(_st, _m, eff, sim_bt, port_sim.to_numpy(dtype=float),
                                     realized, eff_pos, fwd_pos, _h, kr, cr)
                except Exception as _err:
                    print(f"  [books {_st}/{_m} skipped at {pd.Timestamp(asof).date()}] "
                          f"{type(_err).__name__}: {_err}")
    except Exception as err:
        print(f"  [skip {pd.Timestamp(asof).date()}] {type(err).__name__}: {err}")
        continue
    if (j + 1) % max(10, len(backtest_asof_dates) // 10) == 0:
        print(f"  ...processed {j + 1}/{len(backtest_asof_dates)} as-of dates")

# Half-life and geometry diagnostics are research outputs, not tuning on the final OOS observations.
if bt_halflife_records:
    v26_halflife_history = pd.DataFrame(bt_halflife_records)
    _hl_show = (v26_halflife_history.groupby("horizon", sort=False)
                .agg(n=("halflife_years", "size"),
                     median_half_life_y=("halflife_years", "median"),
                     p25_half_life_y=("halflife_years", lambda x: np.nanpercentile(x.replace(np.inf, np.nan).dropna(), 25)
                                      if np.isfinite(x.replace(np.inf, np.nan)).any() else np.inf),
                     p75_half_life_y=("halflife_years", lambda x: np.nanpercentile(x.replace(np.inf, np.nan).dropna(), 75)
                                      if np.isfinite(x.replace(np.inf, np.nan)).any() else np.inf),
                     fallback_rate=("fallback", "mean")))
    print("\nv26 selected common half-life by horizon (shared by blockwise-adaptive and hierarchical-adaptive):")
    display(_hl_show.round(3))
if bt_geometry_records:
    v26_geometry_history = pd.DataFrame(bt_geometry_records)
    print("\nv26 current-regime basis rotation vs static basis (principal angles, degrees):")
    display(v26_geometry_history.groupby("horizon", sort=False)[
        ["rates_angle_deg", "credit_angle_deg", "equity_resid_angle_deg"]].median().round(2))

# Headline frame = HORIZON_MODE (the only one unless BACKTEST_HORIZONS is set).
bt = pd.DataFrame(bt_records_by_horizon[HORIZON_MODE])
bt_scenario_matrix = (np.vstack(bt_scenarios_by_horizon[HORIZON_MODE])
                      if bt_scenarios_by_horizon[HORIZON_MODE] else None)
print(f"\nUsable backtest observations: {len(bt)}")

if len(bt) >= 10:
    _cov = coverage_summary(bt["violation"].to_numpy(), 0.05)

    # ---- Expected-shortfall backtest (Acerbi-Szekely 2014) -------------------------------------
    # The portfolio layer MINIMIZES CVaR, so validating only the VaR left the objective function
    # itself untested. Its own RNG stream, so the engine's scenarios are never perturbed.
    _rng_es = np.random.default_rng(RNG_SEED + 4242)
    _es_args = (bt["realized_return"].to_numpy(), bt["VaR_95"].to_numpy(),
                bt["ES_95"].to_numpy(), bt_scenario_matrix)
    es_test2 = acerbi_szekely_es_test(*_es_args, alpha=0.05, n_sim=ES_BACKTEST_N_SIM,
                                      rng=_rng_es, which=2)
    es_test1 = acerbi_szekely_es_test(*_es_args, alpha=0.05, n_sim=ES_BACKTEST_N_SIM,
                                      rng=np.random.default_rng(RNG_SEED + 4243), which=1)

    summary = pd.Series({
        "observations": _cov["observations"],
        "violations": _cov["violations"],
        "expected_violations_at_5pct": _cov["expected"],
        "empirical_violation_rate": _cov["violation_rate"],
        "kupiec_POF_LR": _cov["kupiec_LR"],
        "kupiec_POF_pvalue": _cov["kupiec_p"],
        "christoffersen_indep_LR": _cov["indep_LR"],
        "christoffersen_indep_pvalue": _cov["indep_p"],
        "christoffersen_cc_LR": _cov["cc_LR"],
        "christoffersen_cc_pvalue": _cov["cc_p"],
        "mean_VaR_95": float(bt["VaR_95"].mean()),
        "mean_ES_95": float(bt["ES_95"].mean()),
        "acerbi_szekely_Z2": es_test2["statistic"],
        "acerbi_szekely_Z2_pvalue": es_test2["p_value"],
        "acerbi_szekely_Z1": es_test1["statistic"],
        "acerbi_szekely_Z1_pvalue": es_test1["p_value"],
    }, name="walk_forward_VaR_ES_backtest")
    print(f"\nVaR/ES backtest (95%, equal-weight equity, {HORIZON_LABEL} horizon):")
    display(summary.to_frame("value"))
    print(
        "Kupiec p tests unconditional coverage (H0: violation rate = 5%); Christoffersen indep p "
        "tests no clustering of breaches; the JOINT conditional-coverage p (chi2(2), LR_cc = LR_uc "
        "+ LR_ind) tests both at once -- a VaR can pass each separately and still fail the joint "
        "null. p-values above 0.05 do not reject."
    )
    print(
        f"Acerbi-Szekely ES test: Z2 = {es_test2['statistic']:.4f} (p = {es_test2['p_value']:.3f}). "
        "E[Z] = 0 under a correct predictive distribution. Z < 0 means realized tail losses exceeded "
        "the model's own ES (ES understated -- the direction that matters); Z > 0 means the ES was "
        "conservative. The null is simulated by resampling each date's own predictive scenario set, "
        "so no distributional assumption is imposed. Z1 conditions on the breaches (severity GIVEN a "
        "breach) and is undefined when there are none; Z2 is defined either way."
    )

    # ---- Coverage by volatility regime ---------------------------------------------------------
    # The long sample's real payoff: a VaR that covers on average but fails in stress is a broken
    # VaR, and a recent-calm-window backtest can never see it. Tercile cut-points come from the
    # realized panel -- a DIAGNOSTIC partition, declared: it cannot flatter the VaR, because every
    # VaR was formed at its own as-of date without them.
    if bt["trailing_vol_21d"].notna().sum() >= 15:
        _tv = bt["trailing_vol_21d"]
        _c1, _c2 = float(_tv.quantile(1 / 3)), float(_tv.quantile(2 / 3))
        bt["vol_regime"] = pd.cut(_tv, [-np.inf, _c1, _c2, np.inf],
                                  labels=["calm", "normal", "stressed"])
        _rows = []
        for _lab in ["calm", "normal", "stressed"]:
            _msk = (bt["vol_regime"] == _lab).to_numpy()
            if _msk.sum() == 0:
                continue
            _c = coverage_summary(bt.loc[_msk, "violation"].to_numpy(), 0.05)
            _row = {"regime": _lab, "n": _c["observations"], "violations": _c["violations"],
                    "expected": _c["expected"], "violation_rate": _c["violation_rate"],
                    "kupiec_p": _c["kupiec_p"],
                    "mean_trailing_vol": float(bt.loc[_msk, "trailing_vol_21d"].mean()),
                    "mean_VaR_95": float(bt.loc[_msk, "VaR_95"].mean()),
                    "mean_ES_95": float(bt.loc[_msk, "ES_95"].mean())}
            if _msk.sum() >= 20 and bt_scenario_matrix is not None:
                _z = acerbi_szekely_es_test(bt.loc[_msk, "realized_return"].to_numpy(),
                                            bt.loc[_msk, "VaR_95"].to_numpy(),
                                            bt.loc[_msk, "ES_95"].to_numpy(),
                                            bt_scenario_matrix[_msk], alpha=0.05,
                                            n_sim=ES_BACKTEST_N_SIM,
                                            rng=np.random.default_rng(RNG_SEED + 991), which=2)
                _row["ES_Z2"] = _z["statistic"]
                _row["ES_Z2_p"] = _z["p_value"]
            _rows.append(_row)
        regime_coverage = pd.DataFrame(_rows).set_index("regime")
        print(f"\nCoverage by trailing-volatility regime (terciles at {_c1:.1%} / {_c2:.1%} "
              "annualized 21-day realized vol, each known at its as-of date):")
        display(regime_coverage.round(4))
        print("A well-calibrated engine violates at ~5% in EVERY regime. Systematically too few "
              "breaches in 'calm' and too many in 'stressed' is the classic unconditional-model "
              "signature -- exactly what the conditional-volatility (FHS) filter exists to remove.")

    # ---- Coverage by calendar year (long backtests only) ----------------------------------------
    _years = bt["asof"].dt.year
    if _years.nunique() >= 3:
        _rows = []
        for _y, _g in bt.groupby(_years):
            _c = coverage_summary(_g["violation"].to_numpy(), 0.05)
            _rows.append({"year": int(_y), "n": _c["observations"], "violations": _c["violations"],
                          "expected": round(_c["expected"], 2), "violation_rate": _c["violation_rate"],
                          "mean_VaR_95": float(_g["VaR_95"].mean()),
                          "worst_loss": float(-_g["realized_return"].min())})
        year_coverage = pd.DataFrame(_rows).set_index("year")
        print("\nCoverage by calendar year (the regimes the long sample bought: crises are visible here):")
        display(year_coverage.round(4))
        if _grid_spacing > HORIZON_BDAYS:
            print(f"Sampling note: one {HORIZON_BDAYS}-day window every {_grid_spacing} business days, so the "
                  f"windows cover about {HORIZON_BDAYS / _grid_spacing:.0%} of the span; a crisis week that falls "
                  "between two samples is not tested (a worst loss that looks mild in a crisis year is the sign). "
                  "Longer horizons tile the span far more completely -- see the per-horizon table below.")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].hist(bt["pit"], bins=10, range=(0, 1), edgecolor="white", alpha=0.85)
    axes[0].axhline(len(bt) / 10.0, color="black", lw=1, ls="--", label="uniform")
    axes[0].set_title("PIT / rank histogram (should be ~uniform)")
    axes[0].set_xlabel("PIT of realized return")
    axes[0].legend()
    axes[1].plot(bt["asof"], bt["VaR_95"], label="VaR 95% (loss)", lw=1.3)
    axes[1].plot(bt["asof"], bt["ES_95"], label="ES 95% (loss)", lw=1.0, ls="--", alpha=0.8)
    axes[1].plot(bt["asof"], -bt["realized_return"], label="realized loss", lw=0.9, alpha=0.8)
    viol = bt[bt["violation"] == 1]
    axes[1].scatter(viol["asof"], -viol["realized_return"], color="red", s=25, zorder=5, label="violation")
    axes[1].set_title(f"VaR / ES vs realized {HORIZON_LABEL} loss over time")
    axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print("Too few usable observations for a meaningful backtest; increase BACKTEST_N_DATES "
          "or widen the data window (at 3M / 1Y: SAMPLE_MODE='long' with BACKTEST_MODE='full').")

# ---- Coverage as a function of the horizon (BACKTEST_HORIZONS) ------------------------------------
# The 1Y engine cannot be validated on its own (about 20 independent windows in 24 years: a VaR
# breached twice as often as it should passes Kupiec 87 times in 100). 1M and 3M can, and they share
# every refit with it: if coverage holds from 1W through 3M, 1Y is one step beyond what was tested; if
# it drifts with the horizon, the drift says in which direction the long-horizon dynamics are wrong.
if BACKTEST_HORIZONS is not None:
    _rows = []
    _ref = min(_bt_modes, key=lambda m: _bt_h[m])
    _ref_df = pd.DataFrame(bt_records_by_horizon[_ref])
    for _hi, _m in enumerate(_bt_modes):
        _d = pd.DataFrame(bt_records_by_horizon[_m])
        _row = {"horizon": _m, "bdays": _bt_h[_m], "n": len(_d)}
        if len(_d) >= 10:
            _c = coverage_summary(_d["violation"].to_numpy(), 0.05)
            _z = acerbi_szekely_es_test(_d["realized_return"].to_numpy(), _d["VaR_95"].to_numpy(),
                                        _d["ES_95"].to_numpy(), np.vstack(bt_scenarios_by_horizon[_m]),
                                        alpha=0.05, n_sim=ES_BACKTEST_N_SIM,
                                        rng=np.random.default_rng(RNG_SEED + 5300 + _hi), which=2)
            _row.update({"violations": _c["violations"], "expected": _c["expected"],
                         "violation_rate": _c["violation_rate"], "kupiec_p": _c["kupiec_p"],
                         "indep_p": _c["indep_p"], "cc_p": _c["cc_p"],
                         "mean_VaR_95": float(_d["VaR_95"].mean()), "mean_ES_95": float(_d["ES_95"].mean()),
                         "ES_Z2": _z["statistic"], "ES_Z2_p": _z["p_value"]})
        if len(_d) and len(_ref_df) and _m != _ref:
            _mg = _d[["asof", "VaR_95"]].merge(_ref_df[["asof", "VaR_95"]], on="asof", suffixes=("", "_ref"))
            if len(_mg):
                _row[f"VaR_vs_sqrt_t_{_ref}"] = float(np.median(
                    _mg["VaR_95"] / (np.sqrt(_bt_h[_m] / float(_bt_h[_ref])) * _mg["VaR_95_ref"])))
        _rows.append(_row)
    horizon_coverage = pd.DataFrame(_rows).set_index("horizon")
    # v23: how much of the backtest span each horizon's windows actually cover (a thinned 1W grid samples
    # one week in several and can miss a crash week; 1M/3M/1Y windows tile most of the span).
    if backtest_asof_dates:
        _span = max(1, _sel_positions[-1] - _sel_positions[0] + _h_min)
        horizon_coverage["share_of_span_covered"] = [
            min(1.0, len(bt_records_by_horizon[m]) * _bt_h[m] / float(_span)) for m in horizon_coverage.index]
    print("\nCoverage by horizon (same refits; each horizon on its own non-overlapping windows):")
    display(horizon_coverage.round(4))
    # v23: coverage by calendar year AT EVERY horizon, so the crisis years can be read where the windows
    # actually cover them (violations / windows, and the worst realized loss in that year's windows).
    _yr = {}
    for _m in _bt_modes:
        _d = pd.DataFrame(bt_records_by_horizon[_m])
        if len(_d) == 0:
            continue
        _g = _d.groupby(_d["asof"].dt.year)
        _yr[(_m, "n")] = _g.size()
        _yr[(_m, "viol")] = _g["violation"].sum()
        _yr[(_m, "worst_loss")] = -_g["realized_return"].min()
    if _yr:
        horizon_year_coverage = pd.DataFrame(_yr).rename_axis("year")
        print("\nCoverage by calendar year at every horizon (windows starting in that year: count, violations, "
              "worst realized loss):")
        display(horizon_year_coverage.round(4))
    print(f"A calibrated engine violates at ~5% at EVERY horizon. VaR_vs_sqrt_t_{_ref} is the median, over "
          f"common as-of dates, of the direct h-day VaR over the sqrt(t)-scaled {_ref} VaR: the engine's "
          "own term structure, which the coverage columns test against realized returns. Fewer than 10 "
          "windows (typically 1Y) leaves the tests blank: that horizon is reported, not validated.")

# ---- v24: multi-asset books and structures, same refits ------------------------------------------------
# Each row: one structure x one book x one horizon, on that horizon's non-overlapping windows. The
# equity_ew rows of the headline structure repeat the tables above (a consistency check).
if _bt_extra and bt_book_records:
    _order = [(s_, b_, m_) for s_ in _bt_structs for b_ in _bt_books for m_ in _bt_modes if (s_, b_, m_) in bt_book_records]
    _rows, _prow = [], []
    for _i, (_s, _b, _m) in enumerate(_order):
        _d = pd.DataFrame(bt_book_records[(_s, _b, _m)])
        _row = {"structure": _s, "book": _b, "horizon": _m, "n": len(_d)}
        if len(_d) >= 10:
            _c = coverage_summary(_d["violation"].to_numpy(), 0.05)
            _z = acerbi_szekely_es_test(_d["realized_return"].to_numpy(), _d["VaR_95"].to_numpy(),
                                        _d["ES_95"].to_numpy(), np.vstack(bt_book_scen[(_s, _b, _m)]),
                                        alpha=0.05, n_sim=ES_BACKTEST_N_SIM,
                                        rng=np.random.default_rng(RNG_SEED + 6100 + _i), which=2)
            _row.update({"violations": _c["violations"], "violation_rate": _c["violation_rate"],
                         "kupiec_p": _c["kupiec_p"], "cc_p": _c["cc_p"], "mean_VaR_95": float(_d["VaR_95"].mean()),
                         "mean_ES_95": float(_d["ES_95"].mean()), "ES_Z2": _z["statistic"], "ES_Z2_p": _z["p_value"]})
        _rows.append(_row)
        for _plab, _msk in ((f"before {_bt_split.date()}", _d["asof"] < _bt_split), (f"from {_bt_split.date()}", _d["asof"] >= _bt_split)):
            _dd = _d[_msk.to_numpy()]
            if len(_dd) == 0:
                continue
            _c = coverage_summary(_dd["violation"].to_numpy(), 0.05)
            _prow.append({"structure": _s, "book": _b, "horizon": _m, "period": _plab, "n": len(_dd),
                          "violations": _c["violations"], "violation_rate": _c["violation_rate"],
                          "kupiec_p": _c["kupiec_p"] if len(_dd) >= 10 else np.nan,
                          "mean_VaR_95": float(_dd["VaR_95"].mean()), "worst_loss": float(-_dd["realized_return"].min())})
    book_coverage = pd.DataFrame(_rows).set_index(["structure", "book", "horizon"])
    book_coverage_by_period = pd.DataFrame(_prow).set_index(["structure", "book", "horizon", "period"])
    print("\nMulti-asset books (v26 fair architecture lab): 95% VaR/ES coverage by structure, book and horizon (same refits, dates and seeds "
          "across structures; books priced identically on simulated and realized moves):")
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 250):
        display(book_coverage.round(4))
    book_coverage.to_csv("v26_book_coverage.csv")

    # v26 factorial attribution on the simplest common metric: distance of the empirical VaR breach
    # rate from the nominal 5%. Positive "gain" means the second model is closer to 5%.
    _bc = book_coverage.reset_index()
    if set(V26_STRUCTURES).issubset(set(_bc["structure"].unique())):
        _fa = []
        for (_b, _m), _g in _bc.groupby(["book", "horizon"], sort=False):
            if not {"blockwise_static", "blockwise_adaptive",
                    "hierarchical_static", "hierarchical_adaptive"}.issubset(set(_g["structure"])):
                continue
            _g = _g.set_index("structure")
            if _g["violation_rate"].isna().any():
                continue
            _gap = (_g["violation_rate"] - 0.05).abs()
            _fa.append({
                "book": _b, "horizon": _m,
                "blockwise_static_gap": float(_gap["blockwise_static"]),
                "blockwise_adaptive_gap": float(_gap["blockwise_adaptive"]),
                "hierarchical_static_gap": float(_gap["hierarchical_static"]),
                "hierarchical_adaptive_gap": float(_gap["hierarchical_adaptive"]),
                "regime_gain_blockwise": float(_gap["blockwise_static"] - _gap["blockwise_adaptive"]),
                "hierarchy_gain_static": float(_gap["blockwise_static"] - _gap["hierarchical_static"]),
                "regime_gain_hierarchical": float(_gap["hierarchical_static"] - _gap["hierarchical_adaptive"]),
                "hierarchy_gain_adaptive": float(_gap["blockwise_adaptive"] - _gap["hierarchical_adaptive"]),
            })
        if _fa:
            v26_factorial_var_attribution = pd.DataFrame(_fa).set_index(["book", "horizon"])
            print("\nv26 factorial VaR-calibration attribution (positive gain = closer to the nominal 5% breach rate):")
            display(v26_factorial_var_attribution.round(4))
            v26_factorial_var_attribution.to_csv("v26_factorial_var_attribution.csv")
    print(f"\nThe same books before / from {_bt_split.date()} (BACKTEST_REGIME_SPLIT): the stock-yield sign flip the v23 "
          "run measured falls in the second period.")
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 250):
        display(book_coverage_by_period.round(4))
    book_coverage_by_period.to_csv("v26_book_coverage_by_period.csv")
    if bt_stock_bond:
        _sb = []
        for _s in _bt_structs:
            for _m in _bt_modes:
                if (_s, _m) not in bt_stock_bond:
                    continue
                _d = pd.DataFrame(bt_stock_bond[(_s, _m)])
                for _plab, _msk in (("whole span", np.ones(len(_d), dtype=bool)),
                                    (f"before {_bt_split.date()}", (_d["asof"] < _bt_split).to_numpy()),
                                    (f"from {_bt_split.date()}", (_d["asof"] >= _bt_split).to_numpy())):
                    _dd = _d[_msk]
                    if len(_dd) < 8:
                        continue
                    _sb.append({"structure": _s, "horizon": _m, "period": _plab, "n": len(_dd),
                                "predicted_corr_mean": float(_dd["pred_corr"].mean()),
                                "realized_corr": float(np.corrcoef(_dd["eq_real"], _dd["ust_real"])[0, 1])})
        if _sb:
            stock_bond_check = pd.DataFrame(_sb).set_index(["structure", "horizon", "period"])
            print("\nStock-bond correlation, equal-weight equity vs the 10Y Treasury return: the engine's prediction at each "
                  "as-of date (scenario correlation, averaged) against the realized correlation across those windows. "
                  "This is the quantity a 40/60 hedge lives on:")
            with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 250):
                display(stock_bond_check.round(3))
            stock_bond_check.to_csv("v26_stock_bond_check.csv")

            _whole = stock_bond_check.reset_index()
            _whole = _whole[_whole["period"] == "whole span"].copy()
            if len(_whole):
                _whole["corr_abs_error"] = (_whole["predicted_corr_mean"] - _whole["realized_corr"]).abs()
                _depfa = []
                for _m, _g in _whole.groupby("horizon", sort=False):
                    _g = _g.set_index("structure")
                    req = {"blockwise_static", "blockwise_adaptive",
                           "hierarchical_static", "hierarchical_adaptive"}
                    if not req.issubset(set(_g.index)):
                        continue
                    e = _g["corr_abs_error"]
                    _depfa.append({
                        "horizon": _m,
                        "blockwise_static_abs_error": float(e["blockwise_static"]),
                        "blockwise_adaptive_abs_error": float(e["blockwise_adaptive"]),
                        "hierarchical_static_abs_error": float(e["hierarchical_static"]),
                        "hierarchical_adaptive_abs_error": float(e["hierarchical_adaptive"]),
                        "regime_gain_blockwise": float(e["blockwise_static"] - e["blockwise_adaptive"]),
                        "hierarchy_gain_static": float(e["blockwise_static"] - e["hierarchical_static"]),
                        "regime_gain_hierarchical": float(e["hierarchical_static"] - e["hierarchical_adaptive"]),
                        "hierarchy_gain_adaptive": float(e["blockwise_adaptive"] - e["hierarchical_adaptive"]),
                    })
                if _depfa:
                    v26_factorial_dependence_attribution = pd.DataFrame(_depfa).set_index("horizon")
                    print("\nv26 factorial stock-bond attribution (positive gain = lower absolute correlation error):")
                    display(v26_factorial_dependence_attribution.round(4))
                    v26_factorial_dependence_attribution.to_csv("v26_factorial_dependence_attribution.csv")
    # ---- v26: realized dependence by aggregation horizon ----------------------------------------
    # This is intentionally a DATA diagnostic, not a fitted model.  It asks whether the post-2022
    # stock-bond sign change exists at daily/weekly frequency or emerges only after aggregation.
    def _fisher_corr_ci(r, n, level=0.95):
        if not np.isfinite(r) or n <= 3 or abs(r) >= 1:
            return (np.nan, np.nan)
        z = np.arctanh(np.clip(r, -0.999999, 0.999999))
        # 1.959964 for 95%; kept explicit to avoid another dependency.
        q = 1.959963984540054
        se = 1.0 / np.sqrt(n - 3.0)
        return (float(np.tanh(z - q * se)), float(np.tanh(z + q * se)))

    def _nonoverlap_corr_frame(frame, h, offset=0):
        a = frame.iloc[int(offset):]
        nblk = len(a) // int(h)
        if nblk < 2:
            return np.nan, nblk
        a = a.iloc[:nblk * int(h)]
        x = a["eq"].to_numpy().reshape(nblk, int(h)).sum(axis=1)
        y = a["bond_proxy"].to_numpy().reshape(nblk, int(h)).sum(axis=1)
        return float(np.corrcoef(x, y)[0, 1]), int(nblk)

    _eq_daily = np.log(stock_prices_raw[TICKERS]).diff().mean(axis=1)
    _bond_daily = -rates_raw["DGS10"].diff()  # sign-equivalent first-order Treasury return proxy
    _dep = pd.concat([_eq_daily.rename("eq"), _bond_daily.rename("bond_proxy")], axis=1).dropna()
    _hrows = []
    for _plab, _sub in (
        ("whole span", _dep),
        (f"before {_bt_split.date()}", _dep.loc[_dep.index < _bt_split]),
        (f"from {_bt_split.date()}", _dep.loc[_dep.index >= _bt_split]),
    ):
        for _hdep in (1, 5, 21, 63):
            _r0, _n0 = _nonoverlap_corr_frame(_sub, _hdep, 0)
            _lo, _hi = _fisher_corr_ci(_r0, _n0)
            _phase = []
            for _off in range(min(_hdep, len(_sub))):
                _rr, _nn = _nonoverlap_corr_frame(_sub, _hdep, _off)
                if _nn >= 4 and np.isfinite(_rr):
                    _phase.append(_rr)
            _hrows.append({
                "period": _plab, "horizon_days": _hdep, "n_nonoverlap": _n0,
                "corr_anchor0": _r0, "fisher95_lo": _lo, "fisher95_hi": _hi,
                "phase_corr_median": float(np.median(_phase)) if _phase else np.nan,
                "phase_corr_p10": float(np.quantile(_phase, 0.10)) if _phase else np.nan,
                "phase_corr_p90": float(np.quantile(_phase, 0.90)) if _phase else np.nan,
            })
    horizon_dependence_check = pd.DataFrame(_hrows).set_index(["period", "horizon_days"])
    print("\nV26 realized horizon-dependence diagnostic: EW-equity log return vs -DGS10 change. "
          "Anchor-0 uses non-overlapping blocks and a Fisher 95% CI; phase columns summarize the same "
          "non-overlapping correlation over every possible starting offset (descriptive, not independent tests):")
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 250):
        display(horizon_dependence_check.round(3))
    horizon_dependence_check.to_csv("v26_horizon_dependence.csv")

    if bt_mincvar_failures:
        mincvar_failure_table = pd.DataFrame(bt_mincvar_failures).sort_values(
            ["structure", "horizon", "asof"]
        )
        print("\nMin-CVaR failures (isolated in v26: these do NOT remove stock-bond or policy-book records):")
        with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 250):
            display(mincvar_failure_table)
        mincvar_failure_table.to_csv("v26_mincvar_failures.csv", index=False)

    if bt_mincvar_weights:
        _wr = []
        for (_s, _m), _lst in bt_mincvar_weights.items():
            _d = pd.DataFrame(_lst)
            for _plab, _msk in ((f"before {_bt_split.date()}", _d["asof"] < _bt_split), (f"from {_bt_split.date()}", _d["asof"] >= _bt_split)):
                _dd = _d[_msk.to_numpy()]
                if len(_dd):
                    _wr.append({"structure": _s, "horizon": _m, "period": _plab, "n": len(_dd),
                                **{c_: float(_dd[c_].mean()) for c_ in ("equities", "UST_10Y", "corporate")}})
        print("\nWhat the per-date min-CVaR LP held (average weights; long-only, max weight "
              f"{CVAR_MAX_WEIGHT:.0%}, chosen on half the scenarios and evaluated on the other half):")
        display(pd.DataFrame(_wr).set_index(["structure", "horizon", "period"]).round(3))
    if bt_k_global:
        print("Global factors selected at each refit: " + "; ".join(
            f"{s_}: k_G {int(np.min(v_))}-{int(np.max(v_))} (median {np.median(v_):.0f})" for s_, v_ in bt_k_global.items()) + ".")
    print("Reading: a book breached more often than 5% in one period only, with a predicted stock-bond correlation on the "
          "wrong side of the realized one, is the regime error. A structure that fixes it must do so without losing "
          "coverage elsewhere -- the equity_ew rows guard that.")



## 4. Horizon repricing

Scenarios are mapped back into instrument values via:

- equity repricing from simulated horizon log returns,
- scenario-wise repricing of a synthetic semi-annual coupon Treasury note from the simulated key-rate curve.


In [ ]:
# Treasury note definition and current repricing
issue_date = pd.Timestamp(effective_date)
horizon_date = issue_date + BDay(HORIZON_BDAYS)
maturity_date = issue_date + pd.DateOffset(years=10)

coupon_schedule = generate_coupon_schedule(issue_date, maturity_date, freq=PAY_FREQ)

# Issue #8: the bond is priced off the full simulated curve. The tenor grid is taken from the
# active key-rate columns (ascending order), so the pricer is consistent with whatever tenors the
# rates block uses.
RATE_TENOR_YEARS = [TENOR_YEARS[t] for t in key_rates.columns]

current_bond_price = price_coupon_bond_from_curve(
    eval_date=issue_date,
    payment_dates=coupon_schedule,
    key_rate_series=current_key_rates,
    tenor_years=RATE_TENOR_YEARS,
    coupon_rate=COUPON_RATE,
    face_value=FACE_VALUE,
    freq=PAY_FREQ,
)

# Issue #10: clean/dirty decomposition. The DCF above returns the dirty (full) PV; accrued interest
# separates the clean price. At issue the note has just been struck, so accrued ~ 0.
accrued_at_issue = accrued_interest(issue_date, coupon_schedule, COUPON_RATE, FACE_VALUE, PAY_FREQ)
accrued_at_horizon = accrued_interest(horizon_date, coupon_schedule, COUPON_RATE, FACE_VALUE, PAY_FREQ)

print(f"Synthetic Treasury note dirty price at issue: {current_bond_price:,.4f}")
print(f"  accrued at issue: {accrued_at_issue:,.4f}  ->  clean: {current_bond_price - accrued_at_issue:,.4f}")
print(f"Issue date   : {issue_date.date()}")
print(f"Horizon date : {pd.Timestamp(horizon_date).date()}")
print(f"Maturity date: {maturity_date.date()}")
print(
    "Pricing convention (declared approximation): FRED CMT par yields are linearly interpolated "
    "across tenor and treated as continuously-compounded zero rates. Corporate bonds add a flat "
    "option-adjusted spread (OAS) for their rating bucket on top of that curve. P&L below is "
    "dirty-to-dirty; accrued interest is reported only for the clean/dirty split."
)

bond_horizon_prices = key_rate_scenarios.apply(
    lambda row: price_coupon_bond_from_curve(
        eval_date=horizon_date,
        payment_dates=coupon_schedule,
        key_rate_series=row,
        tenor_years=RATE_TENOR_YEARS,
        coupon_rate=COUPON_RATE,
        face_value=FACE_VALUE,
        freq=PAY_FREQ,
    ),
    axis=1,
)
bond_horizon_prices.name = "UST_10Y_2pct"
# Total-return horizon value at EVERY horizon: add the coupon cash paid inside the window. With the
# note issued on the valuation date the first coupon is six months out, so this is exactly zero at
# 1W/1M/3M and one coupon at 1Y -- but it is no longer a 1Y-only branch, and the out-of-sample lab
# below adds the same amount (up to v20 it did not, at 1Y).
_govt_coupon_income = coupons_paid_between(issue_date, horizon_date, coupon_schedule, COUPON_RATE, FACE_VALUE, PAY_FREQ)
bond_horizon_prices = bond_horizon_prices + _govt_coupon_income
bond_pnl = bond_horizon_prices - current_bond_price

print(f"\nMean dirty PV at horizon: {bond_horizon_prices.mean():,.4f} "
      f"(accrued at horizon: {accrued_at_horizon:,.4f})")
display(distribution_summary(bond_horizon_prices).to_frame("UST_10Y_2pct_horizon_value"))
display(loss_summary_from_pnl(bond_pnl).to_frame(f"UST_10Y_2pct_{HORIZON_LABEL.replace('-', '_')}_pnl"))

# --- Corporate bonds: discount at the simulated Treasury curve + the simulated OAS for the rating.
# This is where the equity-credit link bites: scenarios with falling equities tend to carry widening
# spreads (joint bootstrap), so a corporate bond loses on both the rate and the credit channel.
corp_coupon_schedules = {}
corp_current_prices = {}
corp_horizon_prices = {}
corp_horizon_ratings = {}

# Credit migration setup (from 1M on). The SYSTEMATIC credit factor M per scenario is the standardized
# credit state, oriented so a spread blow-out is a LOW M: every name's migration latent shares it, so
# downgrades/defaults cluster exactly in the scenarios where the simulated spreads widened. Normal
# scores (rank -> Phi^{-1}) keep M ~ N(0,1) so the marginal horizon-rating distribution stays faithful
# to the transition matrix while the loading injects the wrong-way dependence.
def _migration_factor_from_spreads(spread_scenarios):
    """Systematic credit factor M per scenario (normal scores; widening -> low M)."""
    chg = (spread_scenarios - bucket_anchor_spreads).mean(axis=1)
    ranks = chg.rank() / (len(chg) + 1.0)
    return -norm.ppf(ranks.to_numpy())


def corporate_horizon_values(spec, key_rate_scen, spread_scen, M=None, mig_rng=None):
    """Horizon value of one corporate bond per scenario -- ONE pricing rule for in-sample and OOS.

    With migration on, each scenario draws a horizon rating (default included) from the horizon
    transition matrix, and the bond reprices on the migrated rating's simulated OAS (recovery on
    default). The coupon cash paid inside the window is always added, so the value is a total-return
    value (zero at 1W/1M/3M with issue on the valuation date). Returns (values, ratings, default_mask).
    """
    rating, cpn = spec["rating"], spec["coupon"]
    sched_c = corp_coupon_schedules[spec["name"]]
    cpn_income = coupons_paid_between(issue_date, horizon_date, sched_c, cpn, FACE_VALUE, PAY_FREQ)
    if CREDIT_MIGRATION_ENABLED:
        h_ratings, defmask = simulate_horizon_ratings(rating, M, MIGRATION_FACTOR_LOADING,
                                                      migration_thresholds, mig_rng)
        repriced = reprice_corporate_bond_with_migration(
            eval_date=horizon_date, payment_dates=sched_c, key_rate_scenarios=key_rate_scen,
            tenor_years=RATE_TENOR_YEARS, all_spread_scenarios=spread_scen,
            horizon_ratings=h_ratings, coupon_rate=cpn, recovery_rate=RECOVERY_RATE,
            face_value=FACE_VALUE, freq=PAY_FREQ,
        )
        return repriced + cpn_income, h_ratings, defmask
    repriced = reprice_corporate_bond_scenarios(
        eval_date=horizon_date, payment_dates=sched_c, key_rate_scenarios=key_rate_scen,
        tenor_years=RATE_TENOR_YEARS, spread_scenarios=spread_scen[rating],
        coupon_rate=cpn, face_value=FACE_VALUE, freq=PAY_FREQ,
    )
    return repriced + cpn_income, None, None


systematic_credit_M, _mig_rng = None, None
if CREDIT_MIGRATION_ENABLED:
    transition_matrix_1y = standard_1y_transition_matrix(tuple(r for r in TRANSITION_RATINGS if r != "D"))
    transition_matrix, transition_info = transition_matrix_for_horizon(transition_matrix_1y, HORIZON_YEARS)
    migration_thresholds = build_migration_thresholds(transition_matrix)
    systematic_credit_M = _migration_factor_from_spreads(credit_spread_scenarios)
    _mig_rng = np.random.default_rng(RNG_SEED + 777)
    if transition_info["n_negative_rates"] or abs(HORIZON_YEARS - 1.0) > 1e-12:
        print(f"\nRating transitions over {HORIZON_MODE}: one-year matrix scaled through its generator "
              f"(P(h) = exp(h Q)); {transition_info['n_negative_rates']} negative off-diagonal rates in "
              f"log P zeroed (worst {transition_info['worst_negative_rate']:.1e}), embedding error "
              f"max|exp(Q) - P| = {transition_info['embedding_error']:.1e}.")

print("\nCorporate bonds (priced on Treasury-zero + rating OAS"
      + (f"; {HORIZON_MODE} rating migration / default ON):" if CREDIT_MIGRATION_ENABLED else "):"))
for spec in CORP_BONDS:
    name, rating = spec["name"], spec["rating"]
    cpn, mat_y = spec["coupon"], spec["maturity_years"]
    sched_c = generate_coupon_schedule(issue_date, issue_date + pd.DateOffset(years=mat_y), freq=PAY_FREQ)
    corp_coupon_schedules[name] = sched_c

    cur_spread = float(bucket_anchor_spreads[rating])
    corp_current_prices[name] = price_coupon_bond_from_curve(
        eval_date=issue_date, payment_dates=sched_c, key_rate_series=current_key_rates,
        tenor_years=RATE_TENOR_YEARS, coupon_rate=cpn, face_value=FACE_VALUE, freq=PAY_FREQ,
        spread=cur_spread,
    )
    corp_horizon_prices[name], h_ratings, _defmask = corporate_horizon_values(
        spec, key_rate_scenarios, credit_spread_scenarios, systematic_credit_M, _mig_rng)
    pnl_c = corp_horizon_prices[name] - corp_current_prices[name]
    if CREDIT_MIGRATION_ENABLED:
        corp_horizon_ratings[name] = h_ratings
        cpn_income = coupons_paid_between(issue_date, horizon_date, sched_c, cpn, FACE_VALUE, PAY_FREQ)
        dflt = float(_defmask.mean())
        downg = float(np.mean(np.array([TRANSITION_RATINGS.index(r) for r in h_ratings]) > TRANSITION_RATINGS.index(rating)))
        print(f"  {name} ({rating}, {cpn:.2%} {mat_y}Y): current OAS {cur_spread*1e4:,.0f}bps, "
              f"price {corp_current_prices[name]:,.3f} -> mean horizon {corp_horizon_prices[name].mean():,.3f} "
              f"(coupon +{cpn_income:.2f}; P(default) {dflt:.2%}, P(downgrade) {downg:.1%}; mean {HORIZON_MODE} P&L {pnl_c.mean():,.4f})")
    else:
        print(f"  {name} ({rating}, {cpn:.2%} {mat_y}Y): current OAS {cur_spread*1e4:,.0f}bps, "
              f"price {corp_current_prices[name]:,.3f} -> mean horizon {corp_horizon_prices[name].mean():,.3f} "
              f"(mean {HORIZON_MODE.lower()} P&L {pnl_c.mean():,.4f})")

# --- FX forward(s): a long-foreign forward struck at-the-money on the valuation curve, so its MTM
# is zero at inception and the horizon P&L is pure FX (the simulated spot) plus USD discounting (the
# simulated Treasury curve). It is repriced on the SAME simulated spot and curve rows, so an FX move
# and a rate move hit it jointly. For the capital-normalized portfolio layer the forward is treated
# as a fully-funded position: its capital base (current_value) is the USD notional and its horizon
# value is USD notional + MTM, so the instrument return is MTM / USD-notional.
fx_forward_specs = {}
fx_forward_current_values = {}
fx_forward_horizon_values = {}

print("\nFX forwards (long-foreign, struck at-the-money on the valuation curve):")
for spec in FX_FORWARDS:
    name, pair = spec["name"], spec["pair"]
    # When the horizon reaches the forward's tenor (3M and 1Y for the 3M forward) the original maturity
    # would fall at or before the horizon -- at 3M the forward would arrive three days from expiry --
    # so anchor the maturity PAST the horizon: the forward keeps its original tenor outstanding AT the
    # horizon, mirroring the 1W/1M setup. The USD leg is fully simulated (usd_key_rate_scenarios); the
    # foreign funding rate is held constant, an approximation that is immaterial at 1W but a real
    # simplification at 3M-1Y (a simulated foreign curve would be the fix).
    if HORIZON_YEARS >= spec["maturity_years"]:
        fwd_maturity = horizon_date + pd.DateOffset(months=int(round(spec["maturity_years"] * 12)))
    else:
        fwd_maturity = issue_date + pd.DateOffset(months=int(round(spec["maturity_years"] * 12)))
    spot0 = float(current_fx[pair])
    r_for = float(spec["foreign_funding_rate"])
    notl = float(spec["notional_foreign"])
    strike = atm_forward_fx_strike(
        eval_date=issue_date, maturity_date=fwd_maturity, spot=spot0,
        usd_key_rate_series=current_key_rates, usd_tenor_years=RATE_TENOR_YEARS,
        foreign_funding_rate=r_for,
    )
    usd_notional = notl * spot0
    mtm_h = reprice_fx_forward_scenarios(
        eval_date=horizon_date, maturity_date=fwd_maturity, spot_scenarios=fx_spot_scenarios[pair],
        usd_key_rate_scenarios=key_rate_scenarios, usd_tenor_years=RATE_TENOR_YEARS,
        strike=strike, foreign_funding_rate=r_for, notional_foreign=notl,
    )
    fx_forward_specs[name] = {
        "pair": pair, "maturity_date": fwd_maturity, "strike": strike,
        "foreign_funding_rate": r_for, "notional_foreign": notl, "usd_notional": usd_notional,
    }
    fx_forward_current_values[name] = usd_notional            # capital base (MTM at inception ~ 0)
    fx_forward_horizon_values[name] = usd_notional + mtm_h    # so return = MTM / USD-notional
    print(f"  {name} ({pair}, {spec['maturity_years']:.2f}Y): spot {spot0:.4f}, ATM strike {strike:.4f}, "
          f"USD notional {usd_notional:,.2f} -> mean horizon MTM {mtm_h.mean():,.4f} "
          f"(mean {HORIZON_MODE.lower()} return {(mtm_h.mean()/usd_notional):.4%})")

# Assemble the full instrument set: equities + Treasury + corporates + FX forward(s)
instrument_horizon_values = stock_price_scenarios.copy()
instrument_horizon_values["UST_10Y_2pct"] = bond_horizon_prices
for name in corp_horizon_prices:
    instrument_horizon_values[name] = corp_horizon_prices[name].to_numpy()
for name in fx_forward_horizon_values:
    instrument_horizon_values[name] = fx_forward_horizon_values[name].to_numpy()

current_values = current_prices.copy()
current_values["UST_10Y_2pct"] = current_bond_price
for name in corp_current_prices:
    current_values[name] = corp_current_prices[name]
for name in fx_forward_current_values:
    current_values[name] = fx_forward_current_values[name]

instrument_pnl = instrument_horizon_values.sub(current_values, axis=1)
instrument_returns = instrument_pnl.div(current_values, axis=1)

print("\nHorizon values:")
display(instrument_horizon_values.describe().T)

print(f"Instrument-level {HORIZON_LABEL} P&L:")
display(instrument_pnl.describe().T)

print(f"Capital-normalized {HORIZON_LABEL} return/loss metrics:")
display(pd.DataFrame({
    col: loss_summary_from_pnl(instrument_returns[col], alpha=CVAR_ALPHA)
    for col in instrument_returns.columns
}).T)


In [ ]:

# Selected instrument-level visualizations
plot_histogram(instrument_horizon_values[REPRESENTATIVE_EQUITY], f"Horizon-value distribution — {REPRESENTATIVE_EQUITY}", "Horizon value")
plot_histogram(instrument_pnl[REPRESENTATIVE_EQUITY], f"P&L distribution — {REPRESENTATIVE_EQUITY}", f"{HORIZON_LABEL.capitalize()} P&L")
plot_histogram(instrument_horizon_values["UST_10Y_2pct"], "Horizon-value distribution — UST_10Y_2pct", "Horizon value")
plot_histogram(instrument_pnl["UST_10Y_2pct"], "P&L distribution — UST_10Y_2pct", f"{HORIZON_LABEL.capitalize()} P&L")



## 5. Portfolio layer under capital-normalized scenario returns

Portfolio comparisons are performed on **horizon simple returns** (one week by default), not raw dollar P&L per instrument unit. This keeps tail-risk comparisons economically interpretable when assets have different nominal prices.


In [ ]:
# Candidate portfolio weights in capital-normalized return space
all_assets = instrument_returns.columns.tolist()
equity_assets = TICKERS.copy()
corp_assets = [spec["name"] for spec in CORP_BONDS]
fx_assets = [spec["name"] for spec in FX_FORWARDS]

equal_weight = pd.Series(1.0 / len(all_assets), index=all_assets)

equity_only = pd.Series(0.0, index=all_assets)
equity_only.loc[equity_assets] = 1.0 / len(equity_assets)

# Treasury-heavy: 60% Treasury, 40% spread across equities, no corporate credit (sums to 1 over the
# enlarged universe — the previous fixed-denominator version no longer summed to one once corporate
# bonds were added).
bond_heavy = pd.Series(0.0, index=all_assets)
bond_heavy.loc[equity_assets] = 0.40 / len(equity_assets)
bond_heavy["UST_10Y_2pct"] = 0.60

# Credit-tilt: deliberately loads the corporate (credit-spread) names to expose the new risk block.
credit_tilt = pd.Series(0.0, index=all_assets)
credit_tilt.loc[equity_assets] = 0.40 / len(equity_assets)
credit_tilt["UST_10Y_2pct"] = 0.20
if corp_assets:
    for c in corp_assets:
        credit_tilt[c] = 0.40 / len(corp_assets)

# FX-overlay: equities + Treasury with a sleeve allocated to the FX forward(s), to exercise the FX
# block in the portfolio. The forward's capital base is its USD notional, so a weight here is the
# fraction of capital carried as the (fully-funded) FX position. Sums to one over the universe.
fx_overlay = pd.Series(0.0, index=all_assets)
fx_overlay.loc[equity_assets] = 0.40 / len(equity_assets)
fx_overlay["UST_10Y_2pct"] = 0.40
if fx_assets:
    for f in fx_assets:
        fx_overlay[f] = 0.20 / len(fx_assets)

inverse_vol = inverse_vol_weights_from_scenarios(instrument_returns)

# Issue #3: exact long-only min-CVaR via the Rockafellar-Uryasev LP (no more random search).
min_cvar_long_only, min_cvar_info = search_long_only_min_cvar_portfolio(
    instrument_returns=instrument_returns,
    alpha=CVAR_ALPHA,
    max_weight=CVAR_MAX_WEIGHT,
)

portfolio_weights = {
    "equal_weight": equal_weight,
    "equity_only": equity_only,
    "bond_heavy": bond_heavy,
    "credit_tilt": credit_tilt,
    "fx_overlay": fx_overlay,
    "inverse_vol": inverse_vol,
    "min_cvar_long_only": min_cvar_long_only,
}

portfolio_returns = {}
portfolio_metrics = {}
for name, w in portfolio_weights.items():
    port_ret, metrics = portfolio_metrics_from_instrument_returns(w, instrument_returns, alpha=CVAR_ALPHA)
    portfolio_returns[name] = port_ret
    portfolio_metrics[name] = metrics

print("Long-only min-CVaR LP diagnostics:")
display(min_cvar_info.to_frame("value"))

print("Portfolio weights:")
display(pd.DataFrame(portfolio_weights).T)

print(f"Portfolio {HORIZON_LABEL} return/loss metrics:")
display(pd.DataFrame(portfolio_metrics).T.sort_values("ES_95_loss"))


In [ ]:
# Out-of-sample (fresh-draw) portfolio tail risk (issue #4)
# The min-CVaR weights are chosen to minimize ES on the in-sample scenario set, so the in-sample
# ES is optimistically biased (selection on noise). We simulate an INDEPENDENT scenario set with a
# different seed, reprice every instrument (equities, Treasury, corporates, FX forward), and
# re-evaluate the SAME weights to get an honest ES. The repricing must be the SAME as in-sample, or the
# in/out gap mixes selection-on-noise with a change of pricing model: up to v20 this cell dropped the
# coupon income and, at 1Y, the rating migration / default leg that the in-sample corporates carry.
# It now adds the same coupon cash and calls the same corporate pricer (own migration RNG stream).

# Same engine as the in-sample draw (v24: main_sim_kwargs, so a hierarchical engine is honoured too),
# independent seed.
sim_oos = simulate_joint_horizon(**{**main_sim_kwargs, "rng_seed": VALIDATION_SEED})

key_rate_scenarios_oos = sim_oos["key_rate_scenarios"]
credit_spread_scenarios_oos = sim_oos["credit_spread_scenarios"]
fx_spot_scenarios_oos = sim_oos["fx_spot_scenarios"]

bond_horizon_prices_oos = key_rate_scenarios_oos.apply(
    lambda row: price_coupon_bond_from_curve(
        eval_date=horizon_date,
        payment_dates=coupon_schedule,
        key_rate_series=row,
        tenor_years=RATE_TENOR_YEARS,
        coupon_rate=COUPON_RATE,
        face_value=FACE_VALUE,
        freq=PAY_FREQ,
    ),
    axis=1,
) + _govt_coupon_income

instrument_horizon_values_oos = sim_oos["stock_price_scenarios"].copy()
instrument_horizon_values_oos["UST_10Y_2pct"] = bond_horizon_prices_oos
_M_oos = _migration_factor_from_spreads(credit_spread_scenarios_oos) if CREDIT_MIGRATION_ENABLED else None
_mig_rng_oos = np.random.default_rng(VALIDATION_SEED + 777) if CREDIT_MIGRATION_ENABLED else None
for spec in CORP_BONDS:
    _vals_oos, _, _ = corporate_horizon_values(spec, key_rate_scenarios_oos, credit_spread_scenarios_oos,
                                               _M_oos, _mig_rng_oos)
    instrument_horizon_values_oos[spec["name"]] = np.asarray(_vals_oos, dtype=float)

# FX forward(s) on the independent draw: same struck strike / maturity / notional as the in-sample
# leg (from fx_forward_specs), repriced on the fresh spot and curve, then expressed as USD notional
# + MTM so it matches the fully-funded current_value used in-sample.
for name, fwd in fx_forward_specs.items():
    mtm_oos = reprice_fx_forward_scenarios(
        eval_date=horizon_date, maturity_date=fwd["maturity_date"],
        spot_scenarios=fx_spot_scenarios_oos[fwd["pair"]], usd_key_rate_scenarios=key_rate_scenarios_oos,
        usd_tenor_years=RATE_TENOR_YEARS, strike=fwd["strike"],
        foreign_funding_rate=fwd["foreign_funding_rate"], notional_foreign=fwd["notional_foreign"],
    )
    instrument_horizon_values_oos[name] = (fwd["usd_notional"] + mtm_oos).to_numpy()

instrument_returns_oos = instrument_horizon_values_oos.sub(current_values, axis=1).div(current_values, axis=1)

oos_rows = {}
for name, w in portfolio_weights.items():
    _, m_in = portfolio_metrics_from_instrument_returns(w, instrument_returns, alpha=CVAR_ALPHA)
    _, m_oos = portfolio_metrics_from_instrument_returns(w, instrument_returns_oos, alpha=CVAR_ALPHA)
    oos_rows[name] = {
        "VaR_95_in_sample": m_in["VaR_95_loss"],
        "VaR_95_out_of_sample": m_oos["VaR_95_loss"],
        "ES_95_in_sample": m_in["ES_95_loss"],
        "ES_95_out_of_sample": m_oos["ES_95_loss"],
    }

oos_table = pd.DataFrame(oos_rows).T
oos_table["ES_in_sample_optimism"] = oos_table["ES_95_out_of_sample"] - oos_table["ES_95_in_sample"]
print(f"In-sample vs out-of-sample {HORIZON_LABEL} tail risk (independent scenario draw):")
display(oos_table.sort_values("ES_95_out_of_sample"))
print(
    "A positive 'ES_in_sample_optimism' for min_cvar_long_only is the selection-on-noise effect: "
    "the weights were fit to the in-sample tail, so their ES degrades out-of-sample. Policy "
    "portfolios (equal_weight, equity_only, bond_heavy, credit_tilt, fx_overlay) are not fit to the "
    "scenarios and should show only sampling noise between the two columns."
)


In [ ]:

# Portfolio-level distributions and weighted tail contributions
for name, port_ret in portfolio_returns.items():
    plot_histogram(port_ret, f"Portfolio {HORIZON_LABEL} return distribution — {name}", f"{HORIZON_LABEL.capitalize()} portfolio return")

tail_table = tail_contribution_table(
    portfolio_return=portfolio_returns["min_cvar_long_only"],
    instrument_returns=instrument_returns,
    weights=portfolio_weights["min_cvar_long_only"],
    alpha=CVAR_ALPHA,
)

display(tail_table)


## Research defaults vs full validation

This notebook is intentionally the **engine**, not the full model-selection lab.

The default run uses the current hierarchical/direct-horizon architecture and a lighter recent
walk-forward battery. For the research evidence behind architecture, duration-PIT and propagation
choices, run the dedicated notebooks:

1. `02_dependence_architecture_ablation.ipynb`
2. `03_duration_pit_factorial.ipynb`
3. `04_validation_scores.ipynb`

The blockwise architecture remains the permanent hard benchmark.
